In [ ]:
# === Importação de Biliotecas ===
import os
import gc
import math
import time
import math
import time
#import tqdm
from tqdm.notebook import tqdm
from sklearn.metrics import auc
import zipfile
import subprocess
import numpy as np
import pandas as pd
from fpdf import FPDF
import seaborn as sns
import miceforest as mf
import matplotlib.pyplot as plt
from scipy.stats import bootstrap
from sklearn.metrics import log_loss
import matplotlib.patches as mpatches
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold
from matplotlib.backends.backend_pdf import PdfPages
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, balanced_accuracy_score, roc_auc_score, roc_curve, confusion_matrix


In [ ]:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r"X does not have valid feature names, but .* was fitted with feature names",
    category=UserWarning,
)

import os
os.environ["PYTHONWARNINGS"] = (
     "ignore:X does not have valid feature names, but .* was fitted with feature names:UserWarning"
 )

In [ ]:
from joblib import load

# === Parametros Genéricos ===
# === Definição de Parâmetros - Obrigatório ser informados pelo usuario
BASE_PATH = "WDBC_SURROGATE_XGBOOST" 
FILE_IN = Path('../data/processed/wdbc.csv')

os.makedirs(BASE_PATH, exist_ok=True)

METHOD = "" #Informado na célula de cada método

direcao_esperada = {
    'radius_mean': +1,              # tumores malignos tendem a ter maior raio médio
    'texture_mean': +1,             # textura mais heterogênea indica malignidade
    'perimeter_mean': +1,           # perímetro maior indica malignidade
    'area_mean': +1,                # áreas maiores associadas a malignidade
    'smoothness_mean': +1,          # maior irregularidade de superfície
    'compactness_mean': +1,         # maior compacidade associada a malignidade
    'concavity_mean': +1,           # concavidade mais pronunciada em tumores malignos
    'concave_points_mean': +1,      # mais pontos côncavos em malignos
    'symmetry_mean': 0,             # simetria pode não ter direção clara
    'fractal_dimension_mean': +1,   # maior complexidade indica malignidade

    'radius_se': +1,                # variação maior do raio → malignidade
    'texture_se': +1,               # variação de textura maior → malignidade
    'perimeter_se': +1,
    'area_se': +1,
    'smoothness_se': 0,
    'compactness_se': +1,
    'concavity_se': +1,
    'concave_points_se': +1,
    'symmetry_se': 0,
    'fractal_dimension_se': +1,

    'radius_worst': +1,             # valores "worst" (máximos) são fortes preditores
    'texture_worst': +1,
    'perimeter_worst': +1,
    'area_worst': +1,
    'smoothness_worst': +1,
    'compactness_worst': +1,
    'concavity_worst': +1,
    'concave_points_worst': +1,
    'symmetry_worst': +1,
    'fractal_dimension_worst': +1,

    'diagnosis': 0,                 # variável alvo (não se aplica direção)
    'id': 0                         # identificador, sem significado clínico
}

PATH_CSV = Path('../model_reports/xgboost/basico/csv')
#X_test_basic_full.csv
#X_train_basic_full.csv

X_train = pd.read_csv(f"{PATH_CSV}X_train_basic_full.csv")  
X_test = pd.read_csv(f"{PATH_CSV}X_test_basic_full.csv")  

X_train = X_train.drop(X_train.columns[0], axis=1)
X_test = X_test.drop(X_test.columns[0], axis=1)

X = pd.concat([X_train, X_test], ignore_index=True).sort_values(by="id").reset_index(drop=True)
X_ = X.copy()
X_['diagnosis'] = X_['diagnosis'].replace({'M': 1, 'B': 0}).astype(int)

EXCLUDE_COLS = ['id', 'y_pred', 'y_proba', 'y','diagnosis']

ENTRADAS_SELECIONADAS = pd.read_csv(Path('../clusters_reports/random_forest/csv/entradas_selecionadas_final_id_only.csv'))
ENTRADAS_SELECIONADAS = ENTRADAS_SELECIONADAS.T.squeeze().to_list() 

model = load(Path('../common/models/xgboost/v1/xgb_basic_ts30_rs42.pkl'))


In [ ]:
# === Surrogate Decision Tree Interpretabilidade ===
# 1) Bibliotecas
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 2) Configurações de saída
METHOD = "Surrogate_Tree"  # nome do método para compor o caminho
PATH_FILE_OUT = os.path.join(BASE_PATH, METHOD)
NM_FILE_SURROGATE = "XAI_Surrogate_Tree"
os.makedirs(PATH_FILE_OUT, exist_ok=True)

# 3) Definição do espaço de features do surrogate
#    Usa X_train, mas removendo colunas que não são features (EXCLUDE_COLS)
X_train_feat = X_train.drop(columns=EXCLUDE_COLS, errors="ignore")
X_test_feat  = X_test.drop(columns=EXCLUDE_COLS, errors="ignore")

X_surrogate = X_train_feat  # manter DataFrame para nomes de colunas
y_surrogate = model.predict_proba(X_train_feat)[:, 1]

# 3.1) Treino da árvore (grid leve para equilibrar simplicidade x fidelidade)
tree_base = DecisionTreeRegressor(random_state=42)
param_grid = {
    "max_depth": [3, 4, 5, 6],
    "min_samples_leaf": [20, 50, 100, 200],
    "ccp_alpha": [0.0, 1e-4, 5e-4, 1e-3]
}
grid = GridSearchCV(
    tree_base,
    param_grid=param_grid,
    scoring="neg_mean_squared_error",
    cv=3,
    n_jobs=-1
)
grid.fit(X_surrogate, y_surrogate)

surrogate_model = grid.best_estimator_
feature_names = list(X_surrogate.columns)

# 3.2) Relato rápido de fidelidade global (no próprio X_train)
y_hat_train = surrogate_model.predict(X_train_feat)
rmse_train = float(np.sqrt(mean_squared_error(y_surrogate, y_hat_train)))
r2_train   = float(r2_score(y_surrogate, y_hat_train))
print("Melhores hiperparâmetros:", grid.best_params_)
print({"rmse_teacher_train": rmse_train, "r2_teacher_train": r2_train})

# 3.3) Exporta regras globais em texto (árvore inteira)
rules_text = export_text(surrogate_model, feature_names=feature_names)
with open(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_rules.txt"), "w", encoding="utf-8") as f:
    f.write(rules_text)

# Helper: caminho (regras) percorrido por uma instância na árvore
def explain_instance_path_tree(estimator: DecisionTreeRegressor, x_row: pd.Series, feature_names):
    """
    Retorna dicionário com o caminho de splits percorrido pela instância e a predição no leaf.
    """
    x = x_row.values.reshape(1, -1) if hasattr(x_row, "values") else np.asarray(x_row).reshape(1, -1)
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold

    node = 0
    path = []
    while tree.children_left[node] != tree.children_right[node]:
        f_idx = feat[node]
        t = thr[node]
        fname = feature_names[f_idx]
        val = float(x[0, f_idx])
        go_left = val <= t
        cond = f"{fname} <= {t:.6f}" if go_left else f"{fname} > {t:.6f}"
        path.append({
            "node": int(node),
            "split_feature": fname,
            "threshold": float(t),
            "value": val,
            "decision": cond
        })
        node = tree.children_left[node] if go_left else tree.children_right[node]

    leaf_pred = float(estimator.predict(x)[0])
    return {"path": path, "leaf_pred": leaf_pred}

# 4) Explicações para X_test (por instância)
surrogate_results = []

for i, idx in enumerate(X_test_feat.index):
    # linha de features para passar no modelo
    x_series_feat = X_test_feat.loc[idx]
    
    # probabilidade do XGB (professor) usando somente as features
    prob_xgb= float(
        model.predict_proba(X_test_feat.loc[[idx]])[0, 1]
    )
    
    # probabilidade do surrogate (árvore)
    prob_surrogate = float(
        surrogate_model.predict(X_test_feat.loc[[idx]])[0]
    )
    
    erro_aprox = abs(prob_xgb- prob_surrogate)

    # caminho de regras percorrido pela instância
    local_expl = explain_instance_path_tree(surrogate_model, x_series_feat, feature_names)
    path_str = " AND ".join([p["decision"] for p in local_expl["path"]]) if local_expl["path"] else "(all)"

    # tamanho real (usa coluna 'y' de X_test)
    if 'y' in X_test.columns:
        y_true = int(X_test.loc[idx, 'y'])
        diagnostico = 'maligno' if y_true == 1 else 'benigno'
    else:
        diagnostico = None

    # se existir coluna 'id', registra também
    id_val = X_test.loc[idx, 'id'] if 'id' in X_test.columns else idx

    linha = {
        'id': id_val,
        'linha': idx,
        'prob_modelo': prob_xgb,
        'soma_surrogate': prob_surrogate,
        'erro_aproximacao': erro_aprox,
        'leaf_pred': local_expl["leaf_pred"],
        'regras_instancia': path_str,
        'diagnostico': diagnostico
    }

    surrogate_results.append(linha)

# 5) Salva em Excel (tabela por instância + importâncias)
df_surrogate = pd.DataFrame(surrogate_results)

# (opcional) adiciona importâncias como arquivo separado (globais da árvore)
importancias = getattr(surrogate_model, "feature_importances_", None)
if importancias is not None:
    imp_df = pd.DataFrame({"feature": feature_names, "importance": importancias}).sort_values(
        "importance", ascending=False
    )
    imp_path = os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_importances.xlsx")
    imp_df.to_excel(imp_path, index=False)
    print(f"💡 Importâncias globais salvas em: {imp_path}")

out_path = os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}.xlsx")
df_surrogate.to_excel(out_path, index=False)
print(f"✅ Arquivo salvo em {out_path}")

# 6) Gráfico do erro de aproximação no X_test
plt.figure(figsize=(10, 5))
plt.hist(df_surrogate["erro_aproximacao"], bins=20, edgecolor='black')
plt.title(f"Erro de Aproximação do Surrogate Decision Tree (média = {df_surrogate['erro_aproximacao'].mean():.4f})")
plt.xlabel("Erro de Aproximação |XGB - árvore|")
plt.ylabel("Frequência")
plt.grid(True)
plt.tight_layout()
plt.show()

# (extra) Dispersão XGB vs Surrogate (boa para ver fidelidade)
plt.figure(figsize=(6, 6))
plt.scatter(df_surrogate["prob_modelo"], df_surrogate["soma_surrogate"], alpha=0.6)
plt.plot([0, 1], [0, 1], linestyle="--")
rmse = np.sqrt(((df_surrogate["prob_modelo"] - df_surrogate["soma_surrogate"])**2).mean())
plt.title(f"XGB vs Surrogate (árvore) — RMSE={rmse:.4f}")
plt.xlabel("Probabilidade XGB (professor)")
plt.ylabel("Probabilidade Surrogate (árvore)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# === Surrogate Decision Tree Interpretabilidade [Analises Complementares] ===
# Helper para extrair regras (folhas), suporte e um score da folha
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

def extract_rules_table(estimator, X, feature_names):
    """
    Gera um DataFrame com:
      - rule_text: conj. de condições (caminho até a folha)
      - support: nº de instâncias de X que caem nessa folha
      - leaf_pred_mean:
          * Regressor: valor médio da folha (constante da árvore)
          * Classifier: prob. da classe positiva (se existir classe '1'), senão prob. da classe predita (confiança)
    """
    import numpy as np
    import pandas as pd

    # Garante DataFrame e colunas corretas
    if not hasattr(X, "iloc"):
        X_df = pd.DataFrame(X, columns=feature_names)
    else:
        if set(feature_names).issubset(set(X.columns)):
            X_df = X[feature_names].copy()
        else:
            X_df = X.copy()

    arr = X_df.values
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold

    is_clf = isinstance(estimator, DecisionTreeClassifier)
    classes_ = getattr(estimator, "classes_", None)

    rules = []

    def recurse(node, constraints):
        left = tree.children_left[node]
        right = tree.children_right[node]
        if left == right:  # folha
            # máscara das instâncias que satisfazem TODAS as condições
            mask = np.ones(len(arr), dtype=bool)
            for f_i, op, t in constraints:
                if op == "<=":
                    mask &= (arr[:, f_i] <= t)
                else:
                    mask &= (arr[:, f_i] > t)
            support = int(mask.sum())

            # valor da folha
            if is_clf:
                counts = tree.value[node][0]
                total = counts.sum()
                if total > 0:
                    if classes_ is not None and len(classes_) == 2 and 1 in classes_:
                        idx1 = list(classes_).index(1)
                        leaf_pred_mean = float(counts[idx1] / total)
                    else:
                        leaf_pred_mean = float(counts.max() / total)  # confiança da classe majoritária
                else:
                    leaf_pred_mean = float("nan")
            else:
                leaf_pred_mean = float(tree.value[node][0, 0])  # regressão: média do y na folha

            rules.append((constraints.copy(), support, leaf_pred_mean))
        else:
            f_i = feat[node]
            t = thr[node]
            recurse(left,  constraints + [(f_i, "<=", t)])
            recurse(right, constraints + [(f_i, ">",  t)])

    recurse(0, [])

    out = []
    for cons, support, leaf_pred_mean in rules:
        parts = [f"{feature_names[f]} {op} {t:.6f}" for f, op, t in cons]
        out.append({
            "rule_text": " AND ".join(parts) if parts else "(all)",
            "support": support,
            "leaf_pred_mean": leaf_pred_mean
        })

    return (pd.DataFrame(out)
              .sort_values("support", ascending=False)
              .reset_index(drop=True))

# ===================== Regras + Suporte (preparo) =====================
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

# cria rules_df se não existir
try:
    rules_df
except NameError:
    # usa o mesmo espaço de features do surrogate (X_surrogate = X_train_feat)
    rules_df = extract_rules_table(surrogate_model, X_surrogate, feature_names)

rules_df = rules_df.copy()
rules_df["support"] = rules_df["support"].astype(int)
rules_df["leaf_pred_mean"] = rules_df["leaf_pred_mean"].astype(float)

# ordenado por suporte e com cumulativo (%)
rules_sorted = rules_df.sort_values("support", ascending=False).reset_index(drop=True)
total_support = rules_sorted["support"].sum()
rules_sorted["cum_support"] = rules_sorted["support"].cumsum()
rules_sorted["cum_support_pct"] = 100.0 * rules_sorted["cum_support"] / max(1, total_support)

# salva tabela enriquecida
rules_xlsx = os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_rules_with_support.xlsx")
rules_sorted.to_excel(rules_xlsx, index=False)
print(f"📄 Tabela de regras salva em: {rules_xlsx}")

# ===================== 1) Histograma do suporte por regra =====================
plt.figure(figsize=(10,5))
plt.hist(rules_df["support"].values, bins=20, edgecolor="black")
plt.title("Distribuição do Suporte das Regras (folhas da árvore)")
plt.xlabel("Suporte (nº de instâncias na folha)")
plt.ylabel("Frequência de regras")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_hist_suporte_regras.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ===================== 2) Top-N regras por suporte (barra horizontal) =====================
TOP_N = 20  # ajuste se quiser
topN = rules_sorted.head(TOP_N).copy()

plt.figure(figsize=(12, 8))
y_labels = [f"#{i+1}: {txt}" for i, txt in enumerate(topN["rule_text"])]
plt.barh(y_labels, topN["support"].values, edgecolor="black")
plt.gca().invert_yaxis()
plt.xlabel("Suporte")
plt.title(f"Top-{TOP_N} Regras por Suporte")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_top{TOP_N}_regras_suporte.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ===================== 3) Curva de Pareto do suporte (cumulativo) =====================
plt.figure(figsize=(10,5))
x = np.arange(1, len(rules_sorted)+1)
plt.bar(x, rules_sorted["support"].values, edgecolor="black", alpha=0.6)
plt.plot(x, rules_sorted["cum_support_pct"].values, marker="o")
plt.axhline(80, linestyle="--")  # linha de referência 80%
plt.xlabel("Regras (ordenadas por suporte)")
plt.ylabel("Suporte (barras) / Suporte acumulado (%)")
plt.title("Pareto do Suporte das Regras")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_pareto_suporte.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ===================== 4) Frequência de features nas regras (não-ponderada e ponderada por suporte) =====================
def _features_from_rule(rule_text: str):
    # rule_text no formato "feat1 <= thr AND feat2 > thr2 ..."
    feats = []
    for part in str(rule_text).split("AND"):
        tok = part.strip().split()
        if len(tok) > 0:
            feats.append(tok[0])  # primeira palavra é o nome da feature
    return feats

# contagem simples
cnt = Counter()
# contagem ponderada por suporte da regra
cnt_w = Counter()

for _, row in rules_df.iterrows():
    feats = _features_from_rule(row["rule_text"])
    cnt.update(feats)
    # ponderada: some o suporte para cada feature presente na regra
    for f in set(feats):
        cnt_w[f] += int(row["support"])

freq_df = (pd.DataFrame({"feature": list(cnt.keys()), "count": list(cnt.values())})
           .sort_values("count", ascending=False)
           .reset_index(drop=True))

freqw_df = (pd.DataFrame({"feature": list(cnt_w.keys()), "support_weighted": list(cnt_w.values())})
            .sort_values("support_weighted", ascending=False)
            .reset_index(drop=True))

# salva tabelas
freq_xlsx = os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_features_freq_não_ponderado.xlsx")
freqw_xlsx = os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_features_freq_ponderado_suporte.xlsx")
freq_df.to_excel(freq_xlsx, index=False)
freqw_df.to_excel(freqw_xlsx, index=False)
print(f"📄 Frequência de features salva em: {freq_xlsx}")
print(f"📄 Frequência ponderada por suporte salva em: {freqw_xlsx}")

# gráficos (Top-K)
TOP_K = 15

plt.figure(figsize=(10,6))
plt.barh(freq_df["feature"].head(TOP_K)[::-1], freq_df["count"].head(TOP_K)[::-1], edgecolor="black")
plt.title(f"Top-{TOP_K} Features mais frequentes nas Regras (não-ponderado)")
plt.xlabel("Contagem em regras")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_top{TOP_K}_features_freq_nao_pond.png"),
            dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(10,6))
plt.barh(freqw_df["feature"].head(TOP_K)[::-1], freqw_df["support_weighted"].head(TOP_K)[::-1], edgecolor="black")
plt.title(f"Top-{TOP_K} Features por Suporte (ponderado)")
plt.xlabel("Suporte acumulado das regras onde a feature aparece")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_top{TOP_K}_features_freq_pond.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# ===================== 5) Distribuição de leaf_pred_mean (ponderada por suporte) =====================
# histograma da média prevista nos nós-folha
vals = rules_df["leaf_pred_mean"].values
wts  = rules_df["support"].values  # mantido se quiser usar depois

plt.figure(figsize=(10,5))
plt.hist(vals, bins=20, edgecolor="black")
plt.title("Distribuição de leaf_pred_mean (não ponderado)")
plt.xlabel("leaf_pred_mean (prob. média na folha)")
plt.ylabel("Nº de folhas")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_hist_leaf_pred_mean.png"),
            dpi=150, bbox_inches="tight")
plt.show()

# alternativa: scatter suporte vs leaf_pred_mean
plt.figure(figsize=(8,6))
plt.scatter(rules_df["leaf_pred_mean"], rules_df["support"], alpha=0.7)
plt.xlabel("leaf_pred_mean")
plt.ylabel("Suporte")
plt.title("Suporte por folha vs. leaf_pred_mean")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_scatter_support_vs_leafpred.png"),
            dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# === A) BOX PLOT DO SUPORTE DAS REGRAS (GERAL E POR CLASSE) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# 1) Mapeia cada instância de X_train (ESPAÇO DE FEATURES) para o ID da folha
#    X_surrogate = X_train_feat (definido no bloco principal do surrogate)
leaf_id_per_row = surrogate_model.apply(X_surrogate)  # array [n_train]

# y_train: usa coluna 'y' do X_train original, se existir;
#          se não existir, tenta usar variável y_train global (fallback).
if 'y' in X_train.columns:
    y_train_series = X_train['y'].reset_index(drop=True)
elif 'y_train' in globals():
    y_train_series = pd.Series(y_train).reset_index(drop=True)
else:
    raise RuntimeError("Não encontrei rótulos de treino: nem X_train['y'], nem variável y_train.")

# 2) Suporte total por folha e suporte por classe
df_leafmap = pd.DataFrame({"leaf_id": leaf_id_per_row, "y": y_train_series.values})
support_total = df_leafmap.groupby("leaf_id").size().rename("support")
support_by_class = df_leafmap.groupby(["leaf_id", "y"]).size().unstack(fill_value=0)

# garante colunas 0/1 mesmo se faltar alguma (assumindo problema binário 0/1)
for c in [0, 1]:
    if c not in support_by_class.columns:
        support_by_class[c] = 0
support_by_class = support_by_class[[0, 1]]  # ordem

# 3) (Opcional) extrai o texto da regra por leaf_id para enriquecer análises
def _leaf_rules_with_ids(estimator, feature_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    out = []
    def rec(node, cons):
        left, right = tree.children_left[node], tree.children_right[node]
        if left == right:
            parts = [f"{feature_names[f]} {op} {t:.6f}" for f, op, t in cons]
            out.append({"leaf_id": int(node), "rule_text": " AND ".join(parts) if parts else "(all)"})
        else:
            f = feat[node]; t = thr[node]
            rec(left,  cons + [(f, "<=", t)])
            rec(right, cons + [(f, ">",  t)])
    rec(0, [])
    return pd.DataFrame(out)

rules_ids_df = _leaf_rules_with_ids(surrogate_model, feature_names)
leaf_summary = (rules_ids_df
                .merge(support_total.reset_index(), on="leaf_id", how="left")
                .merge(support_by_class.reset_index(), on="leaf_id", how="left")
                .fillna(0))

# 4) BOX PLOT (GERAL)
sup = leaf_summary["support"].to_numpy()
q1, med, q3 = np.quantile(sup, [0.25, 0.5, 0.75])
iqr = q3 - q1

fig, ax = plt.subplots(figsize=(4.8, 7))
ax.boxplot(
    sup, vert=True, notch=True, patch_artist=True,
    boxprops=dict(facecolor="#4C78A8", alpha=0.55),
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.4), capprops=dict(linewidth=1.4),
    flierprops=dict(marker='o', markersize=4, alpha=0.5)
)
# jitter para visualizar densidade
rng = np.random.default_rng(0)
ax.scatter(rng.normal(1.0, 0.035, sup.size), sup, s=18, alpha=0.55, edgecolors="none")

ax.set_title("Box Plot — Suporte das Regras (geral)")
ax.set_ylabel("Suporte (nº de instâncias por regra)")
ax.set_xticks([])
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_boxplot_suporte_geral.png"),
            dpi=180, bbox_inches="tight")
plt.show()

print({
    "n_regras": int(sup.size),
    "mean": float(np.mean(sup)),
    "q1": float(q1),
    "median": float(med),
    "q3": float(q3),
    "iqr": float(iqr)
})

# 5) BOX PLOT (POR CLASSE REAL) — robusto a rótulos não {0,1}
# support_by_class tem colunas iguais aos rótulos existentes em y_train (podem ser 0/1 ou strings)
cls_labels = list(support_by_class.columns)
if len(cls_labels) == 1:
    # garante 2 colunas para fins de plot, adicionando a segunda com zeros
    missing_label = "OUTRA" if cls_labels[0] != "OUTRA" else "OUTRA_2"
    support_by_class[missing_label] = 0
    cls_labels = list(support_by_class.columns)

# 'leaf_summary' já contém rule_text, support e as colunas por classe (os mesmos rótulos de cls_labels)
sup_arrays = [leaf_summary[c].to_numpy() for c in cls_labels]
labels     = [f"Classe {str(c)}" for c in cls_labels]

fig, ax = plt.subplots(figsize=(7.2, 6.5))
bp = ax.boxplot(
    sup_arrays,
    labels=labels,
    notch=True, patch_artist=True,
    boxprops=dict(facecolor="#4C78A8", alpha=0.55),
    medianprops=dict(color="black", linewidth=2),
    whiskerprops=dict(linewidth=1.4), capprops=dict(linewidth=1.4),
    flierprops=dict(marker='o', markersize=4, alpha=0.5)
)

# jitter para visualizar densidade em cada classe
rng = np.random.default_rng(42)
for i, arr in enumerate(sup_arrays, start=1):
    ax.scatter(rng.normal(i, 0.04, arr.size), arr, s=16, alpha=0.5, edgecolors="none")

ax.set_title("Box Plot — Suporte das Regras por Classe (y_train)")
ax.set_ylabel("Suporte (nº de instâncias)")
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_boxplot_suporte_por_classe.png"),
            dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# ==== Regras com valores reais + Suporte Treino/Teste + Plots Top-20 ====
import numpy as np, pandas as pd, re, os, textwrap
import matplotlib.pyplot as plt

TOP_N    = 20
X_cols   = list(feature_names)
OUT_BASE = PATH_FILE_OUT if 'PATH_FILE_OUT' in globals() else "."
os.makedirs(OUT_BASE, exist_ok=True)

# ---------- (1) Suporte por folha em TREINO e TESTE ----------
# usa o mesmo espaço de features do surrogate:
#   - treino: X_surrogate (== X_train_feat)
#   - teste:  X_test_feat  (features filtradas)
leaf_tr = surrogate_model.apply(X_surrogate)
leaf_te = surrogate_model.apply(X_test_feat)

sup_tr = pd.Series(leaf_tr).value_counts().rename("support_train")
sup_te = pd.Series(leaf_te).value_counts().rename("support_test")

leaf_sup = (
    pd.concat([sup_tr, sup_te], axis=1)
      .fillna(0).astype(int)
      .assign(support_total=lambda d: d["support_train"] + d["support_test"])
      .reset_index()
      .rename(columns={"index": "leaf_id"})
)

# Sanidade (tem que bater com tamanhos do split)
assert int(leaf_sup["support_train"].sum()) == len(X_surrogate), \
    f"Train support sum {leaf_sup['support_train'].sum()} != len(X_surrogate) {len(X_surrogate)}"
assert int(leaf_sup["support_test"].sum()) == len(X_test_feat), \
    f"Test support sum {leaf_sup['support_test'].sum()} != len(X_test_feat) {len(X_test_feat)}"

# ---------- (2) Extrai texto das regras por leaf_id (na escala do surrogate) ----------
def _leaf_rules_with_ids(estimator, feat_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    out = []

    def rec(node, cons):
        l, r = tree.children_left[node], tree.children_right[node]
        if l == r:
            parts = [f"{feat_names[f]} {op} {t:.6f}" for (f, op, t) in cons]
            out.append({
                "leaf_id": int(node),
                "rule_text": " AND ".join(parts) if parts else "(all)"
            })
        else:
            f, t = feat[node], thr[node]
            rec(l, cons + [(f, "<=", t)])
            rec(r, cons + [(f, ">",  t)])

    rec(0, [])
    return pd.DataFrame(out)

rules_ids = _leaf_rules_with_ids(surrogate_model, X_cols)

# ---------- (3) "Desnormaliza" thresholds usando X (dados brutos) ----------
# Xtr_scaled: espaço usado pelo surrogate (features filtradas)
Xtr_scaled = X_surrogate.copy()

# Xtr_raw: dados reais originais, alinhados pelos índices de X_train
try:
    Xtr_raw = X.loc[X_train.index, X_cols].copy()
except Exception as e:
    raise RuntimeError(
        "Erro ao montar Xtr_raw: verifique se 'X' contém as colunas de 'feature_names' "
        "e se os índices de X_train existem em X."
    ) from e

def _make_mapper(col, n_q=401):
    """Retorna função que mapeia threshold na escala 'scaled' -> valor 'real' por quantis (ordem preservada)."""
    xs = np.asarray(Xtr_scaled[col], dtype=float)
    xr = np.asarray(Xtr_raw[col],    dtype=float)
    if (not np.all(np.isfinite(xs))) or (not np.all(np.isfinite(xr))) or np.ptp(xs) == 0:
        return lambda t: float(t)
    qs = np.linspace(0.0, 1.0, n_q)
    vs = np.quantile(xs, qs)   # quantis na escala do surrogate
    vr = np.quantile(xr, qs)   # quantis no dado real

    def f(t):  # scaled -> raw
        return float(np.interp(t, vs, vr, left=vr[0], right=vr[-1]))
    return f

mappers = {c: _make_mapper(c) for c in X_cols}

def _to_raw_rule_text(txt):
    """
    Converte 'feat <= thr' na escala do surrogate para 'feat <= valor_real'.
    Binárias (0/1) são exibidas como 'feat = 0' ou 'feat = 1'.
    """
    parts_out = []
    for chunk in str(txt).split("AND"):
        s = chunk.strip()
        m = re.match(r'^(.+?)\s*(<=|>)\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)$', s)
        if not m:
            parts_out.append(s)
            continue

        feat, op, val_s = m.groups()
        feat = feat.strip()
        t_scaled = float(val_s)

        # Binária no dado REAL (0/1) -> mostrar '='
        xr = np.asarray(Xtr_raw[feat], dtype=float)
        uniq = np.unique(xr[np.isfinite(xr)])
        if uniq.size <= 2 and set(np.round(uniq).astype(int)).issubset({0, 1}):
            parts_out.append(f"{feat} = {'0' if op == '<=' else '1'}")
            continue

        # Contínua: aplica mapeamento scaled->real
        t_raw = mappers.get(feat, lambda z: z)(t_scaled)
        if np.isfinite(t_raw) and abs(t_raw - round(t_raw)) < 1e-6:
            val_fmt = f"{int(round(t_raw))}"
        else:
            val_fmt = f"{t_raw:.6f}"
        parts_out.append(f"{feat} {op} {val_fmt}")
    return " AND ".join(parts_out)

# ---------- (4) Tabela única com todas as regras (train+test) ----------
leaf_summary = leaf_sup.merge(rules_ids, on="leaf_id", how="left")
leaf_summary["rule_text_raw"] = leaf_summary["rule_text"].apply(_to_raw_rule_text)
leaf_summary["pct_total"] = 100.0 * leaf_summary["support_total"] / max(1, leaf_summary["support_total"].sum())
leaf_summary = leaf_summary.sort_values("support_total", ascending=False).reset_index(drop=True)

# Exporta a tabela completa (todas as folhas)
out_table = os.path.join(OUT_BASE, f"{NM_FILE_SURROGATE}_rules_train_test_RAW.xlsx")
leaf_summary[["leaf_id", "rule_text_raw", "support_train", "support_test", "support_total", "pct_total"]] \
    .to_excel(out_table, index=False)
print(f"📄 Tabela de regras (valores reais) salva em: {out_table}")

# ---------- (5) Gráfico 1 — Top-20 absoluto (treino+teste), regras em valores reais ----------
def _wrap(s, w=90):
    return "\n".join(textwrap.wrap(str(s), width=w))

total_all = int(leaf_summary["support_total"].sum())
topN = leaf_summary.head(TOP_N).copy()
labels = [f"#{i+1}: {_wrap(txt)}" for i, txt in enumerate(topN["rule_text_raw"], start=1)]

plt.figure(figsize=(12, 9))
bars = plt.barh(labels, topN["support_total"].values, edgecolor="black")
plt.gca().invert_yaxis()
plt.xlabel("Suporte (nº de instâncias)")
plt.title(f"Top-{TOP_N} Regras por Suporte (valores absolutos, regras em valores REAIS)")

# anota valor absoluto e %
for b, v, p in zip(bars, topN["support_total"].values, topN["pct_total"].values):
    y = b.get_y() + b.get_height() / 2
    plt.text(
        b.get_width() + max(0.5, total_all * 0.005),
        y,
        f"{int(v)}  ({p:.1f}%)",
        va="center",
        fontsize=9
    )

plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(
    os.path.join(OUT_BASE, f"{NM_FILE_SURROGATE}_top{TOP_N}_regras_abs_RAW.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()

# ---------- (6) Gráfico 2 — Top-20 empilhado (Treino vs Teste), regras em valores reais ----------
plt.figure(figsize=(12, 9))
y_pos = np.arange(len(topN))
plt.barh(y_pos, topN["support_train"].values, edgecolor="black", label="Treino")
plt.barh(
    y_pos,
    topN["support_test"].values,
    left=topN["support_train"].values,
    edgecolor="black",
    alpha=0.75,
    label="Teste"
)

plt.yticks(y_pos, labels)
plt.gca().invert_yaxis()
plt.xlabel("Suporte (nº de instâncias)")
plt.title(
    f"Top-{TOP_N} Regras por Suporte — Treino vs Teste "
    f"(valores absolutos, regras em valores REAIS)"
)

tot_stack = (topN["support_train"].values + topN["support_test"].values)
for yi, v in enumerate(tot_stack):
    plt.text(
        v + max(0.5, total_all * 0.005),
        yi,
        f"{int(v)}",
        va="center",
        fontsize=9
    )

plt.legend(loc="lower right")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(
    os.path.join(OUT_BASE, f"{NM_FILE_SURROGATE}_top{TOP_N}_regras_abs_stack_RAW.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# === TESTE APENAS: Regras (valores reais) + contagem por classe (0/1) e 2 plots lado a lado ===
import numpy as np, pandas as pd, re, os, textwrap
import matplotlib.pyplot as plt

TOP_N = 20
X_cols = list(feature_names)

# ---------- 1) Mapeamento folha no TESTE e classes ----------
# usa o espaço de features filtradas do surrogate
leaf_te = surrogate_model.apply(X_test_feat)  # id da folha por amostra de teste

# garante alinhamento por índice
df_te = pd.DataFrame({"leaf_id": leaf_te}, index=X_test_feat.index)

# y_test: tenta usar coluna 'y' do X_test, senão variável y_test global
if 'y' in X_test.columns:
    y_test_series = X_test['y'].reindex(X_test_feat.index)
elif 'y_test' in globals():
    y_test_series = pd.Series(y_test).reindex(X_test_feat.index)
else:
    raise RuntimeError("Não encontrei rótulos de teste: nem X_test['y'], nem variável y_test.")

df_te["y"] = y_test_series.values

# contagens por folha e classe
ct = df_te.groupby(["leaf_id", "y"]).size().unstack(fill_value=0)

# garante colunas 0 e 1, mesmo que falte alguma
for c in [0, 1]:
    if c not in ct.columns:
        ct[c] = 0
ct = ct[[0, 1]]  # ordem fixada

leaf_sup_test = (
    ct.rename(columns={0: "support_test_0", 1: "support_test_1"})
      .assign(support_test_total=lambda d: d["support_test_0"] + d["support_test_1"])
      .reset_index()
)

# sanidade: soma deve ser len(X_test_feat)
assert int(leaf_sup_test["support_test_total"].sum()) == len(X_test_feat), \
    f"Test support sum {leaf_sup_test['support_test_total'].sum()} != len(X_test_feat) {len(X_test_feat)}"

# ---------- 2) Extrai regras por leaf_id (na escala do surrogate) ----------
def _leaf_rules_with_ids(estimator, feat_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    out = []

    def rec(node, cons):
        l, r = tree.children_left[node], tree.children_right[node]
        if l == r:
            parts = [f"{feat_names[f]} {op} {t:.6f}" for (f, op, t) in cons]
            out.append({
                "leaf_id": int(node),
                "rule_text": " AND ".join(parts) if parts else "(all)"
            })
        else:
            f, t = feat[node], thr[node]
            rec(l, cons + [(f, "<=", t)])
            rec(r, cons + [(f, ">",  t)])

    rec(0, [])
    return pd.DataFrame(out)

rules_ids = _leaf_rules_with_ids(surrogate_model, X_cols)

# ---------- 3) Desnormaliza thresholds para valores REAIS (via quantile mapping) ----------
# escala do surrogate: mesmo espaço usado no treino
Xtr_scaled = X_surrogate.copy()

# escala real original (agora usando X em vez de database_corrigido)
try:
    Xtr_raw = X.loc[X_train.index, X_cols].copy()
except Exception as e:
    raise RuntimeError(
        "Erro ao montar Xtr_raw: verifique se 'X' contém as colunas de 'feature_names' "
        "e se os índices de X_train existem em X."
    ) from e

def _make_mapper(col, n_q=401):
    xs = np.asarray(Xtr_scaled[col], dtype=float)
    xr = np.asarray(Xtr_raw[col],    dtype=float)
    if (not np.all(np.isfinite(xs))) or (not np.all(np.isfinite(xr))) or np.ptp(xs) == 0:
        return lambda t: float(t)
    qs = np.linspace(0.0, 1.0, n_q)
    vs = np.quantile(xs, qs)   # quantis na escala surrogate
    vr = np.quantile(xr, qs)   # quantis na escala real
    def f(t):  # scaled -> real
        return float(np.interp(t, vs, vr, left=vr[0], right=vr[-1]))
    return f

mappers = {c: _make_mapper(c) for c in X_cols}

def _to_raw_rule_text(txt):
    parts_out = []
    for chunk in str(txt).split("AND"):
        s = chunk.strip()
        m = re.match(r'^(.+?)\s*(<=|>)\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)$', s)
        if not m:
            parts_out.append(s)
            continue
        feat, op, val_s = m.groups()
        feat = feat.strip()
        t_scaled = float(val_s)

        # binárias 0/1 no dado real
        xr = np.asarray(Xtr_raw[feat], dtype=float)
        uniq = np.unique(xr[np.isfinite(xr)])
        if uniq.size <= 2 and set(np.round(uniq).astype(int)).issubset({0, 1}):
            parts_out.append(f"{feat} = {'0' if op == '<=' else '1'}")
            continue

        t_raw = mappers.get(feat, lambda z: z)(t_scaled)
        if np.isfinite(t_raw) and abs(t_raw - round(t_raw)) < 1e-6:
            val_fmt = f"{int(round(t_raw))}"
        else:
            val_fmt = f"{t_raw:.6f}"
        parts_out.append(f"{feat} {op} {val_fmt}")
    return " AND ".join(parts_out)

# ---------- 4) Tabela final (TESTE), já com regras em valores reais ----------
leaf_test_summary = leaf_sup_test.merge(rules_ids, on="leaf_id", how="left")
leaf_test_summary["rule_text_raw"] = leaf_test_summary["rule_text"].apply(_to_raw_rule_text)
leaf_test_summary = leaf_test_summary.sort_values("support_test_total", ascending=False).reset_index(drop=True)

# exporta a tabela completa do teste
out_xlsx = os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_rules_TEST_only_RAW.xlsx")
leaf_test_summary[["leaf_id", "rule_text_raw", "support_test_0", "support_test_1", "support_test_total"]] \
    .to_excel(out_xlsx, index=False)
print(f"📄 Tabela de regras (TESTE, valores reais) salva em: {out_xlsx}")

# ---------- 5) Plots lado a lado (Classe 0 vs Classe 1), mostrando as REGRAS ----------
def _wrap(s, w=90):
    return "\n".join(textwrap.wrap(str(s), width=w))

topN = leaf_test_summary.head(TOP_N).copy()
labels = [f"#{i+1}: {_wrap(txt)}" for i, txt in enumerate(topN["rule_text_raw"], start=1)]

fig, axes = plt.subplots(ncols=2, nrows=1, figsize=(16, 9), sharey=True)
ax0, ax1 = axes

# Classe 0
bars0 = ax0.barh(labels, topN["support_test_0"].values, edgecolor="black")
ax0.invert_yaxis()
ax0.set_xlabel("Suporte em TESTE (Classe 0)")
ax0.set_title("Top-N Regras (TESTE) — Classe 0")
for b, v in zip(bars0, topN["support_test_0"].values):
    ax0.text(
        b.get_width() + max(0.2, len(X_test_feat) * 0.003),
        b.get_y() + b.get_height() / 2,
        f"{int(v)}",
        va="center",
        fontsize=9
    )
ax0.grid(True, axis="x", linestyle="--", alpha=0.5)

# Classe 1
bars1 = ax1.barh(labels, topN["support_test_1"].values, edgecolor="black")
ax1.set_xlabel("Suporte em TESTE (Classe 1)")
ax1.set_title("Top-N Regras (TESTE) — Classe 1")
for b, v in zip(bars1, topN["support_test_1"].values):
    ax1.text(
        b.get_width() + max(0.2, len(X_test_feat) * 0.003),
        b.get_y() + b.get_height() / 2,
        f"{int(v)}",
        va="center",
        fontsize=9
    )
ax1.grid(True, axis="x", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig(
    os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_TEST_top{TOP_N}_classe0_vs_classe1_RAW.png"),
    dpi=180,
    bbox_inches="tight"
)
plt.show()


In [ ]:
# === 6.1) Box plot do erro de aproximação (com quantis anotados) ===
err = df_surrogate["erro_aproximacao"].to_numpy()

if err.size == 0:
    print("⚠️ df_surrogate está vazio: nenhum erro de aproximação para plotar.")
else:
    # Quantis e whiskers (regra 1.5*IQR)
    q1, med, q3 = np.quantile(err, [0.25, 0.5, 0.75])
    iqr = q3 - q1
    low_whisk = np.min(err[err >= (q1 - 1.5*iqr)]) if err.size else np.nan
    up_whisk  = np.max(err[err <= (q3 + 1.5*iqr)]) if err.size else np.nan
    p5, p95   = np.quantile(err, [0.05, 0.95])

    # Resumo no console
    print({
        "n": int(err.size),
        "mean": float(err.mean()),
        "std": float(err.std(ddof=1)),
        "p05": float(p5),
        "q1": float(q1),
        "median": float(med),
        "q3": float(q3),
        "p95": float(p95),
        "IQR": float(iqr),
        "whiskers": (float(low_whisk), float(up_whisk)),
    })

    # Box + jitter
    fig, ax = plt.subplots(figsize=(4.2, 7))
    bp = ax.boxplot(
        err,
        vert=True, notch=True, patch_artist=True,
        boxprops=dict(facecolor="#4C78A8", alpha=0.55),
        medianprops=dict(color="black", linewidth=2),
        whiskerprops=dict(linewidth=1.4),
        capprops=dict(linewidth=1.4),
        flierprops=dict(marker='o', markersize=4, alpha=0.5)
    )

    # Pontos com leve jitter para ver densidade
    rng = np.random.default_rng(42)
    xj = rng.normal(loc=1.0, scale=0.03, size=err.size)
    ax.scatter(xj, err, s=18, alpha=0.55, edgecolors="none")

    # Linhas/labels dos quantis
    for y, label in [
        (q1, "Q1"),
        (med, "Q2 (mediana)"),
        (q3, "Q3"),
        (low_whisk, "Whisker inf."),
        (up_whisk, "Whisker sup."),
    ]:
        ax.axhline(y, color="gray", linestyle="--", linewidth=0.8)
        ax.text(1.13, y, f"{label}: {y:.3f}", va="center", fontsize=9)

    ax.set_title(
        f"Erro de Aproximação |RF - árvore| — Box Plot\n"
        f"(n={err.size}, média={err.mean():.3f}, IQR={iqr:.3f})"
    )
    ax.set_ylabel("Erro absoluto")
    ax.set_xticks([])
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_boxplot_erro.png"), dpi=180)
    plt.show()

    # === 6.2) (Opcional) Box plot por classe real ===
    # 'tamanho_real' já foi criado como 'pequeno' (0) e 'grande' (1) no seu loop
    if "tamanho_real" in df_surrogate.columns:
        dados_cls0 = df_surrogate.loc[
            df_surrogate["tamanho_real"] == "pequeno", "erro_aproximacao"
        ].to_numpy()
        dados_cls1 = df_surrogate.loc[
            df_surrogate["tamanho_real"] == "grande", "erro_aproximacao"
        ].to_numpy()

        # só plota se tiver pelo menos 1 ponto em alguma classe
        if dados_cls0.size > 0 or dados_cls1.size > 0:
            fig, ax = plt.subplots(figsize=(6.5, 6.5))
            bp = ax.boxplot(
                [dados_cls0, dados_cls1],
                labels=["Classe 0 (pequeno)", "Classe 1 (grande)"],
                notch=True, patch_artist=True,
                boxprops=dict(facecolor="#4C78A8", alpha=0.55),
                medianprops=dict(color="black", linewidth=2),
                whiskerprops=dict(linewidth=1.4),
                capprops=dict(linewidth=1.4),
                flierprops=dict(marker='o', markersize=4, alpha=0.5)
            )

            # jitter por classe (reutiliza o mesmo rng para consistência)
            xj0 = rng.normal(loc=1.0, scale=0.04, size=dados_cls0.size)
            xj1 = rng.normal(loc=2.0, scale=0.04, size=dados_cls1.size)
            ax.scatter(xj0, dados_cls0, s=16, alpha=0.5, edgecolors="none")
            ax.scatter(xj1, dados_cls1, s=16, alpha=0.5, edgecolors="none")

            ax.set_title("Erro de Aproximação por Classe (Box Plot + pontos)")
            ax.set_ylabel("Erro absoluto |RF - árvore|")
            ax.grid(axis="y", linestyle="--", alpha=0.5)
            plt.tight_layout()
            plt.savefig(
                os.path.join(PATH_FILE_OUT, f"{NM_FILE_SURROGATE}_boxplot_erro_por_classe.png"),
                dpi=180
            )
            plt.show()
        else:
            print("⚠️ Sem dados em 'tamanho_real' para plotar boxplot por classe.")


In [ ]:
# ===================== (C) Top-20 com VALORES REAIS nas regras — SELECIONADOS =====================

import os, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------- CONFIG BÁSICA ----------------------
TOP_N = 20

# garante lista de features usadas pelo surrogate
X_cols = list(feature_names)

# garante diretório de saída local para os SELECIONADOS
if "PATH_FILE_OUT" in globals():
    base_out = PATH_FILE_OUT
elif "BASE_PATH" in globals():
    base_out = BASE_PATH
else:
    base_out = "."

out_dir_sel = os.path.join(base_out, f"{NM_FILE_SURROGATE}_SEL")
os.makedirs(out_dir_sel, exist_ok=True)

# ---------------------- ÍNDICES SELECIONADOS EM X ----------------------
if "ENTRADAS_SELECIONADAS" not in globals():
    raise RuntimeError("ENTRADAS_SELECIONADAS não está definido.")

# --- garantir que X.index seja o 'id' (labels) ---
if "id" in X.columns and X.index.name != "id":
    X = X.set_index("id", drop=False)

# --- tratar ENTRADAS_SELECIONADAS sempre como labels (ids) de X ---
idx_all = pd.Index(ENTRADAS_SELECIONADAS)

# tentar coerção de tipo para ficar igual ao tipo do índice de X
try:
    idx_all = idx_all.astype(X.index.dtype)
except TypeError:
    pass

# ids presentes em X
idx_sel_X = X.index.intersection(idx_all)
if idx_sel_X.empty:
    raise RuntimeError(
        "Nenhum índice de ENTRADAS_SELECIONADAS foi encontrado em X.index.\n"
        f"Exemplo de X.index: {list(X.index[:10])}\n"
        f"Exemplo de ENTRADAS_SELECIONADAS: {list(ENTRADAS_SELECIONADAS[:10])}"
    )

# ===================== PONTO CRÍTICO =====================
# X_surrogate NÃO tem id no índice → precisamos usar POSIÇÕES
# assumindo que X_surrogate está alinhado à mesma ordem de X usada para criá-lo
# =========================================================

# mapeia cada id selecionado para a POSIÇÃO inteira em X
pos_all = [X.index.get_loc(i) for i in idx_sel_X]

# mantém só as posições que existem em X_surrogate
pos_sel = [p for p in pos_all if p < len(X_surrogate)]

if not pos_sel:
    raise RuntimeError(
        "Não foi possível mapear os ids selecionados para posições válidas em X_surrogate.\n"
        f"len(X_surrogate)={len(X_surrogate)}\n"
        f"Ids em X (idx_sel_X): {list(idx_sel_X[:10])}"
    )

# ids efetivamente usados (apenas aqueles que tiveram posição válida)
idx_sel = X.index[pos_sel]

if len(idx_sel) < len(idx_sel_X):
    print(
        f"⚠️ Aviso: apenas {len(idx_sel)} de {len(idx_sel_X)} ids selecionados "
        "foram mapeados para X_surrogate (demais ficaram fora do range)."
    )

# espaço REAL (não normalizado) — usa ids em comum
X_sel_raw = X.loc[idx_sel, X_cols].copy()

# --- garantir que X_surrogate tenha as colunas usadas pelo surrogate ---
cols_sur = [c for c in X_cols if c in X_surrogate.columns]
if len(cols_sur) < len(X_cols):
    print(
        f"⚠️ Aviso: apenas {len(cols_sur)} de {len(X_cols)} colunas de X_cols "
        "estão presentes em X_surrogate. As demais serão ignoradas."
    )

# espaço do surrogate (normalizado) usando POSIÇÃO
X_sel_scaled = X_surrogate.iloc[pos_sel][cols_sur].copy()

# rótulos (y) a partir de X
if "y" in X.columns:
    y_base = X["y"]
elif "classe" in X.columns:
    y_base = X["classe"]
else:
    raise RuntimeError("Nem X['y'] nem X['classe'] estão definidos para montar y_SEL.")

y_SEL = y_base.loc[idx_sel]

# ---------------------- MAPEAMENTO scaled -> real USANDO X COMPLETO ----------------------
# usa TODAS as linhas de X_surrogate e X para o mapeamento por quantis
Xtr_scaled = X_surrogate[cols_sur].copy()
Xtr_raw    = X[X_cols].copy()

def _make_mapper(col, n_q=401):
    xs = np.asarray(Xtr_scaled[col], dtype=float)
    xr = np.asarray(Xtr_raw[col],    dtype=float)
    if (not np.all(np.isfinite(xs))) or (not np.all(np.isfinite(xr))) or np.ptp(xs) == 0:
        # se tiver NaN demais ou coluna constante em xs, não faz mapeamento
        return lambda t: float(t)
    qs = np.linspace(0.0, 1.0, n_q)
    vs = np.quantile(xs, qs)
    vr = np.quantile(xr, qs)
    def f(t):
        return float(np.interp(t, vs, vr, left=vr[0], right=vr[-1]))
    return f

mappers = {c: _make_mapper(c) for c in cols_sur}

def _to_raw_rule_text(txt):
    parts_out = []
    for chunk in str(txt).split("AND"):
        s = chunk.strip()
        m = re.match(r'^(.+?)\s*(<=|>)\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)$', s)
        if not m:
            parts_out.append(s)
            continue

        feat, op, val_s = m.groups()
        feat = feat.strip()
        t_scaled = float(val_s)

        # se a feature não existir em Xtr_raw (pode ser alguma coluna do surrogate diferente), mantém texto
        if feat not in Xtr_raw.columns:
            parts_out.append(s)
            continue

        # verifica se a feature é binária (0/1) no dado REAL (Xtr_raw)
        xr = np.asarray(Xtr_raw[feat], dtype=float)
        uniq = np.unique(xr[np.isfinite(xr)])
        if uniq.size <= 2 and set(np.round(uniq).astype(int)).issubset({0, 1}):
            # para binária, mostra como igualdade
            parts_out.append(f"{feat} = {'0' if op == '<=' else '1'}")
            continue

        # contínua: aplica mapping scaled -> real
        t_raw = mappers.get(feat, lambda z: z)(t_scaled)
        if np.isfinite(t_raw) and abs(t_raw - round(t_raw)) < 1e-6:
            val_fmt = f"{int(round(t_raw))}"
        else:
            val_fmt = f"{t_raw:.6f}"
        parts_out.append(f"{feat} {op} {val_fmt}")
    return " AND ".join(parts_out)

# ---------------------- SUPORTE POR FOLHA (APENAS SUBSET) ----------------------
# aplica surrogate no espaço normalizado dos selecionados
leaf_sel = surrogate_model.apply(X_sel_scaled[cols_sur])
df_sel  = pd.DataFrame({"leaf_id": leaf_sel, "y": pd.Series(y_SEL).values}, index=idx_sel)

ct_sel = df_sel.groupby(["leaf_id", "y"]).size().unstack(fill_value=0)
for c in [0, 1]:
    if c not in ct_sel.columns:
        ct_sel[c] = 0
ct_sel = ct_sel[[0, 1]]

leaf_sup_SEL = (
    ct_sel
    .rename(columns={0: "support_sel_0", 1: "support_sel_1"})
    .assign(support_sel_total=lambda d: d["support_sel_0"] + d["support_sel_1"])
    .reset_index()
)

# ---------------------- TABELA DE REGRAS + TEXTO REAL ----------------------
def _leaf_rules_with_ids(estimator, feature_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    out = []
    def rec(node, cons):
        l, r = tree.children_left[node], tree.children_right[node]
        if l == r:
            parts = [f"{feature_names[f]} {op} {t:.6f}" for f, op, t in cons]
            out.append(
                {
                    "leaf_id": int(node),
                    "rule_text": " AND ".join(parts) if parts else "(all)",
                }
            )
        else:
            f, t = feat[node], thr[node]
            rec(l, cons + [(f, "<=", t)])
            rec(r, cons + [(f, ">",  t)])
    rec(0, [])
    return pd.DataFrame(out)

rules_ids = _leaf_rules_with_ids(surrogate_model, X_cols)

leaf_summary_real_SEL = leaf_sup_SEL.merge(rules_ids, on="leaf_id", how="left")
leaf_summary_real_SEL["rule_text_raw"] = leaf_summary_real_SEL["rule_text"].apply(_to_raw_rule_text)
leaf_summary_real_SEL["pct_sel"] = (
    100.0 * leaf_summary_real_SEL["support_sel_total"] /
    max(1, int(leaf_summary_real_SEL["support_sel_total"].sum()))
)
leaf_summary_real_SEL = leaf_summary_real_SEL.sort_values(
    "support_sel_total", ascending=False
).reset_index(drop=True)

# ---------------------- EXPORTA TABELA (SUBSET) ----------------------
out_table_SEL = os.path.join(out_dir_sel, f"{NM_FILE_SURROGATE}_rules_SEL_RAW.xlsx")
leaf_summary_real_SEL[
    ["leaf_id", "rule_text_raw", "support_sel_0", "support_sel_1", "support_sel_total", "pct_sel"]
].to_excel(out_table_SEL, index=False)
print(f"📄 [SEL] Tabela de regras (valores reais) salva em: {out_table_SEL}")

# ---------------------- PLOTS ----------------------
def _wrap(s, w=90):
    return "\n".join(textwrap.wrap(str(s), width=w))

# Plot 1 — Top-20 (total no subset)
topN_real_SEL = leaf_summary_real_SEL.head(TOP_N).copy()
labels = [f"#{i+1}: {_wrap(txt)}" for i, txt in enumerate(topN_real_SEL["rule_text_raw"], start=1)]
total_sel = int(leaf_summary_real_SEL["support_sel_total"].sum())

plt.figure(figsize=(12, 9))
bars = plt.barh(labels, topN_real_SEL["support_sel_total"].values, edgecolor="black")
plt.gca().invert_yaxis()
plt.xlabel("Suporte (nº de instâncias) — SELECIONADOS")
plt.title(f"Top-{TOP_N} Regras por Suporte (VALORES REAIS) — SELECIONADOS")
for b, v, p in zip(bars, topN_real_SEL["support_sel_total"].values, topN_real_SEL["pct_sel"].values):
    y_txt = b.get_y() + b.get_height() / 2
    plt.text(
        b.get_width() + max(0.5, total_sel * 0.005),
        y_txt,
        f"{int(v)}  ({p:.1f}%)",
        va="center",
        fontsize=9
    )
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(out_dir_sel, f"{NM_FILE_SURROGATE}_top{TOP_N}_regras_abs_RAW_SEL.png"),
            dpi=180, bbox_inches="tight")
plt.show()

# Plot 2 — Empilhado por classe (subset)
y_pos = np.arange(len(topN_real_SEL))
plt.figure(figsize=(12, 9))
plt.barh(y_pos, topN_real_SEL["support_sel_0"].values, edgecolor="black", label="Classe 0")
plt.barh(
    y_pos,
    topN_real_SEL["support_sel_1"].values,
    left=topN_real_SEL["support_sel_0"].values,
    edgecolor="black",
    alpha=0.75,
    label="Classe 1"
)
plt.yticks(y_pos, labels)
plt.gca().invert_yaxis()
plt.xlabel("Suporte (nº de instâncias) — SELECIONADOS")
plt.title(f"Top-{TOP_N} Regras — Classe 0 vs Classe 1 (VALORES REAIS) — SELECIONADOS")
tot_stack = (
    topN_real_SEL["support_sel_0"].values +
    topN_real_SEL["support_sel_1"].values
)
for yi, v in enumerate(tot_stack):
    plt.text(
        v + max(0.5, total_sel * 0.005),
        yi,
        f"{int(v)}",
        va="center",
        fontsize=9
    )
plt.legend(loc="lower right")
plt.grid(True, axis="x", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig(os.path.join(out_dir_sel, f"{NM_FILE_SURROGATE}_top{TOP_N}_regras_abs_stack_RAW_SEL.png"),
            dpi=180, bbox_inches="tight")
plt.show()


In [ ]:
# === Comparativo de suporte por regra (barras horizontais) ===
import os, re, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------- CONFIG -------------------------
TOP_N   = 20
X_cols  = list(feature_names)

# diretório de saída
if "PATH_FILE_OUT" in globals():
    OUT_DIR = PATH_FILE_OUT
elif "BASE_PATH" in globals():
    OUT_DIR = BASE_PATH
else:
    OUT_DIR = "."
os.makedirs(OUT_DIR, exist_ok=True)

# --- garantir que X.index seja o 'id' (labels) ---
if "id" in X.columns and X.index.name != "id":
    X = X.set_index("id", drop=False)

# --- garantir colunas existentes no surrogate ---
cols_sur = [c for c in X_cols if c in X_surrogate.columns]
if len(cols_sur) < len(X_cols):
    print(
        f"⚠️ Aviso: apenas {len(cols_sur)} de {len(X_cols)} colunas de X_cols "
        "estão presentes em X_surrogate. As demais serão ignoradas."
    )

# ------------------------- 1) Suportes por folha: TOTAL -------------------------
# usa o mesmo espaço de features do surrogate (todas as instâncias)
X_sur_all = X_surrogate[cols_sur].copy()
leaf_all  = surrogate_model.apply(X_sur_all)        # todas as linhas de X_surrogate

sup_total = pd.Series(leaf_all).value_counts().rename("support_total")

leaf_sup_all = (
    sup_total
    .to_frame()
    .reset_index()
    .rename(columns={"index": "leaf_id"})
)

# ------------------------- 2) Suporte dos SELECIONADOS (se existir) -------------------------
# ENTRADAS_SELECIONADAS deve conter ÍNDICES (labels/id) do X
try:
    ENTRADAS_SELECIONADAS
    _has_sel = (ENTRADAS_SELECIONADAS is not None) and (len(ENTRADAS_SELECIONADAS) > 0)
except NameError:
    _has_sel = False

if _has_sel:
    idx_all = pd.Index(ENTRADAS_SELECIONADAS)

    # tentar coerção de tipo para o mesmo tipo do índice de X
    try:
        idx_all = idx_all.astype(X.index.dtype)
    except TypeError:
        pass

    # ids presentes em X
    idx_sel_X = X.index.intersection(idx_all)
    if idx_sel_X.empty:
        print(
            "⚠️ Nenhum índice de ENTRADAS_SELECIONADAS foi encontrado em X.index.\n"
            f"Exemplo de X.index: {list(X.index[:10])}\n"
            f"Exemplo de ENTRADAS_SELECIONADAS: {list(ENTRADAS_SELECIONADAS)[:10]}"
        )
        sup_sel = pd.Series(dtype=int).rename("support_selected")
    else:
        # mapeia ids -> posições em X
        pos_all = [X.index.get_loc(i) for i in idx_sel_X]

        # mantém apenas posições válidas dentro de X_surrogate
        pos_sel = [p for p in pos_all if p < len(X_surrogate)]

        if not pos_sel:
            print(
                "⚠️ Nenhum dos ids selecionados pôde ser mapeado para posições válidas em X_surrogate.\n"
                f"len(X_surrogate) = {len(X_surrogate)}\n"
                f"Ids em X (idx_sel_X): {list(idx_sel_X[:10])}"
            )
            sup_sel = pd.Series(dtype=int).rename("support_selected")
        else:
            # subset normalizado no surrogate usando POSIÇÕES
            X_sel_scaled = X_surrogate.iloc[pos_sel][cols_sur].copy()
            leaf_sel     = surrogate_model.apply(X_sel_scaled)
            sup_sel      = pd.Series(leaf_sel).value_counts().rename("support_selected")
else:
    sup_sel = pd.Series(dtype=int).rename("support_selected")

# junta total + selecionados (onde existir)
leaf_sup_all = (
    leaf_sup_all
    .merge(
        sup_sel.to_frame(),
        left_on="leaf_id",
        right_index=True,
        how="left"
    )
    .fillna(0)
    .astype({"support_total": int, "support_selected": int})
)

# ------------------------- 3) Texto das regras e dessensibilização (valores reais) -------------------------
def _leaf_rules_with_ids(estimator, feat_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    out  = []

    def rec(node, cons):
        l, r = tree.children_left[node], tree.children_right[node]
        if l == r:  # folha
            parts = [f"{feat_names[f]} {op} {t:.6f}" for (f, op, t) in cons]
            out.append({
                "leaf_id": int(node),
                "rule_text": " AND ".join(parts) if parts else "(all)"
            })
        else:
            f, t = feat[node], thr[node]
            rec(l, cons + [(f, "<=", t)])
            rec(r, cons + [(f, ">",  t)])

    rec(0, [])
    return pd.DataFrame(out)

rules_ids = _leaf_rules_with_ids(surrogate_model, X_cols)

# mapeamento scaled->real via quantis (ordem preservada)
# escala surrogate = X_surrogate; escala real = X (dataset bruto NÃO normalizado)
Xtr_scaled = X_surrogate[cols_sur].copy()
Xtr_raw    = X[X_cols].copy()

def _make_mapper(col, n_q=401):
    xs = np.asarray(Xtr_scaled[col], dtype=float)
    xr = np.asarray(Xtr_raw[col],    dtype=float)
    if (not np.all(np.isfinite(xs))) or (not np.all(np.isfinite(xr))) or np.ptp(xs) == 0:
        # muito NaN ou coluna constante => não mapeia
        return lambda t: float(t)
    qs = np.linspace(0.0, 1.0, n_q)
    vs = np.quantile(xs, qs)
    vr = np.quantile(xr, qs)
    def f(t):
        return float(np.interp(t, vs, vr, left=vr[0], right=vr[-1]))
    return f

mappers = {c: _make_mapper(c) for c in cols_sur}

def _to_raw_rule_text(txt):
    parts_out = []
    for chunk in str(txt).split("AND"):
        s = chunk.strip()
        m = re.match(r'^(.+?)\s*(<=|>)\s*([+-]?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?)$', s)
        if not m:
            parts_out.append(s)
            continue
        feat, op, val_s = m.groups()
        feat = feat.strip()
        t_scaled = float(val_s)

        # se a feature não existir em Xtr_raw, mantém texto bruto
        if feat not in Xtr_raw.columns:
            parts_out.append(s)
            continue

        xr = np.asarray(Xtr_raw[feat], dtype=float)
        uniq = np.unique(xr[np.isfinite(xr)])
        # binárias 0/1 no dado real
        if uniq.size <= 2 and set(np.round(uniq).astype(int)).issubset({0, 1}):
            parts_out.append(f"{feat} = {'0' if op == '<=' else '1'}")
            continue

        t_raw = mappers.get(feat, lambda z: z)(t_scaled)
        val_fmt = (
            f"{int(round(t_raw))}"
            if np.isfinite(t_raw) and abs(t_raw - round(t_raw)) < 1e-6
            else f"{t_raw:.6f}"
        )
        parts_out.append(f"{feat} {op} {val_fmt}")
    return " AND ".join(parts_out)

# leaf_summary com valores reais nas regras
leaf_summary = leaf_sup_all.merge(rules_ids, on="leaf_id", how="left")
leaf_summary["rule_text_raw"] = leaf_summary["rule_text"].apply(_to_raw_rule_text)

# ------------------------- 4) Função genérica para plotar comparativo -------------------------
def _wrap(s, w=90):
    return "\n".join(textwrap.wrap(str(s), width=w))

def plot_comparativo(leaf_df, base_col, comp_col, title, filename, top_n=TOP_N):
    base_total = int(leaf_df[base_col].sum())
    comp_total = int(leaf_df[comp_col].sum()) if comp_col in leaf_df.columns else 0

    # ordena pelas regras mais “relevantes” do conjunto base
    top = leaf_df.sort_values(base_col, ascending=False).head(top_n).copy()

    labels = [f"#{i+1}: {_wrap(txt)}" for i, txt in enumerate(top["rule_text_raw"], start=1)]
    y = np.arange(len(top))
    h = 0.38  # altura de cada barra

    fig, ax = plt.subplots(figsize=(12, 9))

    # barra base (Total)
    ax.barh(
        y - (h/2 if comp_col in leaf_df.columns else 0),
        top[base_col].values,
        height=h,
        edgecolor="black",
        label=f"Total ({base_col})"
    )

    # barra comparativa (Selected), se existir
    if comp_col in leaf_df.columns:
        ax.barh(
            y + h/2,
            top[comp_col].values,
            height=h,
            edgecolor="black",
            alpha=0.75,
            label=f"Selecionados ({comp_col})"
        )

    # eixos/labels
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Suporte (nº de instâncias)")
    ax.set_title(title)
    ax.grid(True, axis="x", linestyle="--", alpha=0.5)
    if comp_col in leaf_df.columns:
        ax.legend(loc="lower right")

    # anota contagem + % relativa a cada total
    for yi, b in enumerate(top[base_col].values):
        ax.text(
            b + max(0.5, base_total * 0.005),
            yi - (h/2 if comp_col in leaf_df.columns else 0),
            f"{int(b)}  ({100 * b / base_total:.1f}%)",
            va="center",
            fontsize=9
        )

    if comp_col in leaf_df.columns and comp_total > 0:
        for yi, c in enumerate(top[comp_col].values):
            ax.text(
                c + max(0.5, comp_total * 0.005),
                yi + h/2,
                f"{int(c)}  ({100 * c / comp_total:.1f}%)",
                va="center",
                fontsize=9
            )

    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, filename), dpi=180, bbox_inches="tight")
    plt.show()

# ------------------------- 5) Plot — Total vs Selecionados (se houver) -------------------------
if _has_sel and not sup_sel.empty:
    plot_comparativo(
        leaf_summary,
        base_col="support_total",
        comp_col="support_selected",
        title=(
            "Comparativo por Regra — Total (todas as instâncias) vs Selecionados\n"
            "(valores absolutos e percentuais por conjunto)"
        ),
        filename=f"{NM_FILE_SURROGATE}_comparativo_TOTAL_vs_SELECIONADOS_RAW.png",
    )
else:
    # se não houver selecionados, faz apenas gráfico para o total
    plot_comparativo(
        leaf_summary,
        base_col="support_total",
        comp_col="__none__",
        title="Suporte por Regra — Total (todas as instâncias)",
        filename=f"{NM_FILE_SURROGATE}_comparativo_TOTAL_RAW.png",
    )


In [ ]:
# === EXPLICAÇÃO LOCAL COM O SURROGATE GLOBAL — 50% da largura ===
import os, textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

OUT_DIR = PATH_FILE_OUT if 'PATH_FILE_OUT' in globals() else "."
os.makedirs(OUT_DIR, exist_ok=True)

# -------- seleção ordenada (posição, id ou índice) --------
def select_rows_ordered(X_test: pd.DataFrame, selection):
    sel = list(selection if isinstance(selection, (list, tuple, np.ndarray, pd.Index)) else [selection])

    # 1) TENTAR COMO POSIÇÃO (iloc)
    if all(isinstance(x, (int, np.integer)) for x in sel):
        n = len(X_test)
        if all(0 <= int(x) < n for x in sel):
            df_sel = X_test.iloc[sel].copy()
            meta = pd.DataFrame({
                "ordem": np.arange(len(sel)),
                "chave_entrada": sel,
                "tipo_selecao": "posicao",
                "posicao": sel,
                "indice_Xtest": list(df_sel.index)
            })
            return df_sel, meta

    # 2) TENTAR COMO COLUNA 'id'
    if "id" in X_test.columns:
        posicoes = []
        chaves_ok = []
        not_found = []
        id_values = np.asarray(X_test["id"].values)

        for key in sel:
            idx_match = np.flatnonzero(id_values == key)
            if idx_match.size > 0:
                pos = int(idx_match[0])
                posicoes.append(pos)
                chaves_ok.append(key)
            else:
                not_found.append(key)

        if posicoes:
            df_sel = X_test.iloc[posicoes].copy()
            meta = pd.DataFrame({
                "ordem": np.arange(len(posicoes)),
                "chave_entrada": chaves_ok,
                "tipo_selecao": "id",
                "posicao": posicoes,
                "indice_Xtest": list(df_sel.index)
            })
            if not_found:
                print(f"⚠️ Aviso: chaves (id) inexistentes em X_test ignoradas: {not_found}")
            return df_sel, meta
        else:
            print(f"⚠️ Nenhuma chave de seleção encontrada na coluna 'id' de X_test. sel={sel[:10]}")

    # 3) TENTAR COMO ÍNDICE (label)
    idx_order, not_found = [], []
    for key in sel:
        if key in X_test.index:
            idx_order.append(key)
        else:
            not_found.append(key)

    df_sel = X_test.loc[idx_order].copy()
    meta = pd.DataFrame({
        "ordem": np.arange(len(idx_order)),
        "chave_entrada": idx_order,
        "tipo_selecao": "indice",
        "posicao": [X_test.index.get_loc(k) for k in idx_order],
        "indice_Xtest": idx_order
    })

    if not_found:
        print(f"⚠️ Aviso: chaves inexistentes em X_test ignoradas: {not_found}")
    return df_sel, meta

# -------- utilitários da árvore --------
def _layout_positions(tree):
    left, right = tree.children_left, tree.children_right
    x_pos, y_pos, leaf_counter = {}, {}, [0]

    def is_leaf(n): return left[n] == right[n]

    def dfs(n, d):
        if is_leaf(n):
            x = leaf_counter[0]
            leaf_counter[0] += 1
        else:
            dfs(left[n], d + 1)
            dfs(right[n], d + 1)
            x = (x_pos[left[n]] + x_pos[right[n]]) / 2.0
        x_pos[n] = x
        y_pos[n] = -d

    dfs(0, 0)
    return x_pos, y_pos, leaf_counter[0]

def _path_nodes(estimator, x_row: pd.Series, feature_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    x = x_row.values if hasattr(x_row, "values") else np.asarray(x_row)
    node = 0
    path = [0]
    while tree.children_left[node] != tree.children_right[node]:
        f_idx = feat[node]
        go_left = (x[f_idx] <= thr[node])
        node = int(tree.children_left[node] if go_left else tree.children_right[node])
        path.append(node)
    return path

def _path_as_text(estimator, x_row: pd.Series, feature_names):
    tree = estimator.tree_
    feat = tree.feature
    thr  = tree.threshold
    x = x_row.values if hasattr(x_row, "values") else np.asarray(x_row)
    node = 0
    parts = []
    while tree.children_left[node] != tree.children_right[node]:
        f_idx = feat[node]
        t = thr[node]
        fname = feature_names[int(f_idx)]
        go_left = (x[f_idx] <= t)
        parts.append(f"{fname} ≤ {t:.6f}" if go_left else f"{fname} > {t:.6f}")
        node = int(tree.children_left[node] if go_left else tree.children_right[node])
    return " AND ".join(parts)

def _node_label(tree, node_id, feature_names):
    feat = tree.feature
    thr  = tree.threshold
    n    = int(tree.n_node_samples[node_id])
    mean = float(tree.value[node_id][0, 0])
    imp  = float(tree.impurity[node_id])  # MSE
    if tree.children_left[node_id] != tree.children_right[node_id]:
        return f"{feature_names[int(feat[node_id])]} ≤ {thr[node_id]:.6f}\nsamples={n} | mse={imp:.3f}"
    else:
        return f"LEAF\nsamples={n}\nmean={mean:.4f}"

# -------- plot compacto + escala de largura --------
def plot_tree_with_path(
    estimator,
    x_row: pd.Series,
    instance_name="instância",
    save_path=None,
    feature_names=None,
    teacher_model=None,
    *,
    FIG_WIDTH_SCALE=0.50,  # usa 50% da largura calculada (0.5 = metade)
    SLOPE_BOOST=1.10,      # aumenta DY (linhas mais inclinadas)
    FONT_SIZE=11.8,
    BOX_W=2.05, BOX_H=0.98,
    H_GAP=0.40,            # gap mínimo entre caixas (anti-overlap)
):
    tree = estimator.tree_

    # ---- layout de nós ----
    def _layout_positions(tree):
        left, right = tree.children_left, tree.children_right
        x_pos, y_pos, leaf_counter = {}, {}, [0]

        def is_leaf(n): 
            return left[n] == right[n]

        def dfs(n, d):
            if is_leaf(n):
                x = leaf_counter[0]
                leaf_counter[0] += 1
            else:
                dfs(left[n], d + 1)
                dfs(right[n], d + 1)
                x = (x_pos[left[n]] + x_pos[right[n]]) / 2.0
            x_pos[n] = x
            y_pos[n] = -d

        dfs(0, 0)
        return x_pos, y_pos, leaf_counter[0]

    def _path_nodes(estimator, x_row: pd.Series, feature_names):
        tree = estimator.tree_
        feat = tree.feature
        thr  = tree.threshold
        x = x_row.values if hasattr(x_row, "values") else np.asarray(x_row)
        node = 0
        path = [0]
        while tree.children_left[node] != tree.children_right[node]:
            f_idx = feat[node]
            go_left = (x[f_idx] <= thr[node])
            node = int(tree.children_left[node] if go_left else tree.children_right[node])
            path.append(node)
        return path

    def _node_label(tree, node_id, feature_names):
        feat = tree.feature
        thr  = tree.threshold
        n    = int(tree.n_node_samples[node_id])
        mean = float(tree.value[node_id][0, 0])
        imp  = float(tree.impurity[node_id])  # MSE
        if tree.children_left[node_id] != tree.children_right[node_id]:
            return f"{feature_names[int(feat[node_id])]}\n≤\n{thr[node_id]:.6f}\n\nmse={imp:.3f}" #samples={n}\n
        else:
            texto = f"LEAF\nmean={mean:.4f}" #\nsamples={n}
            return texto

    # ---- posições + tamanho da figura ----
    x_pos, y_pos, n_leaves = _layout_positions(tree)
    max_depth = int(max(-np.array(list(y_pos.values()))) if y_pos else 0)

    dx = BOX_W + H_GAP
    DY = 1.40 * SLOPE_BOOST

    base_w = max(10, n_leaves * dx + 1.2)
    base_h = max(6,  (max_depth + 1) * DY + 1.2)
    fig_w  = max(8, base_w * FIG_WIDTH_SCALE)
    fig_h  = base_h
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    ax.set_axis_off()
    fig.subplots_adjust(left=0.02, right=0.985, top=0.90, bottom=0.06)
    ax.margins(x=0.01, y=0.03)

    # ---- caminho da instância ----
    path_nodes = _path_nodes(estimator, x_row, feature_names)
    path_set   = set(path_nodes)
    path_edges = set(zip(path_nodes[:-1], path_nodes[1:]))

    # ---- arestas ----
    for n in range(tree.node_count):
        cl, cr = tree.children_left[n], tree.children_right[n]
        if cl == cr:
            continue
        for ch in (cl, cr):
            x1, y1 = x_pos[n] * dx,  y_pos[n] * DY
            x2, y2 = x_pos[ch] * dx, y_pos[ch] * DY
            on_path = (n, ch) in path_edges
            ax.plot(
                [x1, x2], [y1, y2],
                linewidth=2.6 if on_path else 1.0,
                color="#E74C3C" if on_path else "#A0A0A0",
                zorder=1
            )

    # ---- nós ----
    wrap_cols = max(16, int(26 * (14.0 / FONT_SIZE) * (BOX_W / 2.05)))

    for n in range(tree.node_count):
        x, y = x_pos[n] * dx, y_pos[n] * DY

        # preserva as quebras de linha do _node_label
        raw_label = _node_label(tree, n, feature_names)
        linhas = raw_label.split("\n")
        partes = []
        for linha in linhas:
            wrapped = textwrap.wrap(linha, width=wrap_cols)
            if wrapped:
                partes.extend(wrapped)
            else:
                partes.append("")  # mantém linha em branco
        label = "\n".join(partes)

        on_path = n in path_set

        box = FancyBboxPatch(
            (x - BOX_W/2, y - BOX_H/2), BOX_W, BOX_H,
            boxstyle="round,pad=0.03,rounding_size=0.06",
            linewidth=2.8 if on_path else 1.0,
            edgecolor="#E74C3C" if on_path else "#5B7BD5",
            facecolor="#FFEDE7" if on_path else "#ECF3FF",
            zorder=2
        )
        ax.add_patch(box)
        ax.text(
            x, y, label, ha="center", va="center",
            fontsize=FONT_SIZE, color="#1A1A1A",
            fontweight=("bold" if on_path else "normal"),
            zorder=3
        )

    # ---- título + infos XGB / Árvore / Δ ----
    x_arr = np.asarray(x_row.values, dtype=float).reshape(1, -1)

    # saída do surrogate (prob classe 1)
    y_sur = float(estimator.predict(x_arr)[0])
    classe_sur = 1 if y_sur >= 0.5 else 0

    y_rf = np.nan
    classe_rf = None

    if teacher_model is not None:
        try:
            # prob da classe 1 do modelo professor
            y_rf = float(teacher_model.predict_proba(x_arr)[0, 1])
            classe_rf = 1 if y_rf >= 0.5 else 0
        except Exception:
            try:
                y_rf = float(teacher_model.predict(x_arr)[0])
                classe_rf = int(round(y_rf))
            except Exception:
                y_rf = np.nan
                classe_rf = None

    # título principal (sem métricas)
    #ax.set_title(
    #    f"\nCaminho na Árvore (Surrogate Global) — {instance_name}",
    #    fontsize=FONT_SIZE + 0.6,
    #    pad=1
    #)

    # ---- strings formatadas ----
    if classe_rf is not None and np.isfinite(y_rf):
        rf_txt = f"RF = {y_rf:.4f} (classe={classe_rf})"
    else:
        rf_txt = "RF = N/A"

    sur_txt = f"Árvore = {y_sur:.4f} (classe={classe_sur})"

    if np.isfinite(y_rf):
        delta_val = abs(y_rf - y_sur)
        delta_txt = f"|Δ| = {delta_val:.4f}"
    else:
        delta_val = np.nan
        delta_txt = "|Δ| = N/A"

    same_class = (classe_rf is not None) and (classe_rf == classe_sur)

    # ---- RF: prob (classe=x) – negrito normal ----
    # coordenadas mais próximas, tudo na mesma linha
    x_rf     = 0.47   # XGB bem à esquerda
    x_sur    = 0.30   # Árvore logo em seguida
    x_delta  = 0.65   # |Δ| em seguida

    # RF: prob (classe=x) – negrito
    ax.text(
        x_rf, 1.02, rf_txt,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=FONT_SIZE - 0.2,
        fontweight="bold",
        color="#222222"
    )

    # Árvore: prob (classe=x) – negrito com fundo laranja
    ax.text(
        x_sur, 1.02, sur_txt,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=FONT_SIZE - 0.2,
        fontweight="bold",
        color="#222222",
        bbox=dict(
            facecolor="#FFB74D",
            edgecolor="none",
            boxstyle="round,pad=0.25"
        )
    )

        # decide a cor do |Δ| pela consistência das classes
    same_class  = (classe_rf == classe_sur)
    delta_color = "#12671A" if same_class else "#B71C1C"  # verde escuro / vermelho
    print(same_class)

    # |Δ| – verde se mesma classe, vermelho se diferente
    ax.text(
        x_delta, 1.02, delta_txt,
        transform=ax.transAxes,
        ha="left", va="bottom",
        fontsize=FONT_SIZE - 0.2,
        fontweight="bold",
        color=delta_color
    )



    plt.tight_layout(pad=0.25)
    if save_path:
        plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.show()
# -------- executa para ENTRADAS_SELECIONADAS (ordem preservada) --------
# usa o espaço de features do surrogate (X_)
rows_sel, meta = select_rows_ordered(X_, ENTRADAS_SELECIONADAS)
print("\nVerificação da seleção (ordem preservada):")
print(meta.to_string(index=False))

if rows_sel.empty:
    print("⚠️ Nenhuma instância selecionada em X_ para explicar com o surrogate.")
else:
    for i, (idx, x_row) in enumerate(rows_sel.iterrows(), start=1):
        # usa apenas as features usadas no surrogate (evita 'M', 'B', y, etc.)
        x_feat = x_row[feature_names]

        # pega o ID real da instância (coluna 'id' do dataset principal)
        inst_id = x_row["id"] if "id" in x_row.index else idx

        caminho = _path_as_text(surrogate_model, x_feat, feature_names)
        out_png = os.path.join(
            OUT_DIR,
            f"{(NM_FILE_SURROGATE if 'NM_FILE_SURROGATE' in globals() else 'Surrogate')}"
            f"_GLOBAL_path_{i}_ID{inst_id}.png"
        )

        plot_tree_with_path(
            surrogate_model,
            x_feat,  # SOMENTE FEATURES NUMÉRICAS
            instance_name=f"ID: {int(inst_id)}",   # <<< aqui trocamos idx/pos por ID real
            save_path=out_png,
            feature_names=feature_names,
            teacher_model=(model if 'model' in globals() else None),
            FIG_WIDTH_SCALE=0.70,
            SLOPE_BOOST=1.10,
            FONT_SIZE=14,
            BOX_W=3.40,
            BOX_H=1.18,
            H_GAP=0.05
        )
        print(f"➡️  Instância {i} (ID={inst_id})\n    Caminho: {caminho}\n    Arquivo: {out_png}")



In [ ]:
# === Surrogate Decision Tree Interpretabilidade [Funções de Cálculo das Métricas] ===
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from itertools import combinations
from typing import Dict, List, Literal, Optional


def calcular_ic95_percentual(valores):
    # Média e desvio padrão
    media = np.mean(valores)
    std = np.std(valores, ddof=1)
    n = len(valores)
    # Erro padrão
    erro_padrao = std / np.sqrt(n)
    # IC 95% com distribuição normal (Z=1.96)
    ic_low = media - 1.96 * erro_padrao
    ic_high = media + 1.96 * erro_padrao
    return ic_low, ic_high

def calcular_ic95_percentual_median(valores):
    # Média e desvio padrão
    mediana = np.median(valores)
    std = np.std(valores, ddof=1)
    n = len(valores)
    # Erro padrão
    erro_padrao = std / np.sqrt(n)
    # IC 95% com distribuição normal (Z=1.96)
    ic_low = mediana - 1.96 * erro_padrao
    ic_high = mediana + 1.96 * erro_padrao
    return ic_low, ic_high

def formatar_ic_percentual(valor_central, limite_inferior, limite_superior):
    return f"{valor_central:.1f}% [{limite_inferior:.1f}%, {limite_superior:.1f}%]"


# ----------------------------------------------------------
# 1) Função principal (Molnar-style, ponto-a-ponto, probabilidades)
# ----------------------------------------------------------
def calcula_fidelity_surrogate_tree(
    x_row,                 # 1D array-like com as features na MESMA ordem usada no treino
    rf_model,              # modelo professor (XGBoostClassifier)
    tree_surrogate,        # surrogate (DecisionTreeRegressor)
    pos_class=1,           # índice da classe positiva no rf_model.classes_
    clip_probs=True,       # faz clip no [0,1] para a prob. do surrogate
    decimals=8
):
    """
    Retorna um dicionário com probas RF/Surrogate e fidelidade por instância.
    Fidelidade% = (1 - |p_RF - p_tree|) * 100   (Molnar: proximidade de predições)
    """
    x = np.asarray(x_row, dtype=float).reshape(1, -1)

    # Probabilidade do professor (RF) para a classe positiva
    p_rf = float(rf_model.predict_proba(x)[:, pos_class][0])

    # Probabilidade do surrogate (árvore reg.) — clip para [0,1]
    p_tree = float(tree_surrogate.predict(x)[0])
    if clip_probs:
        p_tree = float(np.clip(p_tree, 0.0, 1.0))

    # Erros ponto-a-ponto
    abs_err = abs(p_rf - p_tree)
    sq_err  = (p_rf - p_tree) ** 2

    # Fidelidade local (0–1) e em %
    fid_local       = 1.0 - abs_err
    fid_local_pct   = fid_local * 100.0
    rmse_local      = np.sqrt(sq_err)

    return {
        "pred": {
            "p_rf": round(p_rf, decimals),
            "p_tree": round(p_tree, decimals),
            "abs_err": round(abs_err, decimals),
            "sq_err": round(sq_err, decimals),
            "rmse": round(rmse_local, decimals),
        },
        "fidelity": {
            "mae_fidelity": round(fid_local, decimals),      # em [0,1]
            "mae_percent":  round(fid_local_pct, 3),         # em %
        }
    }
    
# === Infidelity (Yeh et al., 2019) para Surrogate Decision Tree ===
import numpy as np
import pandas as pd

# -------- helpers: caminho e região (intervalos) da folha do surrogate --------
def _leaf_region_for_instance(tree_estimator, x_row, feature_names, eps=1e-9):
    """
    Retorna dicionário {feature: (low, high)} com os limites acumulados
    ao longo do caminho percorrido por x_row na árvore.
    Only features presentes no caminho aparecem no dict.
    """
    x = np.asarray(x_row, dtype=float).reshape(1, -1)
    tree = tree_estimator.tree_
    feat = tree.feature
    thr  = tree.threshold

    intervals = {}  # f_idx -> (low, high)

    node = 0
    while tree.children_left[node] != tree.children_right[node]:
        f_idx = feat[node]
        t     = thr[node]
        val   = float(x[0, f_idx])
        low, high = intervals.get(f_idx, (-np.inf, np.inf))
        if val <= t:
            high = min(high, t)                 # vai à esquerda: x_f <= t
            node = tree.children_left[node]
        else:
            low  = max(low,  t + eps)           # direita: x_f > t (abre folga eps)
            node = tree.children_right[node]
        intervals[f_idx] = (low, high)

    # converte índices para nomes
    name_intervals = {feature_names[i]: intervals[i] for i in intervals}
    return name_intervals

def _sample_inside_leaf(x0, leaf_intervals, X_ref, n_samples=256, rng=None):
    """
    Amostra n_samples pontos x' que respeitam as restrições de folha do surrogate.
    Para cada feature j:
      low/high = interseção( intervalos da folha , [min_ref, max_ref] )
      depois amostra uniformemente em um intervalo local centrado em x0 (25% da largura).
    Features que não aparecem no caminho não têm restrição da árvore; usamos [min_ref,max_ref].
    """
    if rng is None:
        rng = np.random.default_rng(42)

    cols = list(X_ref.columns)
    x0   = np.asarray(x0, dtype=float)
    Xp   = np.tile(x0, (n_samples, 1))

    ref_min = np.asarray(X_ref[cols].min(axis=0), dtype=float)
    ref_max = np.asarray(X_ref[cols].max(axis=0), dtype=float)

    for j, col in enumerate(cols):
        # limites da folha (se existir) + limites globais
        low_leaf, high_leaf = leaf_intervals.get(col, (-np.inf, np.inf))
        low = max(low_leaf,  ref_min[j])
        high= min(high_leaf, ref_max[j])

        # se degenera, fixa em x0
        if not np.isfinite(low):  low  = ref_min[j]
        if not np.isfinite(high): high = ref_max[j]
        if high <= low:           low, high = x0[j], x0[j]

        # faixa local (25% da largura), centrada em x0, recortada a [low,high]
        width = max(1e-12, high - low)
        half  = 0.25 * width
        a = np.clip(x0[j] - half, low, high)
        b = np.clip(x0[j] + half, low, high)
        if b <= a:  # encosta na borda
            a, b = low, high
        if b <= a:  # continua degenerado
            Xp[:, j] = x0[j]
        else:
            Xp[:, j] = rng.uniform(a, b, size=n_samples)

    return Xp

# -------- função principal: infidelity em probas (classe positiva) --------
def calcula_infidelity_surrogate_tree(
    x_row, rf_model, tree_surrogate, X_ref, feature_names,
    pos_class=1, n_perturb=256, random_state=42, return_beta=True
):
    """
    Implementa Infid(Φ,f,x) = E_I[(I^T Φ(x) - (f(x)-f(x-I)))^2]
    - Φ(x) = β(x) ajustado por MQ dentro da folha do surrogate percorrida por x.
    - I = x - x' com x' amostrado dentro da região (folha).
    - f = XGB -> probabilidade da classe positiva.
    Retorna dict com infidelity (MSE), beta(local) e diagnósticos.
    """
    rng = np.random.default_rng(random_state)
    x0  = np.asarray(x_row, dtype=float).reshape(1, -1)
    cols = list(feature_names)

    # 1) região da folha do surrogate para x
    intervals = _leaf_region_for_instance(tree_surrogate, x0, cols)

    # 2) amostras x' dentro da folha
    Xp = _sample_inside_leaf(x0[0], intervals, X_ref[cols], n_samples=n_perturb, rng=rng)

    # 3) constrói I = x - x'
    I = (x0 - Xp)  # shape (K, d)

    # 4) deltas de f: f(x) - f(x')
    p0 = float(rf_model.predict_proba(x0)[:, pos_class][0])
    pP = rf_model.predict_proba(Xp)[:, pos_class].astype(float)
    delta_f = p0 - pP  # shape (K,)

    # 5) β(x) por mínimos quadrados: resolve I β ≈ delta_f
    #    usa lstsq para ser estável em colinearidade
    beta, *_ = np.linalg.lstsq(I, delta_f, rcond=None)  # shape (d,)

    # 6) infidelity = mean( (I β - delta_f)^2 )
    approx = I @ beta
    infid  = float(np.mean((approx - delta_f) ** 2))

    out = {
        "infidelity_mse": infid,
        "n_perturb": int(n_perturb),
        "p_rf_x": p0,
    }
    if return_beta:
        out["beta_local"] = beta
        out["leaf_intervals"] = intervals
    return out

def calcula_faithfulness_surrogate_tree(
    x_row,                   # 1D array-like nas colunas `feature_names`
    rf_model,                # XGBoostClassifier (professor)
    tree_surrogate,          # DecisionTreeRegressor (surrogate)
    feature_names,           # lista de colunas (ordem usada em x_row)
    X_ref,                   # DataFrame de background (ex.: X_train[feature_names])
    pos_class=1,             # índice da classe positiva em rf_model.classes_
    n_perturb=128,           # nº de amostras para cada feature i
    baseline="data",         # "data" (padrão), "mean", "median", "zero"
    corr="spearman",         # "spearman" (padrão), "pearson", "kendall"
    random_state=42,
    use_abs=False            # se True, usa |f(x)-f(x_-i)| ao invés do valor assinado
):
    """
    Faithfulness(Φ, f, x) = corr( Φ_i(x),  E[ f(x) - f(x_-i) ]_i )
    Retorna também o percentual mapeado: ((rho + 1)/2)*100 clippado em [0,100].
    """
    import numpy as np
    import pandas as pd

    def _tree_path_contribs_reg(estimator, x_vec):
        tree = estimator.tree_
        feat = tree.feature
        thr  = tree.threshold
        val  = tree.value
        d    = estimator.n_features_in_
        contrib = np.zeros(d, dtype=float)
        node = 0
        while tree.children_left[node] != tree.children_right[node]:
            parent_value = val[node][0, 0]
            f_idx = feat[node]
            go_left = x_vec[f_idx] <= thr[node]
            child = tree.children_left[node] if go_left else tree.children_right[node]
            child_value = val[child][0, 0]
            contrib[f_idx] += (child_value - parent_value)
            node = child
        return contrib

    def _corr_1d(a, b, method="spearman"):
        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)
        if np.std(a) == 0 or np.std(b) == 0:
            return np.nan
        method = (method or "spearman").lower()
        if method == "pearson":
            return float(np.corrcoef(a, b)[0, 1])
        if method == "kendall":
            try:
                from scipy.stats import kendalltau
                return float(kendalltau(a, b, nan_policy="omit")[0])
            except Exception:
                method = "spearman"
        ra = a.argsort().argsort().astype(float)
        rb = b.argsort().argsort().astype(float)
        return float(np.corrcoef(ra, rb)[0, 1])

    rng   = np.random.default_rng(random_state)
    cols  = list(feature_names)
    x_vec = np.asarray(x_row, dtype=float).reshape(-1)

    # 1) Importâncias locais do surrogate (contribuições de caminho)
    phi = _tree_path_contribs_reg(tree_surrogate, x_vec)

    # 2) Saída do XGB na instância x
    x_df = pd.DataFrame([x_vec], columns=cols)
    p_x  = float(rf_model.predict_proba(x_df)[:, pos_class][0])

    # 3) Impactos por feature via perturbação
    Xbg = X_ref[cols].to_numpy()
    d = len(cols)
    delta = np.zeros(d, dtype=float)
    for i in range(d):
        Xp = np.repeat(x_vec[None, :], n_perturb, axis=0)
        if baseline == "data":
            idx = rng.integers(0, Xbg.shape[0], size=n_perturb)
            repl = Xbg[idx, i]
        elif baseline == "mean":
            repl = np.full(n_perturb, np.nanmean(Xbg[:, i]))
        elif baseline == "median":
            repl = np.full(n_perturb, np.nanmedian(Xbg[:, i]))
        elif baseline == "zero":
            repl = np.zeros(n_perturb, dtype=float)
        else:
            raise ValueError("baseline deve ser 'data', 'mean', 'median' ou 'zero'")
        Xp[:, i] = repl
        p_mask = rf_model.predict_proba(pd.DataFrame(Xp, columns=cols))[:, pos_class].astype(float)
        diff = p_x - p_mask
        delta[i] = float(np.mean(np.abs(diff))) if use_abs else float(np.mean(diff))

    # 4) Correlação e percentuais
    faith = _corr_1d(phi, delta, method=corr)                 # em [-1, 1]
    if np.isnan(faith):
        faith_pct = np.nan
    else:
        faith_pct = ((faith + 1.0) / 2.0) * 100.0             # -1→0%, 0→50%, 1→100%
        faith_pct = float(np.clip(faith_pct, 0.0, 100.0))     # clip seguro

    return {
        "faithfulness": faith,                   # correlação (−1 a 1)
        "faithfulness_percent": faith_pct,       # em %, mapeado e clipado
        "phi": phi,                              # importâncias locais do surrogate
        "delta": delta,                          # impactos médios no XGB por feature
        "p_rf_x": p_x,
        "n_perturb": int(n_perturb),
        "corr": corr,
        "baseline": baseline,
        "use_abs": bool(use_abs)
    }

def calc_simplicity_tree(
    tree_model,
    x_row=None,                 # (opcional) instância para simplicidade por instância
    feature_names=None,         # ordem de colunas usada no treino (se x_row for Series)
    max_depth_cap=None          # (opcional) “profundidade máxima permitida” p/ % global
):
    """
    Retorna métricas de simplicidade:
      - depth_total: profundidade global da árvore (sklearn get_depth)
      - node_count: número total de nós |Nodes(g)|
      - (opcional, se x_row): node_depth da instância e simplicity_instance_percent
        simplicity_instance_percent = (1 - node_depth / depth_total) * 100
      - (opcional, se max_depth_cap): simplicity_global_percent = (1 - depth_total / max_depth_cap) * 100
    """
    depth_total = int(tree_model.get_depth())
    node_count  = int(tree_model.tree_.node_count)

    out = {
        "depth_total": depth_total,
        "|Nodes(g)|": node_count,
    }

    # % global (opcional), relativo a um teto (ex.: hiperparâmetro max_depth)
    if max_depth_cap is None:
        md = tree_model.get_params().get("max_depth", None)
        if isinstance(md, int) and md > 0:
            max_depth_cap = md
    if isinstance(max_depth_cap, (int, float)) and max_depth_cap and max_depth_cap > 0:
        val = (1.0 - depth_total / max_depth_cap) * 100.0
        out["simplicity_global_percent"] = float(np.clip(val, 0.0, 100.0))

    # Simplicidade por instância (opcional)
    if x_row is not None:
        # garante vetor na ordem correta
        if isinstance(x_row, pd.Series):
            x_vec = x_row[feature_names].to_numpy(dtype=float) if feature_names is not None else x_row.to_numpy(dtype=float)
        else:
            x_vec = np.asarray(x_row, dtype=float).ravel()

        # profundidade do caminho raiz→folha da instância:
        # decision_path retorna matriz CSR; nº de nós no caminho = indptr[1]-indptr[0]
        node_indicator = tree_model.decision_path(x_vec.reshape(1, -1))
        path_len = int(node_indicator.indptr[1] - node_indicator.indptr[0])  # inclui raiz e folha
        node_depth = max(0, path_len - 1)  # profundidade em arestas

        # percentil de simplicidade por instância
        if depth_total > 0:
            inst_pct = (1.0 - node_depth / depth_total) * 100.0
        else:
            inst_pct = 100.0  # árvore trivial (só raiz)

        out.update({
            "leaf_id": int(tree_model.apply(x_vec.reshape(1, -1))[0]),
            "node_depth": int(node_depth),
            "simplicity_instance_percent": float(np.clip(inst_pct, 0.0, 100.0)),
        })

    return out

 
from sklearn.base import clone

# ---------- util: contribuição local por caminho (árvore de regressão) ----------
def _path_contributions(tree_model, x_row, n_features):
    """
    Retorna vetor de contribuições por feature para a instância x_row,
    somando Δ (valor_filho - valor_pai) toda vez que a feature aparece no caminho.
    """
    tree = tree_model.tree_
    feat = tree.feature
    thr  = tree.threshold
    left = tree.children_left
    right= tree.children_right
    val  = tree.value.squeeze(-1).squeeze(-1)  # valor médio no nó

    x = np.asarray(x_row, dtype=float).ravel()
    node = 0
    contrib = np.zeros(n_features, dtype=float)

    while left[node] != right[node]:  # enquanto não for folha
        f = feat[node]
        t = thr[node]
        child = left[node] if x[f] <= t else right[node]
        contrib[f] += (val[child] - val[node])
        node = child
    return contrib

# ---------- util: correlação robusta ----------
def _corr(v1, v2, how="spearman"):
    v1 = np.asarray(v1, float); v2 = np.asarray(v2, float)
    # Se todos iguais, correlação indefinida -> 0
    if np.allclose(v1, v1[0]) or np.allclose(v2, v2[0]):
        return 0.0

    how = str(how).lower()
    if how == "pearson":
        c = np.corrcoef(v1, v2)[0, 1]
        return float(np.nan_to_num(c, nan=0.0))
    elif how == "kendall":
        # tenta SciPy; se não houver, aproxima por Spearman
        try:
            from scipy.stats import kendalltau
            c = kendalltau(v1, v2, nan_policy="omit").correlation
            return float(np.nan_to_num(c, nan=0.0))
        except Exception:
            how = "spearman"  # fallback

    # Spearman (ranks via pandas para lidar com empates)
    r1 = pd.Series(v1).rank(method="average").to_numpy()
    r2 = pd.Series(v2).rank(method="average").to_numpy()
    c = np.corrcoef(r1, r2)[0, 1]
    return float(np.nan_to_num(c, nan=0.0))

# ---------- mapping correlação [-1,1] -> % [0,100] (50% quando ρ=0) ----------
def _corr_to_percent(c):
    return float(np.clip((c + 1.0) * 50.0, 0.0, 100.0))

# ---------- função principal ----------

def consistency_surrogate_instance(
    x_row,
    X_train,
    rf_model,
    feature_names,
    base_surrogate=None,
    n_runs=10,
    seeds=None,
    corr_type="spearman",
    resample="bootstrap",      # "bootstrap", "subsample" ou None
    sample_frac=0.80,
    replace=None,
    vary_structure=True,
):
    """
    Re-treina o surrogate R vezes e mede a correlação entre os vetores de explicação
    locais (path contributions) para a MESMA instância. Quanto maior o percentual, melhor.
    """
    import numpy as np
    import pandas as pd
    from sklearn.base import clone
    from sklearn.tree import DecisionTreeRegressor

    # --- prepara dados e alvo do surrogate (prob do RF) ---
    Xtr = X_train[feature_names]
    n_features = len(feature_names)

    pos_class = list(rf_model.classes_).index(1) if 1 in set(rf_model.classes_) else 1
    # >>> CRÍTICO: manter como Series alinhada ao índice de Xtr <<<
    y_tr_full = pd.Series(
        rf_model.predict_proba(Xtr)[:, pos_class],
        index=Xtr.index
    )

    # seeds
    if seeds is None:
        seeds = list(range(202, 202 + n_runs))
    else:
        n_runs = len(seeds)

    # molde do surrogate
    if base_surrogate is None:
        base_surrogate = DecisionTreeRegressor(max_depth=5, min_samples_leaf=50, random_state=0)

    if replace is None:
        replace = (resample == "bootstrap")

    # helpers de correlação/percentual (fallbacks caso não existam globais)
    def _safe_corr(a, b, how="spearman"):
        try:
            return _corr(a, b, how=how)
        except NameError:
            from scipy.stats import spearmanr, kendalltau, pearsonr
            if how == "kendall":
                return float(kendalltau(a, b, nan_policy="omit").correlation)
            elif how == "pearson":
                return float(pearsonr(a, b)[0])
            else:
                return float(spearmanr(a, b, nan_policy="omit").correlation)

    def _to_percent(r):
        try:
            return _corr_to_percent(r)
        except NameError:
            return float((r + 1.0) * 50.0)   # -1→0%, 0→50%, +1→100%

    expl_vectors = []
    x_vec = np.asarray(x_row, dtype=float).ravel()

    # garante limites sensatos
    sample_frac = float(np.clip(sample_frac, 0.0, 1.0))

    for s in seeds:
        # --------- (1) Reamostragem por índice (sempre usando .loc) ----------
        if resample in ("bootstrap", "subsample") and 0.0 < sample_frac < 1.0:
            idx = Xtr.sample(frac=sample_frac, replace=replace, random_state=s).index
            X_sub = Xtr.loc[idx]
        else:
            X_sub = Xtr

        # alvo alinhado AO ÍNDICE de X_sub
        y_sub = y_tr_full.loc[X_sub.index].values

        # --------- (2) Clona e ajusta hiperparâmetros ----------
        surr = clone(base_surrogate)
        params = surr.get_params()
        if vary_structure:
            params["splitter"] = "random"
            if "max_features" not in params or params["max_features"] is None:
                params["max_features"] = "sqrt"
        params["random_state"] = s
        surr.set_params(**params)

        # treino
        surr.fit(X_sub, y_sub)

        # explicação local (usa sua função utilitária existente)
        vec = _path_contributions(surr, x_vec, n_features)
        expl_vectors.append(vec)

    # --------- (3) Correlações entre todos os pares r1<r2 ----------
    corrs = []
    for i in range(n_runs):
        for j in range(i + 1, n_runs):
            corrs.append(_safe_corr(expl_vectors[i], expl_vectors[j], how=corr_type))

    mean_corr = float(np.nanmean(corrs)) if corrs else 0.0
    mean_percent = _to_percent(mean_corr)

    return {
        "n_runs": n_runs,
        "corr_type": corr_type,
        "mean_corr": round(mean_corr, 4),               # em [-1,1]
        "consistency_percent": round(mean_percent, 2),  # 0–100, maior = melhor
        "pairwise_corrs": [None if (c is None or np.isnan(c)) else round(c, 4) for c in corrs],
        "seeds": seeds,
        "resample": resample,
        "sample_frac": sample_frac,
        "vary_structure": bool(vary_structure),
    }


# ---------- utilitários locais ----------
def _node_value(tree_: DecisionTreeRegressor, node: int) -> float:
    return float(tree_.tree_.value[node].squeeze())

def _path_contributions(tree_: DecisionTreeRegressor, x_vec: np.ndarray, n_features: int):
    """Vetor Φ(x): contribuição por feature ao longo do caminho (regressor)."""
    t = tree_.tree_
    feat = t.feature
    thr  = t.threshold
    left = t.children_left
    right= t.children_right

    contrib = np.zeros(n_features, dtype=float)
    node = 0
    while left[node] != right[node]:                   # nó interno
        f = feat[node]
        child = left[node] if x_vec[f] <= thr[node] else right[node]
        delta = _node_value(tree_, child) - _node_value(tree_, node)
        contrib[f] += delta
        node = child
    return contrib

def _features_on_path(tree_: DecisionTreeRegressor, x_vec: np.ndarray):
    """Índices das features usadas no caminho de x."""
    t = tree_.tree_
    feat = t.feature
    thr  = t.threshold
    left = t.children_left
    right= t.children_right

    used = []
    node = 0
    while left[node] != right[node]:
        f = feat[node]
        used.append(f)
        node = left[node] if x_vec[f] <= thr[node] else right[node]
    return sorted(set(used))

def _is_binary_col(series: pd.Series) -> bool:
    vals = pd.unique(series.dropna())
    if len(vals) <= 2:
        vals = np.unique(np.round(vals).astype(int))
        return set(vals).issubset({0, 1})
    return False

def _vec_norm(v: np.ndarray, norm: str = "l2") -> float:
    if norm == "l1":
        return float(np.sum(np.abs(v)))
    return float(np.sqrt(np.sum(v * v)))  # l2 default

# ---------- Robustness ----------
def robustness_surrogate_instance(
    x_row,                         # Series/array na MESMA ordem de feature_names
    tree_surrogate: DecisionTreeRegressor,
    X_train: pd.DataFrame,
    feature_names,
    eps_num: float = 0.05,         # raio relativo p/ numéricas (fração do range)
    eps_bin: float = 0.10,         # prob. de flip para binárias
    n_perturb: int = 256,          # nº de perturbações
    norm: str = "l2",              # "l2" ou "l1"
    on_path_only: bool = True,     # perturbar só as features do caminho de x
    random_state: int = 0
):
    """
    Robustness(x) = max_{||x' - x|| < eps} || Φ(x) - Φ(x') ||  (menor = melhor)
    + 'stability_percent' = 100 * (1 - max_dist / (max_dist + ||Φ(x)||))  (maior = melhor)
    """
    rng = np.random.default_rng(random_state)
    Xtr = X_train[feature_names].copy()
    x = np.asarray(x_row, dtype=float).ravel()
    n_features = len(feature_names)

    # min/max e binárias a partir do treino (para clip e flips)
    mins = Xtr.min().values.astype(float)
    maxs = Xtr.max().values.astype(float)
    is_bin = np.array([_is_binary_col(Xtr[c]) for c in feature_names], dtype=bool)

    # features a perturbar
    if on_path_only:
        idx_path = _features_on_path(tree_surrogate, x)
        pert_idx = np.array(idx_path, dtype=int)
    else:
        pert_idx = np.arange(n_features, dtype=int)

    # explicação base
    phi_x = _path_contributions(tree_surrogate, x, n_features)
    base_norm = _vec_norm(phi_x, norm=norm)

    # gera perturbações e mede distâncias
    dists = []
    for k in range(n_perturb):
        xp = x.copy()
        for j in pert_idx:
            if is_bin[j]:
                if rng.random() < eps_bin:
                    xp[j] = 1.0 - np.round(xp[j])  # flip 0↔1
            else:
                span = max(1e-12, maxs[j] - mins[j])
                delta = rng.uniform(-eps_num, eps_num) * span
                xp[j] = np.clip(xp[j] + delta, mins[j], maxs[j])

        phi_xp = _path_contributions(tree_surrogate, xp, n_features)
        d = _vec_norm(phi_x - phi_xp, norm=norm)
        dists.append(d)

    dists = np.asarray(dists, dtype=float)
    max_dist = float(np.max(dists)) if dists.size else 0.0
    mean_dist = float(np.mean(dists)) if dists.size else 0.0
    p95_dist  = float(np.percentile(dists, 95)) if dists.size else 0.0

    # Percentual (0–100): mapeia "quanto estável" é a explicação
    stability_percent = 100.0 * (1.0 - (max_dist / (max_dist + base_norm + 1e-12)))

    return {
        "robustness_max": round(max_dist, 6),       # menor = melhor
        "robustness_mean": round(mean_dist, 6),
        "robustness_p95": round(p95_dist, 6),
        "stability_percent": round(stability_percent, 2),  # maior = melhor (0–100)
        "base_norm_phi": round(base_norm, 6),
        "n_perturb": int(n_perturb),
        "perturbed_features": [feature_names[i] for i in pert_idx],
        "on_path_only": bool(on_path_only),
        "norm": norm,
    }



def _row_from_selection(X_df, sel, cols):
    """
    Retorna (x_row Series na ordem 'cols', idx_label, posicao).

    Agora assume que 'sel' é, por padrão, o RÓTULO (ID) do índice principal
    e acessa SEMPRE via .loc. Só cai pra posição (.iloc) se o rótulo não
    existir no índice mas for um inteiro válido dentro do range.
    """
    # tenta primeiro como rótulo de índice (ID)
    idx_label = sel

    if idx_label in X_df.index:
        # rótulo existe: usa loc
        pos = int(X_df.index.get_loc(idx_label))
        x_row = X_df.loc[idx_label, cols]
    else:
        # fallback: se for inteiro dentro do range, interpreta como posição
        if isinstance(sel, (int, np.integer)) and 0 <= int(sel) < len(X_df):
            pos = int(sel)
            idx_label = X_df.index[pos]
            x_row = X_df.loc[idx_label, cols]  # ainda usa loc no dataset principal
        else:
            raise KeyError(
                f"Seleção '{sel}' não encontrado como índice do X_df "
                "e não é uma posição inteira válida."
            )

    return x_row, idx_label, pos

# === Função: Completeness por instância (quanto maior, melhor) ===
def completeness_surrogate_instance(tree_model, X_ref, feature_names, x_row):
    """
    Completeness_Tree(L) = |{x ∈ X_ref : x ∈ L}| / |X_ref|
      - L: folha alcançada por x_row no surrogate.
      - X_ref: conjunto de referência (ex.: X_test) usado para medir cobertura.

    Retorna: leaf_id, suporte absoluto e percentual (0–100).
    """
    import numpy as np

    # garante vetor 2D na mesma ordem de treino
    x_vec = np.asarray(x_row, dtype=float).reshape(1, -1)

    # folha da instância e folhas de todo o conjunto de referência
    leaf_id_x   = int(tree_model.apply(x_vec)[0])
    leaves_ref  = tree_model.apply(X_ref[feature_names])

    support_abs = int((leaves_ref == leaf_id_x).sum())
    total_ref   = int(len(X_ref))
    support_pct = 100.0 * support_abs / max(1, total_ref)

    return {
        "leaf_id": leaf_id_x,
        "support_abs": support_abs,
        "support_total": total_ref,
        "completeness_percent": round(support_pct, 3)  # quanto maior, melhor
    }
    
# --- helper: extrai as features (em ordem) usadas no CAMINHO da árvore para x ---
def _path_features_for_instance(tree_model, x_vec, feature_names):
    tr = tree_model.tree_
    feat_idx = tr.feature
    thr = tr.threshold
    left = tr.children_left
    right = tr.children_right

    node = 0
    used = []
    while left[node] != right[node]:  # enquanto não for folha
        f = int(feat_idx[node])
        t = thr[node]
        used.append(f)
        node = left[node] if x_vec[f] <= t else right[node]

    # remove duplicatas preservando a ordem (uma feature pode repetir no caminho)
    seen, ordered = set(), []
    for j in used:
        if j not in seen:
            seen.add(j); ordered.append(j)
    return ordered, [feature_names[j] for j in ordered]

# --- métrica de selectivity (baseada em TESTE) ---
def selectivity_surrogate_instance(
    x_row,                 # 1D (Series/array) nas MESMAS colunas/ordem de feature_names
    X_ref,                 # DataFrame de referência p/ baseline (AQUI: X_test)
    rf_model,              # RandomForest "professor" (classe positiva)
    tree_surrogate,        # DecisionTreeRegressor surrogate
    feature_names,         # ordem das colunas
    baseline="median",     # "median" | "mean" | dict {col:valor} | array-like (mesma ordem)
    clip_zero=True         # clip em 0 quedas negativas
):
    """
    Selectivity(x) = (1/K) * sum_{k=1..K} [ f(x) - f(x \ S_k) ],
    onde S_k são as k primeiras features do CAMINHO da árvore surrogate.
    f(·) = prob. do XGB para a classe positiva.
    Retorna também o percentual (0–100, maior = melhor).
    """
    x = np.asarray(x_row, dtype=float).ravel().copy()

    # probabilidade base do XGB (classe positiva)
    pos_class = list(rf_model.classes_).index(1) if 1 in set(rf_model.classes_) else 1
    p0 = float(rf_model.predict_proba(x.reshape(1, -1))[:, pos_class][0])

    # features do caminho do surrogate
    path_ids, path_names = _path_features_for_instance(tree_surrogate, x, feature_names)
    K = len(path_ids)
    if K == 0:
        return {"selectivity": 0.0, "selectivity_percent": 0.0, "base_prob": p0,
                "K": 0, "path_features": [], "per_k_drop": []}

    # baseline por feature (calculado em X_ref = X_test)
    if isinstance(baseline, str):
        if baseline.lower() == "median":
            base_vec = np.asarray(X_ref[feature_names].median().values, dtype=float)
        elif baseline.lower() == "mean":
            base_vec = np.asarray(X_ref[feature_names].mean().values, dtype=float)
        else:
            raise ValueError("baseline deve ser 'median' ou 'mean' quando string.")
    elif isinstance(baseline, dict):
        base_vec = np.asarray([baseline.get(c, X_ref[c].median()) for c in feature_names], dtype=float)
    else:
        base_arr = np.asarray(baseline, dtype=float).ravel()
        if base_arr.size != len(feature_names):
            raise ValueError("baseline array-like deve ter o mesmo tamanho de feature_names.")
        base_vec = base_arr

    # masking cumulativo nas k primeiras features do caminho
    deltas = []
    x_mask = x.copy()
    for k in range(1, K + 1):
        j = path_ids[k-1]
        x_mask[j] = base_vec[j]
        pk = float(rf_model.predict_proba(x_mask.reshape(1, -1))[:, pos_class][0])
        drop = p0 - pk
        if clip_zero:
            drop = max(0.0, drop)
        deltas.append(drop)

    sel = float(np.mean(deltas))          # em probabilidade
    sel_pct = 100.0 * sel                 # 0–100 (maior = melhor)

    return {
        "selectivity": sel,
        "selectivity_percent": round(sel_pct, 3),
        "base_prob": round(p0, 6),
        "K": K,
        "path_features": path_names,
        "per_k_drop": [round(d, 6) for d in deltas],
    }
    
# --- contribuições locais (absolutas) ao longo do caminho do surrogate ---
def _path_contrib_abs(tree_model, x_vec, feature_names):
    """
    Retorna:
      - contrib_abs: dict {feature_name: soma das |deltas| ao longo do caminho}
      - path_feats:  lista de nomes de features (na ordem em que aparecem no caminho)
    """
    tr = tree_model.tree_
    feat = tr.feature
    thr = tr.threshold
    left = tr.children_left
    right = tr.children_right
    vals = tr.value.squeeze()  # média predita em cada nó

    node = 0
    contrib = {}
    path_feats_idx = []
    while left[node] != right[node]:  # enquanto não for folha
        f = int(feat[node])
        child = left[node] if x_vec[f] <= thr[node] else right[node]
        delta = abs(float(vals[child]) - float(vals[node]))
        name = feature_names[f]
        contrib[name] = contrib.get(name, 0.0) + delta
        path_feats_idx.append(f)
        node = child

    # remove duplicatas mantendo a ordem
    seen, ordered_names = set(), []
    for f in path_feats_idx:
        nm = feature_names[f]
        if nm not in seen:
            seen.add(nm); ordered_names.append(nm)
    return contrib, ordered_names

def soundness_surrogate_instance(
    x_row,                 # 1D (Series/array) nas MESMAS colunas/ordem de feature_names
    tree_surrogate,        # DecisionTreeRegressor (surrogate)
    feature_names,         # ordem das colunas
    irrelevant_features=None,   # lista explícita de irrelevantes (opcional)
    mode_if_none="complement_path"  # "complement_path" ou "none"
):
    """
    Soundness(x) = 1 - (sum_{j∈U} |Φ_j(x)|) / (sum_{i} |Φ_i(x)|),
    Φ_j(x) = contribuição local absoluta da feature j no CAMINHO do surrogate.
    U = conjunto de features irrelevantes. Se não fornecido:
        - "complement_path": usa TODAS as features que NÃO aparecem no caminho como "irrelevantes".
        - "none": não considera irrelevantes (retorna 1.0).
    Retorna também o percentual (0–100).
    """
    x = np.asarray(x_row, dtype=float).ravel()

    contrib_abs, path_feats = _path_contrib_abs(tree_surrogate, x, feature_names)
    denom = float(sum(contrib_abs.values()))
    if denom <= 0.0:
        # caminho não altera a predição (árvore quase constante)
        return {
            "soundness": 1.0,
            "soundness_percent": 100.0,
            "denominator_sum": 0.0,
            "numerator_irrelevant_sum": 0.0,
            "path_features": path_feats,
            "irrelevant_used": [],
            "contrib_abs": contrib_abs,
        }

    # define U (irrelevantes)
    if isinstance(irrelevant_features, (list, tuple, set)) and len(irrelevant_features) > 0:
        U = set(map(str, irrelevant_features))
    else:
        if mode_if_none == "complement_path":
            U = set(feature_names) - set(path_feats)
        else:  # "none"
            U = set()

    num = float(sum(v for f, v in contrib_abs.items() if f in U))
    snd = 1.0 - (num / denom)
    snd = max(0.0, min(1.0, snd))
    return {
        "soundness": snd,
        "soundness_percent": round(100.0 * snd, 3),
        "denominator_sum": round(denom, 6),
        "numerator_irrelevant_sum": round(num, 6),
        "path_features": path_feats,
        "irrelevant_used": sorted(list(U)),
        "contrib_abs": {k: round(v, 6) for k, v in contrib_abs.items()},
    }

def directional_soundness_surrogate_instance(
    x_row,                                   # 1D (mesma ordem de feature_names)
    tree_surrogate,                          # DecisionTreeRegressor/Classifier (surrogate)
    feature_names: List[str],
    expected_directions: Dict[str, int],     # ex.: {"polidipsia": +1, "idade": -1, ...}
    pos_class: Optional[int] = 1,            # se o surrogate for classifier: índice da classe positiva
    denom: Literal["R", "R_used"] = "R",     # denominador: |R| (padrão) ou apenas R que apareceu no caminho
    eps_zero: float = 1e-9                   # tolerância para contribuição ~0
) -> dict:
    """
    Directional-Soundness(x) = (#features em R cujo sinal(contrib) == sinal(GT)) / denom
    onde denom = |R| (padrão) ou |{i em R : |Phi_i(x)|>0}| se denom="R_used".
    Retorna percentuais (0-100, maior=melhor) e diagnóstico por feature.
    """

    # --- 1) vetor x na ordem correta
    x_vec = np.asarray(pd.Series(x_row, index=feature_names), dtype=float).ravel()

    # --- 2) util: valor do nó (probabilidade estimada) para medir deltas
    tree = tree_surrogate.tree_
    def _node_value(node: int) -> float:
        v = tree.value[node]
        if isinstance(tree_surrogate, DecisionTreeRegressor):
            return float(v[0, 0])
        else:  # classificador: probabilidade da classe positiva
            counts = v[0]
            tot = counts.sum()
            if tot <= 0:
                return 0.0
            idx = 1 if (pos_class is None) else pos_class
            if idx >= len(counts):  # segurança
                idx = np.argmax(counts)
            return float(counts[idx] / tot)

    # --- 3) segue o caminho da instância e acumula deltas por feature
    feat_idx = tree.feature
    thr      = tree.threshold
    left     = tree.children_left
    right    = tree.children_right

    # encontra caminho até a folha
    node = 0
    path_nodes = [0]
    while left[node] != right[node]:
        f = feat_idx[node]
        t = thr[node]
        if x_vec[f] <= t:
            node = left[node]
        else:
            node = right[node]
        path_nodes.append(node)

    # deltas por aresta -> contribuição por feature (soma dos deltas onde a feature aparece)
    contrib = {f: 0.0 for f in range(len(feature_names))}
    for k in range(len(path_nodes) - 1):
        n_parent = path_nodes[k]
        n_child  = path_nodes[k+1]
        f = int(feat_idx[n_parent])
        delta = _node_value(n_child) - _node_value(n_parent)
        contrib[f] += float(delta)

    # mapeia para nomes
    phi_by_name = {feature_names[i]: contrib[i] for i in range(len(feature_names))}

    # --- 4) monta R (features relevantes com direção esperada)
    R = list(expected_directions.keys())
    if len(R) == 0:
        return {
            "directional_soundness_percent": np.nan,
            "matches": 0, "total": 0,
            "detail": [], "path_features": [feature_names[int(feat_idx[n])] for n in path_nodes[:-1]]
        }

    # conjunto usado no denominador
    if denom == "R_used":
        R_denom = [f for f in R if abs(phi_by_name.get(f, 0.0)) > eps_zero]
    else:
        R_denom = R[:]

    total = max(1, len(R_denom))  # evita div/0
    detail = []
    matches = 0

    for f in R:
        phi = float(phi_by_name.get(f, 0.0))
        s_phi = 0 if abs(phi) <= eps_zero else (1 if phi > 0 else -1)
        s_gt  = 0 if expected_directions[f] == 0 else (1 if expected_directions[f] > 0 else -1)
        used  = abs(phi) > eps_zero
        ok    = (s_phi == s_gt) and (f in R_denom)  # conta só se entra no denominador

        if ok:
            matches += 1

        detail.append({
            "feature": f,
            "phi": phi,
            "sign_phi": s_phi,
            "sign_expected": s_gt,
            "used_in_path": used,
            "match": bool(ok),
        })

    percent = 100.0 * matches / total
    return {
        "directional_soundness_percent": round(percent, 2),
        "matches": int(matches),
        "total": int(total),
        "denominator_mode": denom,
        "detail": detail,
        "path_features": [feature_names[int(feat_idx[n])] for n in path_nodes[:-1]],
    }

def _path_contrib_vector(tree_model, x_vec, feature_names):
    """
    Retorna um vetor (len = n_features) com a soma dos |deltas| de cada feature
    ao longo do caminho percorrido pela instância (da raiz até a folha).
    """
    tr   = tree_model.tree_
    feat = tr.feature
    thr  = tr.threshold
    left = tr.children_left
    right= tr.children_right
    vals = tr.value.squeeze()

    n_features = len(feature_names)
    vec = np.zeros(n_features, dtype=float)

    node = 0
    while left[node] != right[node]:  # enquanto não for folha
        f = int(feat[node])
        child = left[node] if x_vec[f] <= thr[node] else right[node]
        delta = abs(float(vals[child]) - float(vals[node]))
        vec[f] += delta
        node = child
    return vec


def stability_surrogate_instance(
    x_row,                 # Series/array 1D nas MESMAS colunas e ordem de 'feature_names'
    tree_surrogate,        # DecisionTreeRegressor/Classifier usado como surrogate
    feature_names,         # ordem das colunas
    sigma=0.05,            # intensidade do ruído (escala relativa ao desvio-padrão de cada feature)
    n_perturb=256,         # nº de vizinhos x' = x + δ
    norm="l2",             # "l2" (padrão) ou "l1" para a distância entre explicações
    X_train=None,          # opcional: DataFrame para estimar std e min/max por feature
    clip_to_train_range=True  # se True, mantém x' dentro do [min,max] do treino
):
    """
    Stability(x) ≈ 1 - E[ ||Φ(x) - Φ(x+δ)|| / (||Φ(x)|| + ||Φ(x+δ)|| + eps) ]  ∈ [0,1]
    Retorna também o valor em porcentagem (quanto MAIOR, mais estável).
    """
    x = np.asarray(x_row, dtype=float).ravel()
    n_features = len(feature_names)

    # vetor de explicação da instância (contribuições absolutas no caminho)
    phi_x = _path_contrib_vector(tree_surrogate, x, feature_names)

    # estatísticas do treino para dimensionar o ruído e limitar o range
    if X_train is not None:
        Xtr = X_train[feature_names]
        std = np.asarray(Xtr.std(ddof=0), dtype=float)
        std[~np.isfinite(std) | (std == 0)] = 1.0
        if clip_to_train_range:
            vmin = np.asarray(Xtr.min(), dtype=float)
            vmax = np.asarray(Xtr.max(), dtype=float)
    else:
        std = np.ones(n_features, dtype=float)
        if clip_to_train_range:
            vmin = np.full(n_features, -np.inf)
            vmax = np.full(n_features,  np.inf)

    ord_val = 2 if norm.lower() == "l2" else 1
    eps = 1e-12

    dists = []
    for _ in range(int(n_perturb)):
        noise = np.random.normal(loc=0.0, scale=sigma, size=n_features) * std
        x_pert = x + noise
        if clip_to_train_range:
            x_pert = np.clip(x_pert, vmin, vmax)

        phi_pert = _path_contrib_vector(tree_surrogate, x_pert, feature_names)

        num = np.linalg.norm(phi_x - phi_pert, ord=ord_val)
        den = np.linalg.norm(phi_x, ord=ord_val) + np.linalg.norm(phi_pert, ord=ord_val) + eps
        dists.append(num / den)

    mean_dist = float(np.mean(dists)) if dists else 0.0
    stability_raw = max(0.0, min(1.0 - mean_dist, 1.0))
    stability_percent = 100.0 * stability_raw

    return {
        "stability": round(stability_raw, 6),               # em [0,1]
        "stability_percent": round(stability_percent, 3),   # 0–100 (maior = mais estável)
        "mean_normalized_distance": round(mean_dist, 6),
        "n_perturb": int(n_perturb),
        "sigma": float(sigma),
        "norm": norm.lower(),
        "phi_x": phi_x,                  # vetor de explicações base (útil para depuração)
        "distances": dists,              # lista das distâncias normalizadas por perturbação
    }
    
def calcula_sufficiency_surrogate_tree(
    x_row,                      # Series/array 1D na MESMA ordem de 'feature_names'
    rf_model,                   # modelo caixa-preta (p.ex. XGBoostClassifier)
    tree_surrogate,             # DecisionTreeRegressor/Classifier (surrogate)
    feature_names,              # lista/ordem das colunas
    X_ref,                      # DataFrame de referência (p/ baseline; use X_train)
    class_index=1,              # índice da classe-alvo em predict_proba
    percent_list=(0.01, 0.05, 0.10, 0.20),  # %Ks que você quer avaliar
    percent_mode="path",        # "path": % sobre nº de features do caminho; "all": % sobre D total
    ranking="abs",              # "abs" (|delta|, padrão) ou "signed" (delta)
    masking="median_mode",      # "median_mode" | "mean_mode" | "zero" | "sample"
    random_state=42
):
    """
    Retorna:
      {
        'scores': {'1%':..., '5%':..., '10%':..., '20%':..., 'AOPC_suff(%)':..., 'AOPC_suff_raw':...},
        'details': {...debug...}
      }
    Métrica (maior = melhor):
      score_K = 100 * (1 - clip( (f(x) - f(x|S_K)) / max(f(x), eps), 0, 1 ))
    onde S_K são as K features do CAMINHO do surrogate com maior contribuição local.
    """

    rng = np.random.default_rng(random_state)
    feat_names = list(feature_names)
    x = np.asarray(x_row, dtype=float).ravel()

    # -------- helpers internos --------
    def _build_baselines(Xdf, how):
        Xd = Xdf[feat_names]
        if how == "median_mode":
            base = Xd.median(numeric_only=True).reindex(feat_names).astype(float).to_dict()
            # se houver categóricas/0-1, median funciona bem; mode (fallback) se NaN
            for c in feat_names:
                if pd.isna(base[c]):
                    mode_vals = Xd[c].mode(dropna=True)
                    base[c] = float(mode_vals.iloc[0]) if len(mode_vals) else 0.0
            return base
        if how == "mean_mode":
            base = Xd.mean(numeric_only=True).reindex(feat_names).astype(float).to_dict()
            for c in feat_names:
                if pd.isna(base[c]):
                    mode_vals = Xd[c].mode(dropna=True)
                    base[c] = float(mode_vals.iloc[0]) if len(mode_vals) else 0.0
            return base
        if how == "zero":
            return {c: 0.0 for c in feat_names}
        if how == "sample":
            row = Xd.sample(n=1, random_state=rng.integers(0, 10_000)).iloc[0]
            return {c: float(row[c]) for c in feat_names}
        # default
        return {c: 0.0 for c in feat_names}

    def _mask_x_keep_S(x_vec, S_idx, baselines):
        """retorna x|S: mantém valores originais apenas em S; demais = baseline."""
        xS = np.array([baselines[c] for c in feat_names], dtype=float)
        xS[S_idx] = x_vec[S_idx]
        return xS

    def _path_contribs(tree_model, x_vec):
        """contribuições locais por feature ao longo do caminho (delta de valor entre nós)."""
        tr   = tree_model.tree_
        feat = tr.feature
        thr  = tr.threshold
        left = tr.children_left
        right= tr.children_right
        vals = tr.value.squeeze()

        contrib = {}
        order_idx = []  # ordem de aparição no caminho
        node = 0
        while left[node] != right[node]:
            f = int(feat[node])
            child = left[node] if x_vec[f] <= thr[node] else right[node]
            delta = float(vals[child]) - float(vals[node])
            val = abs(delta) if ranking == "abs" else delta
            name = feat_names[f]
            contrib[name] = contrib.get(name, 0.0) + val
            order_idx.append(f)
            node = child

        # índices únicos do caminho mantendo a ordem
        seen, path_idx = set(), []
        for j in order_idx:
            if j not in seen:
                seen.add(j); path_idx.append(j)
        return contrib, path_idx

    # -------- 1) f(x) no XGB --------
    x_df = pd.DataFrame([x], columns=feat_names)
    fx = float(rf_model.predict_proba(x_df)[:, class_index][0])

    # -------- 2) ranking das features do caminho --------
    contrib, path_idx = _path_contribs(tree_surrogate, x)
    # se árvore não usar nenhuma feature (ex.: constante), retorna 100% por definição
    if len(path_idx) == 0:
        return {
            "scores": {f"{int(p*100)}%": 100.0 for p in percent_list} | {"AOPC_suff(%)": 100.0, "AOPC_suff_raw": 0.0},
            "details": {"fx": fx, "Ks": [0]*len(percent_list), "topK_features": {}, "path_features": []}
        }

    # ordena por contribuição (desc)
    contrib_vec = np.array([contrib.get(feat_names[j], 0.0) for j in path_idx], dtype=float)
    order = np.argsort(-np.abs(contrib_vec)) if ranking == "abs" else np.argsort(-contrib_vec)
    path_idx_ordered = [path_idx[i] for i in order]

    # -------- 3) baseline para mascarar --------
    baselines = _build_baselines(X_ref, masking)

    # -------- 4) monta S_K e calcula f(x|S_K) para cada % --------
    D_all = len(feat_names)
    M_path = len(path_idx_ordered)
    Ks, fx_Sk, drops, S_lists = [], [], [], []

    for p in percent_list:
        if percent_mode == "all":
            K = int(np.ceil(max(1, p * D_all)))
        else:  # "path"
            K = int(np.ceil(max(1, p * M_path)))

        K = min(K, M_path)  # não pode passar do tamanho do caminho
        top_idx = path_idx_ordered[:K]

        xS = _mask_x_keep_S(x, top_idx, baselines)
        fxS = float(rf_model.predict_proba(pd.DataFrame([xS], columns=feat_names))[:, class_index][0])
        drop = fx - fxS

        Ks.append(K)
        fx_Sk.append(fxS)
        drops.append(drop)
        S_lists.append([feat_names[j] for j in top_idx])

    # -------- 5) normalização (0–100, maior = melhor) --------
    eps = 1e-12
    scores = []
    for drop in drops:
        scores.append(100.0 * (1.0 - np.clip(drop / max(fx, eps), 0.0, 1.0)))

    aopc_raw = float(np.mean(drops)) if drops else 0.0
    aopc_pct = 100.0 * (1.0 - np.clip(aopc_raw / max(fx, eps), 0.0, 1.0))

    labels = [f"{int(p*100)}%" for p in percent_list]
    scores_dict = {lbl: float(sc) for lbl, sc in zip(labels, scores)}
    scores_dict["AOPC_suff(%)"] = float(aopc_pct)
    scores_dict["AOPC_suff_raw"] = float(aopc_raw)

    details = {
        "fx": fx,
        "Ks": Ks,
        "percents": labels,
        "fx_Sk": {lbl: val for lbl, val in zip(labels, fx_Sk)},
        "drop_Sk": {lbl: val for lbl, val in zip(labels, drops)},
        "topK_features": {lbl: feats for lbl, feats in zip(labels, S_lists)},
        "path_features": [feat_names[j] for j in path_idx_ordered],
        "ranking": ranking,
        "percent_mode": percent_mode,
        "masking": masking
    }
    return {"scores": scores_dict, "details": details}

In [ ]:
# === Surrogate Decision Tree Interpretabilidade [Cálculo das Métricas] ===

from sklearn.metrics import mean_squared_error, r2_score
from joblib import Parallel, delayed
import numpy as np
import pandas as pd
import re
import time
import psutil, multiprocessing, os

inicio = time.time()

SURROGATE_TREE_TABLE_METRICS             = []
SURROGATE_TREE_FIDELITY_VALUES           = []
SURROGATE_TREE_INFIDELITY_VALUES         = []
SURROGATE_TREE_FAITHFULLNESS_VALUES      = []
SURROGATE_TREE_CALIBRATION_VALUES        = []
SURROGATE_TREE_SELECTIVITY_VALUES        = []
SURROGATE_TREE_SOUNDNESS_VALUES          = []
SURROGATE_TREE_RELEVANCE_VALUES          = []
SURROGATE_TREE_SIMPLICITY_VALUES         = []
SURROGATE_TREE_ROBUSTENESS_VALUES        = []
SURROGATE_TREE_ROBUSTENESS_ADV_VALUES    = []
SURROGATE_TREE_STABILITY_VALUES          = []
SURROGATE_TREE_MONOTONIA_VALUES          = []
SURROGATE_TREE_COMPLETENESS_VALUES       = []
SURROGATE_TREE_SUFFICIENCY_VALUES        = []
SURROGATE_TREE_CONSISTENCY_VALUES        = []
SURROGATE_TREE_CLASS_CONSISTENCY_VALUES  = []
SURROGATE_TREE_DISCRIMINABILITY_VALUES   = []
SURROGATE_TREE_DISPARITY_VALUES          = []
SURROGATE_TREE_DIR_SOUNDNESS_VALUES      = []
SURROGATE_TREE_SUFFICIENCY_1_VALUES      = []
SURROGATE_TREE_SUFFICIENCY_5_VALUES      = []
SURROGATE_TREE_SUFFICIENCY_10_VALUES     = []
SURROGATE_TREE_SUFFICIENCY_20_VALUES     = []

PERC_LIST = (0.01, 0.05, 0.10, 0.20)  # 1%, 5%, 10%, 20%

SURR_ABSERR_VALUES     = []   # |p_rf - p_tree|
SURR_RMSE_VALUES       = []   # RMSE local

# ---------- 0) Preparos/fallbacks ----------
# feat_cols_eff: ordem/colunas usadas no treino do surrogate/RF
if 'feat_cols_eff' not in globals():
    if 'feature_names' in globals():
        feat_cols_eff = list(feature_names)
    else:
        # remove rótulos alvo se existirem no X (dataset completo)
        feat_cols_eff = [c for c in X.columns if c not in {'classe', 'y', 'diagnosis'}]

# checagem de presença no X (dataset completo)
missing_cols = [c for c in feat_cols_eff if c not in X.columns]
if missing_cols:
    raise ValueError(f"As seguintes colunas de feat_cols_eff não estão em X: {missing_cols}")

# classe positiva
if 'POS_CLASS' not in globals():
    POS_CLASS = list(model.classes_).index(1) if 1 in set(model.classes_) else 1

# ----------------------------------------------------------
# 3) Normalizar ENTRADAS_SELECIONADAS -> sempre labels (índices de X) p/ usar .loc
# ----------------------------------------------------------
ENTRADAS_SELECIONADAS = list(ENTRADAS_SELECIONADAS)

# Se queremos trabalhar por id, garanta que o índice de X seja o 'id'
if "id" in X.columns and X.index.name != "id":
    X = X.set_index("id", drop=False)

# Interseção direta: tratamos SEMPRE como labels (loc) do X
idx_all = pd.Index(ENTRADAS_SELECIONADAS)
IDX_LABELS = list(X.index.intersection(idx_all))

if len(IDX_LABELS) == 0:
    raise ValueError(
        "Nenhuma ENTRADA_SELECIONADA foi encontrada em X.index.\n"
        f"Exemplo de X.index: {list(X.index[:10])}\n"
        f"Exemplo de ENTRADAS_SELECIONADAS: {ENTRADAS_SELECIONADAS[:10]}"
    )

if len(IDX_LABELS) < len(ENTRADAS_SELECIONADAS):
    print(
        f"⚠️ Aviso: apenas {len(IDX_LABELS)} de {len(ENTRADAS_SELECIONADAS)} "
        "índices de ENTRADAS_SELECIONADAS estão presentes em X.index."
    )

# (Opcional) se em algum lugar você ainda precisar das posições (iloc):
IDX_POS = [X.index.get_loc(lbl) for lbl in IDX_LABELS]


# ----------------------------------------------------------
# 4) Worker por instância (ID) — calcula TODAS as métricas para aquele ID
# ----------------------------------------------------------
def _compute_metrics_for_idx_label(idx_label):
    """
    Worker paralelo:
      - pega a linha de X via .loc[idx_label, feat_cols_eff]
      - calcula TODAS as métricas para essa instância
      - retorna dicionário com os valores
    """
    # x_row SEMPRE via loc em X (dataset completo, índice original)
    x_row = X.loc[idx_label, feat_cols_eff]

    # ---------------- Fidelity ----------------
    res_fid = calcula_fidelity_surrogate_tree(
        x_row=x_row,
        rf_model=model,
        tree_surrogate=surrogate_model,
        pos_class=POS_CLASS,
        clip_probs=True,
        decimals=8
    )
    fidelity_percent = float(res_fid["fidelity"]["mae_percent"])
    abs_err = float(res_fid["pred"]["abs_err"])
    rmse_local = float(res_fid["pred"]["rmse"])

    # ---------------- Infidelity ----------------
    res_inf = calcula_infidelity_surrogate_tree(
        x_row=x_row,
        rf_model=model,
        tree_surrogate=surrogate_model,
        X_ref=X,                       # usa X como referência
        feature_names=feat_cols_eff,
        pos_class=POS_CLASS,
        n_perturb=256,
        random_state=42,
        return_beta=False
    )
    # menor MSE = melhor -> aqui mantemos como "menor melhor" em %
    infidelity_percent = float(res_inf["infidelity_mse"] * 100.0)

    # ---------------- Faithfulness ----------------
    res_faith = calcula_faithfulness_surrogate_tree(
        x_row=x_row,
        rf_model=model,
        tree_surrogate=surrogate_model,
        feature_names=feat_cols_eff,
        X_ref=X_train,     # segue usando X_train como referência
        pos_class=POS_CLASS,
        n_perturb=128,
        baseline="zero",
        corr="spearman",
        random_state=42,
        use_abs=False
    )
    faithfulness_percent = float(res_faith["faithfulness_percent"])

    # ---------------- Simplicity ----------------
    res_simp = calc_simplicity_tree(
        surrogate_model,
        x_row=x_row,
        feature_names=feat_cols_eff
    )
    simplicity_percent = float(res_simp.get("simplicity_instance_percent", 0.0))

    # ---------------- Consistency ----------------
    res_cons = consistency_surrogate_instance(
        x_row=x_row,
        X_train=X_train,
        rf_model=model,
        feature_names=feat_cols_eff,
        base_surrogate=surrogate_model,
        n_runs=10,
        corr_type="spearman",
        resample="bootstrap",
        sample_frac=0.8,
        vary_structure=True
    )
    consistency_percent = float(res_cons["consistency_percent"])

    # ---------------- Robustness (stability_percent) ----------------
    res_rob = robustness_surrogate_instance(
        x_row=x_row,
        tree_surrogate=surrogate_model,
        X_train=X_train,
        feature_names=feat_cols_eff,
        eps_num=0.05,
        eps_bin=0.10,
        n_perturb=512,
        norm="l2",
        on_path_only=True,
        random_state=42
    )
    robusteness_percent = float(res_rob["stability_percent"])

    # ---------------- Completeness ----------------
    res_comp = completeness_surrogate_instance(
        tree_model=surrogate_model,
        X_ref=X,                 # aqui também usamos X (dataset inteiro)
        feature_names=feat_cols_eff,
        x_row=x_row
    )
    completeness_percent = float(res_comp["completeness_percent"])

    # ---------------- Selectivity ----------------
    res_sel = selectivity_surrogate_instance(
        x_row=x_row,
        X_ref=X,                 # X como referência
        rf_model=model,
        tree_surrogate=surrogate_model,
        feature_names=feat_cols_eff,
        baseline="median",
        clip_zero=True
    )
    selectivity_percent = float(res_sel["selectivity_percent"])

    # ---------------- Soundness ----------------
    IRRELEVANT_FEATURES = ['fraqueza', 'rigidez_muscular', 'cura_retardada',
                           'desfoque_visual', 'coceira']

    res_snd = soundness_surrogate_instance(
        x_row=x_row,
        tree_surrogate=surrogate_model,
        feature_names=feat_cols_eff,
        irrelevant_features=IRRELEVANT_FEATURES,
        mode_if_none="complement_path"
    )
    soundness_percent = float(res_snd["soundness_percent"])

    # ---------------- Directional Soundness ----------------
    res_dir = directional_soundness_surrogate_instance(
        x_row=x_row,
        tree_surrogate=surrogate_model,
        feature_names=feat_cols_eff,
        expected_directions=direcao_esperada,
        pos_class=POS_CLASS,
        denom="R_used",
        eps_zero=1e-8
    )
    dir_soundness_percent = float(res_dir["directional_soundness_percent"])

    # ---------------- Stability (ruído pequeno no input) ----------------
    res_stab = stability_surrogate_instance(
        x_row=x_row,
        tree_surrogate=surrogate_model,
        feature_names=feat_cols_eff,
        sigma=0.05,
        n_perturb=256,
        norm="l2",
        X_train=X_train
    )
    stability_percent = float(res_stab["stability_percent"])

    # ---------------- Sufficiency (1%,5%,10%,20%) ----------------
    res_suff = calcula_sufficiency_surrogate_tree(
        x_row=x_row,
        rf_model=model,
        tree_surrogate=surrogate_model,
        feature_names=feat_cols_eff,
        X_ref=X_train,
        class_index=POS_CLASS,
        percent_list=(0.01, 0.05, 0.10, 0.20),
        percent_mode="path",
        ranking="abs",
        masking="median_mode"
    )

    suff_1    = float(res_suff["scores"]["1%"])
    suff_5    = float(res_suff["scores"]["5%"])
    suff_10   = float(res_suff["scores"]["10%"])
    suff_20   = float(res_suff["scores"]["20%"])
    suff_aopc = float(res_suff["scores"]["AOPC_suff(%)"])

    return {
        "idx_instancia": idx_label,
        "fidelity_percent": fidelity_percent,
        "infidelity_percent": infidelity_percent,
        "faithfulness_percent": faithfulness_percent,
        "simplicity_percent": simplicity_percent,
        "consistency_percent": consistency_percent,
        "robusteness_percent": robusteness_percent,
        "completeness_percent": completeness_percent,
        "selectivity_percent": selectivity_percent,
        "soundness_percent": soundness_percent,
        "dir_soundness_percent": dir_soundness_percent,
        "stability_percent": stability_percent,
        "suff_1": suff_1,
        "suff_5": suff_5,
        "suff_10": suff_10,
        "suff_20": suff_20,
        "suff_aopc": suff_aopc,
        "abs_err": abs_err,
        "rmse_local": rmse_local,
    }

# ----------------------------------------------------------
# 5) Execução paralela sobre os IDs (labels) com N_JOBS
# ----------------------------------------------------------
RAM_GB   = psutil.virtual_memory().total / 1024**3
MAX_CPU  = multiprocessing.cpu_count()

NOTEBOOKS_RODANDO = 2   # ajuste se quiser
N_JOBS = 8  # ou algo como: max(1, min(MAX_CPU-1, 8))

results = Parallel(n_jobs=N_JOBS)(
    delayed(_compute_metrics_for_idx_label)(idx_label)
    for idx_label in IDX_LABELS
)

# ----------------------------------------------------------
# 6) Acumular resultados nas listas globais
# ----------------------------------------------------------
for res in results:
    idx_instancia        = res["idx_instancia"]
    fidelity_percent     = res["fidelity_percent"]
    infidelity_percent   = res["infidelity_percent"]
    faithfulness_percent = res["faithfulness_percent"]
    simplicity_percent   = res["simplicity_percent"]
    consistency_percent  = res["consistency_percent"]
    robusteness_percent  = res["robusteness_percent"]
    completeness_percent = res["completeness_percent"]
    selectivity_percent  = res["selectivity_percent"]
    soundness_percent    = res["soundness_percent"]
    dir_soundness_percent = res["dir_soundness_percent"]
    stability_percent    = res["stability_percent"]
    suff_1               = res["suff_1"]
    suff_5               = res["suff_5"]
    suff_10              = res["suff_10"]
    suff_20              = res["suff_20"]

    SURROGATE_TREE_FIDELITY_VALUES.append(fidelity_percent)
    SURROGATE_TREE_INFIDELITY_VALUES.append(infidelity_percent)
    SURROGATE_TREE_FAITHFULLNESS_VALUES.append(faithfulness_percent)
    SURROGATE_TREE_SIMPLICITY_VALUES.append(simplicity_percent)
    SURROGATE_TREE_CONSISTENCY_VALUES.append(consistency_percent)
    SURROGATE_TREE_ROBUSTENESS_VALUES.append(robusteness_percent)
    SURROGATE_TREE_COMPLETENESS_VALUES.append(completeness_percent)
    SURROGATE_TREE_SELECTIVITY_VALUES.append(selectivity_percent)
    SURROGATE_TREE_SOUNDNESS_VALUES.append(soundness_percent)
    SURROGATE_TREE_DIR_SOUNDNESS_VALUES.append(dir_soundness_percent)
    SURROGATE_TREE_STABILITY_VALUES.append(stability_percent)
    SURROGATE_TREE_SUFFICIENCY_1_VALUES.append(suff_1)
    SURROGATE_TREE_SUFFICIENCY_5_VALUES.append(suff_5)
    SURROGATE_TREE_SUFFICIENCY_10_VALUES.append(suff_10)
    SURROGATE_TREE_SUFFICIENCY_20_VALUES.append(suff_20)

    SURROGATE_TREE_TABLE_METRICS.append({
        'instancia': idx_instancia,
        'fidelity(%)': f"{fidelity_percent:.3f}",
        'infidelity(%)': f"{infidelity_percent:.3f}",
        'faithfulness(%)': f"{faithfulness_percent:.3f}",
        'simplicity(%)': f"{simplicity_percent:.3f}",
        'consistency(%)': f"{consistency_percent:.3f}",
        'robustez(%)': f"{robusteness_percent:.3f}",
        'completeness(%)': f"{completeness_percent:.3f}",
        'selectivity(%)': f"{selectivity_percent:.3f}",
        'soundness(%)':  f"{soundness_percent:.3f}",
        'directional soundness(%)': f"{dir_soundness_percent:.3f}",
        'stability(%)': f"{stability_percent:.3f}",
        'sufficiency-1(%)': f"{suff_1:.3f}",
        'sufficiency-5(%)': f"{suff_5:.3f}",
        'sufficiency-10(%)': f"{suff_10:.3f}",
        'sufficiency-20(%)': f"{suff_20:.3f}",
    })

# ---------- 7) Médias + ICs ----------
fidelity_ic_low, fidelity_ic_high           = calcular_ic95_percentual(SURROGATE_TREE_FIDELITY_VALUES)
infidelity_ic_low, infidelity_ic_high       = calcular_ic95_percentual(SURROGATE_TREE_INFIDELITY_VALUES)
faithfulness_ic_low, faithfulness_ic_high   = calcular_ic95_percentual(SURROGATE_TREE_FAITHFULLNESS_VALUES)
simplicity_ic_low, simplicity_ic_high       = calcular_ic95_percentual(SURROGATE_TREE_SIMPLICITY_VALUES)
consistency_ic_low, consistency_ic_high     = calcular_ic95_percentual(SURROGATE_TREE_CONSISTENCY_VALUES)
robustez_ic_low, robustness_ic_high         = calcular_ic95_percentual(SURROGATE_TREE_ROBUSTENESS_VALUES)
completeness_ic_low, completeness_ic_high   = calcular_ic95_percentual(SURROGATE_TREE_COMPLETENESS_VALUES)
selectivity_ic_low, selectivity_ic_high     = calcular_ic95_percentual(SURROGATE_TREE_SELECTIVITY_VALUES)
soundness_ic_low, soundness_ic_high         = calcular_ic95_percentual(SURROGATE_TREE_SOUNDNESS_VALUES)
dir_soundness_ic_low, dir_soundness_ic_high = calcular_ic95_percentual(SURROGATE_TREE_DIR_SOUNDNESS_VALUES)
stability_ic_low, stability_ic_high         = calcular_ic95_percentual(SURROGATE_TREE_STABILITY_VALUES)

SURROGATE_TREE_TABLE_METRICS.append({
    'instancia': 'MÉDIA',
    'fidelity(%)':           formatar_ic_percentual(np.mean(SURROGATE_TREE_FIDELITY_VALUES),           *calcular_ic95_percentual(SURROGATE_TREE_FIDELITY_VALUES)),
    'infidelity(%)':         formatar_ic_percentual(np.mean(SURROGATE_TREE_INFIDELITY_VALUES),         *calcular_ic95_percentual(SURROGATE_TREE_INFIDELITY_VALUES)),
    'faithfulness(%)':       formatar_ic_percentual(np.mean(SURROGATE_TREE_FAITHFULLNESS_VALUES),      *calcular_ic95_percentual(SURROGATE_TREE_FAITHFULLNESS_VALUES)),
    'simplicity(%)':         formatar_ic_percentual(np.mean(SURROGATE_TREE_SIMPLICITY_VALUES),         *calcular_ic95_percentual(SURROGATE_TREE_SIMPLICITY_VALUES)),
    'consistency(%)':        formatar_ic_percentual(np.mean(SURROGATE_TREE_CONSISTENCY_VALUES),        *calcular_ic95_percentual(SURROGATE_TREE_CONSISTENCY_VALUES)),
    'robustez(%)':           formatar_ic_percentual(np.mean(SURROGATE_TREE_ROBUSTENESS_VALUES),        *calcular_ic95_percentual(SURROGATE_TREE_ROBUSTENESS_VALUES)),
    'completeness(%)':       formatar_ic_percentual(np.mean(SURROGATE_TREE_COMPLETENESS_VALUES),       *calcular_ic95_percentual(SURROGATE_TREE_COMPLETENESS_VALUES)),
    'selectivity(%)':        formatar_ic_percentual(np.mean(SURROGATE_TREE_SELECTIVITY_VALUES),        *calcular_ic95_percentual(SURROGATE_TREE_SELECTIVITY_VALUES)),
    'soundness(%)':          formatar_ic_percentual(np.mean(SURROGATE_TREE_SOUNDNESS_VALUES),          *calcular_ic95_percentual(SURROGATE_TREE_SOUNDNESS_VALUES)),
    'directional soundness(%)': formatar_ic_percentual(np.mean(SURROGATE_TREE_DIR_SOUNDNESS_VALUES),   *calcular_ic95_percentual(SURROGATE_TREE_DIR_SOUNDNESS_VALUES)),
    'stability(%)':          formatar_ic_percentual(np.mean(SURROGATE_TREE_STABILITY_VALUES),          *calcular_ic95_percentual(SURROGATE_TREE_STABILITY_VALUES)),
    'sufficiency-1(%)':      formatar_ic_percentual(np.mean(SURROGATE_TREE_SUFFICIENCY_1_VALUES),      *calcular_ic95_percentual(SURROGATE_TREE_SUFFICIENCY_1_VALUES)),
    'sufficiency-5(%)':      formatar_ic_percentual(np.mean(SURROGATE_TREE_SUFFICIENCY_5_VALUES),      *calcular_ic95_percentual(SURROGATE_TREE_SUFFICIENCY_5_VALUES)),
    'sufficiency-10(%)':     formatar_ic_percentual(np.mean(SURROGATE_TREE_SUFFICIENCY_10_VALUES),     *calcular_ic95_percentual(SURROGATE_TREE_SUFFICIENCY_10_VALUES)),
    'sufficiency-20(%)':     formatar_ic_percentual(np.mean(SURROGATE_TREE_SUFFICIENCY_20_VALUES),     *calcular_ic95_percentual(SURROGATE_TREE_SUFFICIENCY_20_VALUES)),
})

SURROGATE_TREE_TABLE_METRICS.append({
    'instancia': 'MEDIANA',
    'fidelity(%)':           formatar_ic_percentual(np.median(SURROGATE_TREE_FIDELITY_VALUES),         *calcular_ic95_percentual_median(SURROGATE_TREE_FIDELITY_VALUES)),
    'infidelity(%)':         formatar_ic_percentual(np.median(SURROGATE_TREE_INFIDELITY_VALUES),       *calcular_ic95_percentual_median(SURROGATE_TREE_INFIDELITY_VALUES)),
    'faithfulness(%)':       formatar_ic_percentual(np.median(SURROGATE_TREE_FAITHFULLNESS_VALUES),    *calcular_ic95_percentual_median(SURROGATE_TREE_FAITHFULLNESS_VALUES)),
    'simplicity(%)':         formatar_ic_percentual(np.median(SURROGATE_TREE_SIMPLICITY_VALUES),       *calcular_ic95_percentual_median(SURROGATE_TREE_SIMPLICITY_VALUES)),
    'consistency(%)':        formatar_ic_percentual(np.median(SURROGATE_TREE_CONSISTENCY_VALUES),      *calcular_ic95_percentual_median(SURROGATE_TREE_CONSISTENCY_VALUES)),
    'robustez(%)':           formatar_ic_percentual(np.median(SURROGATE_TREE_ROBUSTENESS_VALUES),      *calcular_ic95_percentual_median(SURROGATE_TREE_ROBUSTENESS_VALUES)),
    'completeness(%)':       formatar_ic_percentual(np.median(SURROGATE_TREE_COMPLETENESS_VALUES),     *calcular_ic95_percentual_median(SURROGATE_TREE_COMPLETENESS_VALUES)),
    'selectivity(%)':        formatar_ic_percentual(np.median(SURROGATE_TREE_SELECTIVITY_VALUES),      *calcular_ic95_percentual_median(SURROGATE_TREE_SELECTIVITY_VALUES)),
    'soundness(%)':          formatar_ic_percentual(np.median(SURROGATE_TREE_SOUNDNESS_VALUES),        *calcular_ic95_percentual_median(SURROGATE_TREE_SOUNDNESS_VALUES)),
    'directional soundness(%)': formatar_ic_percentual(np.median(SURROGATE_TREE_DIR_SOUNDNESS_VALUES), *calcular_ic95_percentual_median(SURROGATE_TREE_DIR_SOUNDNESS_VALUES)),
    'stability(%)':          formatar_ic_percentual(np.median(SURROGATE_TREE_STABILITY_VALUES),        *calcular_ic95_percentual_median(SURROGATE_TREE_STABILITY_VALUES)),
    'sufficiency-1(%)':      formatar_ic_percentual(np.median(SURROGATE_TREE_SUFFICIENCY_1_VALUES),    *calcular_ic95_percentual_median(SURROGATE_TREE_SUFFICIENCY_1_VALUES)),
    'sufficiency-5(%)':      formatar_ic_percentual(np.median(SURROGATE_TREE_SUFFICIENCY_5_VALUES),    *calcular_ic95_percentual_median(SURROGATE_TREE_SUFFICIENCY_5_VALUES)),
    'sufficiency-10(%)':     formatar_ic_percentual(np.median(SURROGATE_TREE_SUFFICIENCY_10_VALUES),   *calcular_ic95_percentual_median(SURROGATE_TREE_SUFFICIENCY_10_VALUES)),
    'sufficiency-20(%)':     formatar_ic_percentual(np.median(SURROGATE_TREE_SUFFICIENCY_20_VALUES),   *calcular_ic95_percentual_median(SURROGATE_TREE_SUFFICIENCY_20_VALUES)),
})

fim = time.time()
print(f"Tempo de execução: {fim - inicio:.4f} segundos")

df_surrogate_tree_metrics = pd.DataFrame(SURROGATE_TREE_TABLE_METRICS)
df_surrogate_tree_metrics.to_csv(f'{BASE_PATH}/SURROGATE_TREE_METRICS.csv', index=False)

output_file = f"{BASE_PATH}/{BASE_PATH}_SURROGATE_completo.xlsx"
df_surrogate_tree_metrics.to_excel(output_file, index=False)
print(f"✅ Arquivo salvo com sucesso em: {output_file}")

df_surrogate_tree_metrics


In [ ]:
# ===================== contribuições LIME + Tabela de métricas (instância / média / mediana) =====================
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lime.lime_tabular import LimeTabularExplainer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")
import matplotlib.image as mpimg
import glob

# --------------------------------- utils básicos ---------------------------------
def _coerce_binary_series(s):
    s = pd.Series(s).copy()
    map_true  = {"1", 1, True, "true", "True", "sim", "Sim", "yes", "Yes", "y", "Y", "positivo", "Positivo"}
    map_false = {"0", 0, False, "false", "False", "nao", "não", "Nao", "Não", "no", "No", "n", "N", "negativo", "Negativo"}
    def _m(v):
        sv = str(v).strip()
        if sv in map_true:  return 1
        if sv in map_false: return 0
        return v
    s = s.map(_m)
    s = pd.to_numeric(s, errors="coerce")
    return s

THRESHOLD = 0.5  # caso derive y a partir de probabilidade

TOP_K_LIME = 10  # número de features a exibir no plot
globals
def _infer_y_from_X(Xdf, threshold=0.5):
    """
    Extrai y de colunas em X:
      (1) alvo explícito (y/target/label/classe/...),
      (2) probabilidade única (y_proba/proba_1/score_pos/...),
      (3) fallback: coluna ~[0,1] vira proba->y (>= threshold).
    Retorna (y_series, cols_to_drop).
    """
    TARGET_CANDIDATES = ['y','classe','target','label','y_true','diagnosis','outcome']
    PROBA_CANDIDATES  = ['y_proba','proba','proba_1','p1','prob_1','prob_pos','score','score_pos','pred_proba_1']

    # 1) alvo explícito
    for col in TARGET_CANDIDATES:
        if col in Xdf.columns:
            y_raw = Xdf[col]
            y_bin = _coerce_binary_series(y_raw).rename('y')
            return y_bin, [col]

    # 2) probabilidade em única coluna
    for col in PROBA_CANDIDATES:
        if col in Xdf.columns:
            y_bin = (pd.to_numeric(Xdf[col], errors='coerce') >= threshold).astype('Int64').rename('y')
            return y_bin, [col]

    # 3) fallback: heurística p/ coluna [0,1]
    for col in Xdf.columns:
        s = pd.to_numeric(Xdf[col], errors='coerce')
        if s.notna().mean() > 0.9 and (s.between(0, 1, inclusive='both').mean() > 0.9):
            y_bin = (s >= threshold).astype('Int64').rename('y')
            return y_bin, [col]

    raise ValueError("Não encontrei y nem y_proba dentro de X para alinhar o alvo.")

# --------------------------------- preparo do X (remove alvos e probas) ---------------------------------
try:
    EXCLUDE_COLUMNS
except NameError:
    EXCLUDE_COLUMNS = []

y_inferido, y_cols_to_drop = _infer_y_from_X(X)

TARGET_LIKE = ['y','classe','target','label','y_true','diagnosis','outcome']
PROBA_LIKE  = ['y_proba','proba','proba_0','proba_1','prob_0','prob_1','p0','p1',
               'score','score_pos','score_neg','pred_proba_0','pred_proba_1']

cols_drop   = [c for c in (EXCLUDE_COLUMNS + TARGET_LIKE + PROBA_LIKE + y_cols_to_drop) if c in X.columns]
X_feat_full = X.drop(columns=list(set(cols_drop)), errors='ignore').copy()

# --- alinhamento com o modelo: mesmas colunas e ordem que o modelo usou no treino ---
if hasattr(model, "feature_names_in_"):
    feat_cols_eff = [c for c in model.feature_names_in_.tolist() if c in X_feat_full.columns]
    missing = [c for c in model.feature_names_in_.tolist() if c not in X_feat_full.columns]
    if len(missing) > 0:
        raise ValueError(f"As seguintes colunas esperadas pelo modelo não estão em X: {missing}")
    X_feat = X_feat_full.loc[:, feat_cols_eff].copy()
else:
    if hasattr(model, "n_features_in_"):
        n_expected = int(model.n_features_in_)
        if X_feat_full.shape[1] > n_expected:
            X_feat = X_feat_full.iloc[:, :n_expected].copy()
        elif X_feat_full.shape[1] == n_expected:
            X_feat = X_feat_full.copy()
        else:
            raise ValueError(
                f"X tem {X_feat_full.shape[1]} features, mas o modelo espera {n_expected}. "
                f"Adicione as colunas faltantes ou informe feature_names_in_."
            )
    else:
        X_feat = X_feat_full.copy()

# y alinhado
y_alinhado = pd.Series(y_inferido).reindex(X_feat.index).astype('Int64').rename('y')

# --------------------------------- split 8/1/1 consistente ---------------------------------
# Se já vieram do pipeline 8/1/1, apenas reutilize e garanta as colunas corretas:
if all(name in globals() for name in ["X_train", "X_test", "y_train", "y_test"]):
    X_train = X_train.loc[:, X_feat.columns]
    X_test  = X_test.loc[:,  X_feat.columns]
    X_train_real, X_test_real = X_train, X_test
else:
    # Reconstrói 8/1/1 localmente (estratificado)
    # 1) separa 10% para teste
    idx_trval, idx_tst = train_test_split(
        X_feat.index, test_size=0.10, stratify=y_alinhado, random_state=42
    )
    # 2) dos 90% restantes, separa 1/9 (~10%) para validação => sobra ~80% treino
    y_trval = y_alinhado.loc[idx_trval]
    idx_tr, idx_val = train_test_split(
        idx_trval, test_size=(1/9), stratify=y_trval, random_state=42
    )

    X_train = X_feat.loc[idx_tr].copy()          # ~80%
    y_train = y_alinhado.loc[idx_tr].copy()
    X_val   = X_feat.loc[idx_val].copy()         # ~10% (não usado aqui, mas preservado)
    y_val   = y_alinhado.loc[idx_val].copy()
    X_test  = X_feat.loc[idx_tst].copy()         # ~10%
    y_test  = y_alinhado.loc[idx_tst].copy()

    X_train_real, X_test_real = X_train, X_test

# --------------------------------- nomes de classes ---------------------------------
unique_classes = sorted(pd.unique(y_alinhado.dropna()))
if len(unique_classes) == 2 and set(unique_classes) <= {0,1}:
    class_names = ['Risco Baixo','Risco Alto']
else:
    class_names = [f'Classe {c}' for c in unique_classes]

# --------------------------------- LIME explainer (treinado com 100%) ---------------------------------

# --------------------------------- LIME explainer (treinado com 100%) ---------------------------------
#if 'lime_explainer' not in globals():
#    # se não existir, cria usando o X_train atual
#    lime_explainer = LimeTabularExplainer(
#        training_data=X_train.values,
#        feature_names=X_train.columns.tolist(),
#        class_names=class_names,
#        mode='classification',
#        discretize_continuous=True,
#        random_state=42
#    )

# lista de features que o explainer conhece (padrão a ser seguido)
#FEATURES_LIME = list(
#    getattr(lime_explainer, "feature_names", X_train.columns.tolist())
#)

# Probabilidade com MESMAS colunas/ordem do treino
#def _clf_proba(arr2d):
#    return model.predict_proba(pd.DataFrame(arr2d, columns=X_train.columns))
#FEATURES_LIME = lime_explainer.feature_names
#def _clf_proba(arr2d):
#    df = pd.DataFrame(arr2d, columns=FEATURES_LIME)
    # garante que a ordem das colunas é exatamente FEATURES_LIME
#    return model.predict_proba(df[FEATURES_LIME])

# --------------------------------- métricas: helpers ---------------------------------
METRIC_COLS = [
    'fidelity(%)','infidelity(%)','faithfulness(%)','simplicity(%)','consistency(%)',
    'robustez(%)','stability(%)','selectivity(%)','soundness(%)','directional soundness(%)',
    'sufficiency-1(%)','sufficiency-5(%)','sufficiency-10(%)','sufficiency-20(%)','completeness(%)'
]

_num_regex = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")

def _to_float_or_nan(x):
    if pd.isna(x): return np.nan
    if isinstance(x,(int,float,np.number)): return float(x)
    s = str(x)
    m = _num_regex.search(s)
    return float(m.group(0)) if m else np.nan

def get_metrics_instance_mean_median(df_metrics, inst_idx):
    row_inst = df_metrics.loc[df_metrics['instancia'] == inst_idx]
    row_mean = df_metrics.loc[df_metrics['instancia'].astype(str).str.upper().eq('MÉDIA')]
    row_medi = df_metrics.loc[df_metrics['instancia'].astype(str).str.upper().eq('MEDIANA')]

    if row_mean.empty or row_medi.empty:
        tmp = df_metrics.copy()
        for c in METRIC_COLS:
            if c in tmp.columns:
                tmp[c] = tmp[c].map(_to_float_or_nan)
        only_num = tmp.loc[pd.to_numeric(tmp['instancia'], errors='coerce').notna(), METRIC_COLS]
        mean_vals = only_num.mean(numeric_only=True)
        medi_vals = only_num.median(numeric_only=True)
    else:
        mean_vals = row_mean.iloc[0][METRIC_COLS].map(_to_float_or_nan)
        medi_vals = row_medi.iloc[0][METRIC_COLS].map(_to_float_or_nan)

    inst_vals = pd.Series({c: np.nan for c in METRIC_COLS})
    if not row_inst.empty:
        inst_vals = row_inst.iloc[0][METRIC_COLS].map(_to_float_or_nan)

    out = pd.DataFrame({
        "metric": METRIC_COLS,
        "inst":  [inst_vals.get(c, np.nan) for c in METRIC_COLS],
        "mean":  [mean_vals.get(c, np.nan) for c in METRIC_COLS],
        "median":[medi_vals.get(c, np.nan) for c in METRIC_COLS],
    })
    return out


def plot_surrogate_tree_with_metrics(inst_id, df_metrics, out_dir=None):
    """
    Plota, para um dado inst_id:
      - imagem da árvore surrogate GLOBAL (caminho da instância)
      - tabela de métricas (instância / média / mediana)
      - bloco superior com prob RF (baixo/alto)

    inst_id   : valor da coluna 'id' no dataset principal e em df_metrics['instancia']
    df_metrics: df_surrogate_tree_metrics
    out_dir   : diretório onde estão as imagens das árvores surrogate
    """
    if out_dir is None:
        out_dir = OUT_DIR if 'OUT_DIR' in globals() else "."

    # -------------------- 1) Achar índice (label) no X --------------------
    if 'id' in X.columns:
        idx_candidates = X.index[X['id'] == inst_id]
        if len(idx_candidates) == 0:
            raise ValueError(f"Não achei instância com id={inst_id} em X['id'].")
        idx_label = idx_candidates[0]
    else:
        # fallback: supor que o índice de X é o próprio inst_id
        if inst_id not in X.index:
            raise ValueError(f"inst_id={inst_id} não está em X.index e não há coluna 'id'.")
        idx_label = inst_id

    # -------------------- 2) Probabilidade do RF (Risco Baixo / Alto) --------------------
    if 'feat_cols_eff' not in globals():
        raise NameError("feat_cols_eff não está definido no ambiente. Ele deve conter as colunas usadas pelo modelo.")

    x_row = X.loc[idx_label, feat_cols_eff]
    df_feat = pd.DataFrame([x_row], columns=feat_cols_eff)

    probs = model.predict_proba(df_feat)[0]
    classes = list(model.classes_)

    # assume binário 0/1
    if 1 in classes:
        pos_idx = classes.index(1)
    else:
        pos_idx = int(np.argmax(probs))

    if 0 in classes:
        neg_idx = classes.index(0)
    else:
        neg_idx = 1 - pos_idx if len(classes) > 1 else 0

    p_pos = float(probs[pos_idx])
    p_neg = float(probs[neg_idx])

    # -------------------- 3) Métricas (instância / média / mediana) --------------------
    mt = get_metrics_instance_mean_median(df_metrics, inst_id)

    # -------------------- 4) Localizar imagem da árvore surrogate --------------------
    # Procurar qualquer arquivo com padrão *ID{inst_id}.png no out_dir
    pattern = os.path.join(out_dir, f"*ID{inst_id}*.png")
    matches = glob.glob(pattern)
    if not matches:
        raise FileNotFoundError(
            f"Não encontrei imagem da árvore surrogate para id={inst_id} com padrão: {pattern}"
        )
    img_path = matches[0]
    img = mpimg.imread(img_path)

    # -------------------- 5) Figura combinando Árvore + Métricas --------------------
    fig = plt.figure(figsize=(14.0, 6.0), dpi=300)
    gs = fig.add_gridspec(1, 2, width_ratios=[3.0, 2.4], wspace=0.22)

    # ---- Título principal ----
    fig.suptitle(
        f"Surrogate – Instância id: {inst_id}",
        fontsize=20,
        fontweight="bold",
        x=0.02,
        ha="left",
        y=0.96
    )

    # ---- Blocos de risco no topo (como no LIME) ----
    txt_p_neg = f"{p_neg:.2f}".replace(".", ",")
    txt_p_pos = f"{p_pos:.2f}".replace(".", ",")

    fig.text(0.60, 0.94, "Risco Baixo",
             fontsize=14, fontweight="bold",
             color="#1f77ff", ha="center", va="bottom")
    fig.text(0.60, 0.89, txt_p_neg,
             fontsize=18, fontweight="bold",
             color="#1f77ff", ha="center", va="top")

    fig.text(0.82, 0.94, "Risco Alto",
             fontsize=14, fontweight="bold",
             color="#ff7f0e", ha="center", va="bottom")
    fig.text(0.82, 0.89, txt_p_pos,
             fontsize=18, fontweight="bold",
             color="#ff7f0e", ha="center", va="top")

    # ------------------------------------------------------------------
    # 1) Painel esquerdo: IMAGEM DA ÁRVORE SURROGATE GLOBAL
    # ------------------------------------------------------------------
    # 1) Painel esquerdo: IMAGEM DA ÁRVORE SURROGATE GLOBAL
    ax_tree = fig.add_subplot(gs[0, 0])

    # desenha a imagem
    ax_tree.imshow(img)

    ax_tree = fig.add_axes([-0.2, 0.18, 0.75, 0.75])  # [left, bottom, width, height] em coord. da figura (0–1)
    ax_tree.imshow(img)
    ax_tree.axis("off")

    # ------------------------------------------------------------------
    # 2) Painel direito: Tabela de MÉTRICAS (instância, média, mediana)
    # ------------------------------------------------------------------
    axm = fig.add_subplot(gs[0, 1])
    axm.axis("off")

    x_feat    = 0.00
    x_inst    = 0.95
    x_media   = 1.40
    x_mediana = 1.80

    axm.text(x_feat,    0.98, "Métrica",   fontsize=11, fontweight="bold", ha="left",  va="top")
    axm.text(x_inst,    0.98, "Valor",     fontsize=11, fontweight="bold", ha="right", va="top")
    axm.text(x_media,   0.98, "Média",     fontsize=11, fontweight="bold", ha="right", va="top")
    axm.text(x_mediana, 0.98, "Mediana",   fontsize=11, fontweight="bold", ha="right", va="top")

    def _fmt_metric(v):
        return "0,00" if pd.isna(v) else f"{float(v):.3f}".replace(".", ",")

    y0, dy = 0.92, 0.055
    for i, row in mt.iterrows():
        if y0 - i*dy < 0.05:
            break

        axm.text(x_feat,  y0 - i*dy, row['metric'],              ha="left",  va="top", fontsize=9)
        axm.text(x_inst,  y0 - i*dy, _fmt_metric(row['inst']),   ha="right", va="top", fontsize=9)
        axm.text(x_media, y0 - i*dy, _fmt_metric(row['mean']),   ha="right", va="top", fontsize=9)
        axm.text(x_mediana, y0 - i*dy, _fmt_metric(row['median']), ha="right", va="top", fontsize=9)

    plt.subplots_adjust(top=0.82)
    plt.show()
    plt.close(fig)  



# ----------------- rodar para as suas instâncias -----------------
try:
    ENTRADAS_SELECIONADAS
except NameError:
    ENTRADAS_SELECIONADAS = [0]

ENTRADAS_SELECIONADAS = np.asarray(list(ENTRADAS_SELECIONADAS), dtype=int)
# ENTRADAS_SELECIONADAS = ENTRADAS_SELECIONADAS[
#     (ENTRADAS_SELECIONADAS >= 0) & (ENTRADAS_SELECIONADAS < len(X_test))
# ]

# df_lime_metrics deve existir; se não existir, cria um placeholder vazio só para o plot não quebrar
# if 'df_lime_metrics' not in globals():
#     df_lime_metrics = pd.DataFrame({"instancia": [], **{c:[] for c in METRIC_COLS}})

import gc  # coloque lá em cima com os imports

BATCH_SIZE = 5

total = len(ENTRADAS_SELECIONADAS)
for start in range(0, total, BATCH_SIZE):
    batch = ENTRADAS_SELECIONADAS[start:start + BATCH_SIZE]
    print(f"[SURROGATE] Processando instâncias {start}–{start+len(batch)-1} de {total}")

    for idx_pos in batch:
        inst_id = int(idx_pos)  # ou: int(X.iloc[idx_pos]['id']) se seu id vier da coluna 'id'
        plot_surrogate_tree_with_metrics(inst_id, df_surrogate_tree_metrics, out_dir=OUT_DIR)

    # força coleta de lixo entre lotes para reduzir pico de memória
    gc.collect()


In [ ]:
# === Surrogate Decision Tree — Heatmap das Métricas ===
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path

# 1) Escolha da fonte dos dados
df_src = df_surrogate_tree_metrics.copy()

# 2) Remove linhas agregadas
mask_inst = (~df_src["instancia"].isin(["MÉDIA", "MEDIANA"]))
df_f = df_src.loc[mask_inst].copy()

# 3) Remova colunas que você não quer no heatmap (ajuste à vontade)
drop_optional = ["instancia"]  # acrescente aqui caso queira excluir algo específico
df_f.drop(columns=[c for c in drop_optional if c in df_f.columns], inplace=True, errors="ignore")

# 4) Função para extrair o primeiro número (float) de uma string
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
def _coerce_first_float(x):
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        m = _num_re.search(x.replace(",", "."))  # também lida com vírgula decimal, se houver
        return float(m.group(0)) if m else np.nan
    return np.nan

# 5) Converte todas as colunas “percentuais/numéricas” para float
df_num = df_f.applymap(_coerce_first_float)

# 6) Remove colunas constantes (ex.: só 100 ou só 0) ou totalmente NaN
const_cols = []
for c in df_num.columns:
    col = df_num[c].dropna()
    if col.empty or col.nunique(dropna=True) <= 1:
        const_cols.append(c)

df_num = df_num.drop(columns=const_cols, errors="ignore")

if df_num.shape[1] == 0:
    raise ValueError("Não sobraram métricas numéricas variáveis para correlacionar.")

# 7) Correlação de Pearson
corr_matrix = df_num.corr(method="pearson")

# 8) Mantém só correlações “fortes” (|r| > limiar)
corr_threshold = 0.30
keep = (corr_matrix > corr_threshold) | (corr_matrix < -corr_threshold)

# Máscara (True = esconder)
mask = ~keep
np.fill_diagonal(mask.values, True)  # opcional: esconder diagonal

# 9) Ordenação das métricas (mais conectadas primeiro)
order = corr_matrix.abs().sum().sort_values(ascending=False).index
ordered = corr_matrix.loc[order, order]
mask_ordered = mask.loc[order, order]

# 10) Plot
plt.figure(figsize=(14, 10))
sns.heatmap(
    ordered,
    annot=True, fmt=".2f",
    cmap="coolwarm", vmin=-1, vmax=1,
    mask=mask_ordered, linewidths=0.5, cbar_kws={"shrink": 0.85}
)
plt.title(f"Heatmap de Correlação (Pearson) entre Métricas do Surrogate — |r| > {corr_threshold}")
plt.tight_layout()

# 11) Salva
out_dir = PATH_FILE_OUT if "PATH_FILE_OUT" in globals() else "."
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "surrogate_metrics_heatmap_filtrado.png")
plt.savefig(out_path, dpi=180)

# também salva em BASE_PATH, se existir
if "BASE_PATH" in globals():
    base_dir = Path(BASE_PATH)
    base_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(base_dir / "surrogate_metrics_heatmap_filtrado.png", dpi=180)

plt.show()

# (Opcional) para você inspecionar o que entrou:
print("Colunas removidas por serem constantes/NaN:", const_cols)
print("Métricas usadas no heatmap:", list(ordered.columns))
print("Arquivo salvo em:", out_path)
if "BASE_PATH" in globals():
    print("Arquivo também salvo em:", base_dir / "surrogate_metrics_heatmap_filtrado.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
import os
from pathlib import Path

# --- escolha: usar correlação absoluta (ignora o sinal) ou assinada
use_abs = True   # True => distância = 1 - |corr| ; False => 1 - corr

# matriz de correlação -> numpy
C = corr_matrix.to_numpy()
C = np.abs(C) if use_abs else C

# garante simetria exata e diagonal = 1.0 (evita ruído numérico)
C = 0.5 * (C + C.T)
np.fill_diagonal(C, 1.0)

# distância (0=correlação perfeita; 1=sem correlação se use_abs=True)
D = 1.0 - C
np.fill_diagonal(D, 0.0)

# por segurança, trunca qualquer valor negativo residual para 0
D = np.maximum(D, 0.0)

# linkage precisa do formato "condensed"
condensed_D = squareform(D, checks=False)

# método de ligação; "average" é estável para distâncias de correlação
Z = hierarchy.linkage(condensed_D, method='average')

# diretórios de saída
out_dir = PATH_FILE_OUT if "PATH_FILE_OUT" in globals() else "."
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "dendrogram_metricas.png")

# dendrograma
plt.figure(figsize=(12, 6))
dn = hierarchy.dendrogram(Z, labels=corr_matrix.columns.tolist(), leaf_rotation=90)
plt.title(f"Dendrograma das métricas (distância = 1 - {'|corr|' if use_abs else 'corr'})")
plt.ylabel("Distância")
plt.tight_layout()
plt.savefig(out_path, dpi=150)

# também salva em BASE_PATH, se existir
if "BASE_PATH" in globals():
    base_dir = Path(BASE_PATH)
    base_dir.mkdir(parents=True, exist_ok=True)
    plt.savefig(base_dir / "dendrogram_metricas.png", dpi=150)

plt.show()

# (opcional) ordem das métricas segundo o dendrograma
order = [corr_matrix.columns[i] for i in dn["leaves"]]
print(order)
print("Arquivo salvo em:", out_path)
if "BASE_PATH" in globals():
    print("Arquivo também salvo em:", base_dir / "dendrogram_metricas.png")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.spatial.distance import squareform
import os
from pathlib import Path

def venn_from_corr_mds_labels(
    corr_matrix,
    radius=1.0,               # raio (mesmo para todos)
    epsilon_frac=0.06,        # ε aplicado SÓ quando r>0: d <- d + ε*R
    sign_mode="both",         # "both" | "positive" | "negative"
    use_abs=False,            # se True, ignora sinais (sign_mode fica sem efeito)
    random_state=42,
    n_init=8,
    max_iter=600,
    figsize=(8,8),
    face_alpha=0.30,
    edge_lw=1.5,
    label_fontsize=8,         # fonte 8
    label_weight='normal',    # sem negrito
    label_jitter=0.03,        # jitter vertical inicial (fração do raio)
    repel_iters=150,          # iterações de repulsão dos rótulos
    repel_step=0.02,          # passo da repulsão (fração do raio)
    inside_margin=0.12,       # margem p/ manter texto dentro do círculo
    title="Venn adaptado (d = (1 − r)·R, ε>0 só para r>0, filtro de sinal)"
):
    """
    Distância alvo entre centros: d(r) = (1 − r)*R  (+ ε*R se r>0).
    - sign_mode:
        * "both": usa r como está;
        * "positive": se r<=0, força r_eff=-1 (distância máxima 2R);
        * "negative": se r>=0, força r_eff=-1 (distância máxima 2R).
    - use_abs=True ignora sinais e, portanto, sign_mode não tem efeito.
    """

    # ---------- 1) correlação -> distância alvo (com epsilon e filtro de sinal) ----------
    C_raw = corr_matrix.to_numpy(dtype=float)
    C_raw = np.clip(C_raw, -1.0, 1.0)
    n = C_raw.shape[0]

    # máscara de r>0 (para epsilon)
    mask_pos = (C_raw > 0)

    if use_abs:
        C_eff = np.abs(C_raw)
    else:
        # aplica filtro de sinal
        if sign_mode == "positive":
            # mantém só r>0; demais vão para -1 (distância máxima 2R)
            C_eff = np.where(C_raw > 0, C_raw, -1.0)
        elif sign_mode == "negative":
            # mantém só r<0; demais vão para -1
            C_eff = np.where(C_raw < 0, C_raw, -1.0)
        else:
            C_eff = C_raw

    R = float(radius)
    T = (1.0 - C_eff) * R  # d(r) base
    # aplica ε apenas para r>0 (com base no sinal original, como pedido)
    if epsilon_frac and epsilon_frac > 0:
        T = T + (epsilon_frac * R) * mask_pos.astype(float)

    # simetriza e zera diagonal
    T = 0.5 * (T + T.T)
    np.fill_diagonal(T, 0.0)

    # ---------- 2) embedding por MDS (SMACOF; fallback p/ MDS clássico) ----------
    try:
        from sklearn.manifold import MDS
        mds = MDS(
            n_components=2, metric=True, dissimilarity='precomputed',
            n_init=n_init, max_iter=max_iter, random_state=random_state
        )
        X = mds.fit_transform(T)
    except Exception:
        # MDS clássico (Torgerson)
        J = np.eye(n) - np.ones((n, n))/n
        B = -0.5 * J @ (T**2) @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        L = np.clip(eigvals[idx][:2], 0, None)
        V = eigvecs[:, idx][:, :2]
        X = V * np.sqrt(L + 1e-12)

    # reescala global para casar melhor T
    DX = np.sqrt(((X[:,None,:] - X[None,:,:])**2).sum(axis=2))
    iu = np.triu_indices_from(DX, k=1)
    num = np.sum(DX[iu] * T[iu])
    den = np.sum(DX[iu]**2) + 1e-12
    s = num/den if den > 0 else 1.0
    X *= s
    X -= X.mean(axis=0, keepdims=True)  # centraliza

    labels = corr_matrix.columns.tolist()

    # ---------- 3) paleta de cores ----------
    def _lc(s): return str(s).strip().lower()
    color_fixed = {
        "robustness":            "#9467bd",
        "stability":             "#17becf",
        "soundness":             "#f7b6d2",
        "directional soundness": "#ff9896",
        "selectivity":           "#c7c7c7",
        "fidelity":              "#2ca02c",
        "infidelity":            "#d62728",
    }
    cmap = plt.get_cmap("tab20")
    colors, k_auto = [], 0
    for name in labels:
        key = _lc(name)
        colors.append(color_fixed.get(key, cmap(k_auto % 20)))
        if key not in color_fixed:
            k_auto += 1

    # ---------- 4) desenha círculos (mesmo raio) ----------
    fig, ax = plt.subplots(figsize=figsize, dpi=150, facecolor="white")
    for (x, y), col in zip(X, colors):
        circ = Circle((x, y), R, facecolor=col, edgecolor=col,
                      alpha=face_alpha, lw=edge_lw)
        ax.add_patch(circ)

    # ---------- 5) rótulos centralizados + repulsão vertical ----------
    rng = np.random.default_rng(random_state)
    text_xy = X.copy()
    text_xy[:, 1] += (rng.uniform(-label_jitter, label_jitter, size=n)) * R

    def clamp_inside(i):
        cx, cy = X[i]
        tx, ty = text_xy[i]
        vx, vy = tx - cx, ty - cy
        dist = np.hypot(vx, vy)
        maxd = R * (1.0 - inside_margin)
        if dist > maxd and dist > 1e-12:
            k = maxd / dist
            text_xy[i] = np.array([cx + k*vx, cy + k*vy])

    for _ in range(repel_iters):
        moved = False
        for i in range(n):
            for j in range(i+1, n):
                xi, yi = text_xy[i]
                xj, yj = text_xy[j]
                # caixa aproximada do texto
                w = 0.9 * R
                h = 0.35 * R
                if (abs(xi - xj) < w) and (abs(yi - yj) < h):
                    dy = repel_step * R
                    text_xy[i, 1] += dy
                    text_xy[j, 1] -= dy
                    clamp_inside(i)
                    clamp_inside(j)
                    moved = True
        if not moved:
            break

    for (tx, ty), name in zip(text_xy, labels):
        ax.text(tx, ty, name, ha="center", va="center",
                fontsize=label_fontsize, fontweight=label_weight,
                color="black", zorder=3)

    # limites e estética
    xmin = X[:,0].min() - R*1.3
    xmax = X[:,0].max() + R*1.3
    ymin = X[:,1].min() - R*1.3
    ymax = X[:,1].max() + R*1.3
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()

    # ---------- 6) salvar usando PATH_FILE_OUT e BASE_PATH ----------
    filename = "venn_corr_mds_labels.png"

    # PATH_FILE_OUT (se existir)
    if "PATH_FILE_OUT" in globals():
        out_dir = PATH_FILE_OUT
    else:
        out_dir = "."

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path, dpi=180)

    # também salva em BASE_PATH, se existir
    if "BASE_PATH" in globals():
        base_dir = Path(BASE_PATH)
        base_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(base_dir / filename, dpi=180)

    plt.show()

    print("Figura Venn salva em:", out_path)
    if "BASE_PATH" in globals():
        print("Figura também salva em:", base_dir / filename)


# ===== Exemplo de uso =====
# 2) só correlações positivas:
venn_from_corr_mds_labels(
    corr_matrix,
    radius=1.0,
    epsilon_frac=0.5,
    sign_mode="positive",
    use_abs=False
)

# 3) só correlações negativas (se quiser rodar depois):
# venn_from_corr_mds_labels(corr_matrix, radius=1.0, epsilon_frac=4, sign_mode="negative", use_abs=False)


In [ ]:
import re
import unicodedata
import difflib
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from scipy.spatial.distance import squareform
import os
from pathlib import Path

# --- normalizador de nomes (case/acentos/espacos/simbolos) ---
def _normalize_name(s: str) -> str:
    s = str(s)
    # remove acentos
    s = unicodedata.normalize('NFKD', s)
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    s = s.lower().strip()
    # remove marcadores como "(%)", "%", "()", etc
    s = re.sub(r'\(\s*%\s*\)', '', s)
    s = s.replace('%', '')
    # remove espaços, underscores e hifens
    s = re.sub(r'[\s_\-]+', '', s)
    # mantém apenas alfa-num
    s = re.sub(r'[^a-z0-9]', '', s)
    return s

def _resolve_metric_names(corr_columns, requested_names):
    """Mapeia nomes pedidos para colunas reais da corr_matrix via normalização + fuzzy."""
    # dicionário: normalizado -> nome original da coluna
    norm_to_col = {}
    for col in corr_columns:
        norm_to_col.setdefault(_normalize_name(col), col)

    resolved = []
    missing  = []
    suggestions = {}

    for name in requested_names:
        key = _normalize_name(name)
        if key in norm_to_col:
            resolved.append(norm_to_col[key])
        else:
            # tenta fuzzy contra as chaves normalizadas
            choices = list(norm_to_col.keys())
            match = difflib.get_close_matches(key, choices, n=1, cutoff=0.6)
            if match:
                resolved.append(norm_to_col[match[0]])
                suggestions[name] = norm_to_col[match[0]]
            else:
                missing.append(name)

    return resolved, missing, suggestions

def venn_from_corr_subset(
    corr_matrix,
    metrics=None,            # lista com os nomes desejados (em qualquer formato)
    radius=1.0,              # raio R (igual p/ todos)
    epsilon_pos_frac=0.06,   # ε (fração de R) só p/ pares com r>0
    random_state=42,
    n_init=8,
    max_iter=600,
    figsize=(8,8),
    face_alpha=0.30,
    edge_lw=1.5,
    label_fontsize=8,
    label_weight='normal',
    label_jitter=0.03,
    repel_iters=150,
    repel_step=0.02,
    inside_margin=0.12,
    title="Venn adaptado (subset; d = (1 − r)·R; ε>0 só para r>0)"
):
    # ---------- 0) resolver nomes do subset ----------
    if metrics is not None:
        resolved, missing, sugg = _resolve_metric_names(corr_matrix.columns, metrics)
        if missing and not resolved:
            cols_list = ', '.join(map(str, corr_matrix.columns))
            raise ValueError(
                "Nenhuma métrica do subset encontrada nas colunas da corr_matrix.\n"
                f"Pedidas: {metrics}\n"
                f"Colunas disponíveis: {cols_list}"
            )
        if missing:  # avisa mas segue com as resolvidas
            print("[aviso] Métricas não encontradas e ignoradas:", missing)
            if sugg:
                print("[aviso] Sugestões aplicadas:", sugg)
        if not resolved:
            raise ValueError("Após normalização, não sobrou nenhuma métrica válida para plotar.")
        labels = resolved
        C_raw = corr_matrix.loc[labels, labels].to_numpy(dtype=float)
    else:
        labels = corr_matrix.columns.tolist()
        C_raw = corr_matrix.to_numpy(dtype=float)

    # ---------- 1) correlação → distância-alvo ----------
    C_raw = np.clip(C_raw, -1.0, 1.0)
    R = float(radius)

    # d(r) base
    T = (1.0 - C_raw) * R

    # aplica ε apenas quando r>0 (não aplica em r<=0)
    if epsilon_pos_frac and epsilon_pos_frac > 0:
        mask_pos = (C_raw > 0).astype(float)
        T += (epsilon_pos_frac * R) * mask_pos

    # simetriza e zera diagonal
    np.fill_diagonal(T, 0.0)
    T = 0.5 * (T + T.T)

    # ---------- 2) embedding (MDS SMACOF; fallback p/ clássico) ----------
    try:
        from sklearn.manifold import MDS
        mds = MDS(n_components=2, metric=True, dissimilarity='precomputed',
                  n_init=n_init, max_iter=max_iter, random_state=random_state)
        X = mds.fit_transform(T)
    except Exception:
        n = T.shape[0]
        J = np.eye(n) - np.ones((n, n))/n
        B = -0.5 * J @ (T**2) @ J
        eigvals, eigvecs = np.linalg.eigh(B)
        idx = np.argsort(eigvals)[::-1]
        L = np.clip(eigvals[idx][:2], 0, None)
        V = eigvecs[:, idx][:, :2]
        X = V * np.sqrt(L + 1e-12)

    # reescala global para aproximar T
    DX = np.sqrt(((X[:,None,:]-X[None,:,:])**2).sum(axis=2))
    iu = np.triu_indices_from(DX, k=1)
    num = np.sum(DX[iu] * T[iu])
    den = np.sum(DX[iu]**2) + 1e-12
    s = num/den if den > 0 else 1.0
    X *= s
    X -= X.mean(axis=0, keepdims=True)

    # ---------- 3) cores ----------
    def _lc(s): return str(s).strip().lower()
    color_fixed = {
        "robustness":            "#9467bd",
        "stability":             "#17becf",
        "soundness":             "#f7b6d2",
        "directional soundness": "#ff9896",
        "selectivity":           "#c7c7c7",
        "fidelity":              "#2ca02c",
        "infidelity":            "#d62728",
    }
    cmap = plt.get_cmap("tab20")
    colors, k_auto = [], 0
    for name in labels:
        key = _lc(name)
        colors.append(color_fixed.get(key, cmap(k_auto % 20)))
        if key not in color_fixed:
            k_auto += 1

    # ---------- 4) círculos ----------
    fig, ax = plt.subplots(figsize=figsize, dpi=150, facecolor="white")
    for (x, y), col in zip(X, colors):
        circ = Circle((x, y), R, facecolor=col, edgecolor=col, alpha=face_alpha, lw=edge_lw)
        ax.add_patch(circ)

    # ---------- 5) rótulos centralizados + repulsão vertical ----------
    rng = np.random.default_rng(random_state)
    text_xy = X.copy()
    text_xy[:, 1] += (rng.uniform(-label_jitter, label_jitter, size=len(labels))) * R

    def clamp_inside(i):
        cx, cy = X[i]
        tx, ty = text_xy[i]
        vx, vy = tx - cx, ty - cy
        dist = np.hypot(vx, vy)
        maxd = R * (1.0 - inside_margin)
        if dist > maxd and dist > 1e-12:
            k = maxd / dist
            text_xy[i] = np.array([cx + k*vx, cy + k*vy])

    for _ in range(repel_iters):
        moved = False
        for i in range(len(labels)):
            for j in range(i+1, len(labels)):
                xi, yi = text_xy[i]
                xj, yj = text_xy[j]
                # caixa aproximada do texto
                w = 0.9 * R
                h = 0.35 * R
                if (abs(xi - xj) < w) and (abs(yi - yj) < h):
                    dy = repel_step * R
                    text_xy[i, 1] += dy
                    text_xy[j, 1] -= dy
                    clamp_inside(i)
                    clamp_inside(j)
                    moved = True
        if not moved:
            break

    for (tx, ty), name in zip(text_xy, labels):
        ax.text(tx, ty, name, ha="center", va="center",
                fontsize=label_fontsize, fontweight=label_weight,
                color="black", zorder=3)

    # ---------- 6) estética ----------
    xmin = X[:,0].min() - R*1.3
    xmax = X[:,0].max() + R*1.3
    ymin = X[:,1].min() - R*1.3
    ymax = X[:,1].max() + R*1.3
    ax.set_aspect("equal")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    plt.tight_layout()

    # ---------- 7) salvar figura usando PATH_FILE_OUT / BASE_PATH ----------
    filename = "venn_corr_subset.png"

    # diretório principal (PATH_FILE_OUT se existir, senão ".")
    if "PATH_FILE_OUT" in globals():
        out_dir = PATH_FILE_OUT
    else:
        out_dir = "."

    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, filename)
    plt.savefig(out_path, dpi=180)

    # salva também em BASE_PATH, se existir
    if "BASE_PATH" in globals():
        base_dir = Path(BASE_PATH)
        base_dir.mkdir(parents=True, exist_ok=True)
        plt.savefig(base_dir / filename, dpi=180)
        print("Figura também salva em:", base_dir / filename)

    plt.show()

    print("Figura Venn subset salva em:", out_path)


# ===== Exemplo de uso =====
subset = ["Robustness","Fidelity", "Infidelity"]
venn_from_corr_subset(
    corr_matrix,
    metrics=subset,
    radius=1.0,
    epsilon_pos_frac=0.06,
    figsize=(8,8),
    label_fontsize=8
)


In [ ]:
# ===  surrogate_tree Interpretabilidade [Gráfico de Radar das Instâncias]===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil
from pathlib import Path

# Filtra as instâncias, removendo as linhas de agregados
df_surrogate_tree_metrics_filter = df_surrogate_tree_metrics[df_surrogate_tree_metrics["instancia"] != "MÉDIA"].copy()
df_surrogate_tree_metrics_filter = df_surrogate_tree_metrics_filter[df_surrogate_tree_metrics_filter["instancia"] != "MEDIANA"].copy()

# (opcional) Remover coluna que não deve ser incluída
df_surrogate_tree_metrics_filter = df_surrogate_tree_metrics_filter.drop(columns=["class_consistency()"], errors='ignore')

# Remove colunas sempre 100% ou 0% no DF original
cols_to_remove = [
    col for col in df_surrogate_tree_metrics_filter.columns
    if df_surrogate_tree_metrics[col].nunique() == 1
    and (df_surrogate_tree_metrics[col].iloc[0] == '100.0%' or df_surrogate_tree_metrics[col].iloc[0] == '0.0%')
]

# Mantém 'instancia' para o título, só tira completeness e as constantes acima
df_surrogate_tree_metrics_filter = df_surrogate_tree_metrics_filter.drop(
    columns=cols_to_remove + ['completeness(%)'],
    errors='ignore'
)

# Remove '%' e converte para numérico onde possível
df_surrogate_tree_metrics_filter = (
    df_surrogate_tree_metrics_filter
    .replace('%', '', regex=True)
    .apply(pd.to_numeric, errors='ignore')
)

# Lista de métricas e ângulos
metrics = [col for col in df_surrogate_tree_metrics_filter.columns if col != 'instancia']
num_metrics = len(metrics)
angles = np.linspace(0, 2 * np.pi, num_metrics, endpoint=False).tolist()
angles += angles[:1]  # fecha o polígono

# Layout do PDF
plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total_instances = len(df_surrogate_tree_metrics_filter)
pages = ceil(total_instances / plots_per_page)

# Nome do arquivo de saída (usa BASE_PATH se existir)
SAFE_BASE = Path(BASE_PATH) if 'BASE_PATH' in globals() else Path.cwd()
SAFE_BASE.mkdir(parents=True, exist_ok=True)
pdf_path = SAFE_BASE / "graficos_radar_angular_limpo.pdf"

pdf = PdfPages(str(pdf_path))

# Loop de páginas
for page in range(pages):
    fig, axs = plt.subplots(
        rows_per_page,
        plots_per_row,
        figsize=(8.27, 11.69),
        subplot_kw=dict(polar=True)
    )
    fig.suptitle(
        "Análise de Métricas XAI por Instâncias",
        fontsize=14,
        y=0.96,
        color='navy'
    )
    axs = axs.flatten()

    for i in range(plots_per_page):
        idx = page * plots_per_page + i
        if idx >= total_instances:
            axs[i].axis('off')
            continue

        row = df_surrogate_tree_metrics_filter.iloc[idx]

        # valores em [0,1] (assumindo %)
        vals = []
        for m in metrics:
            v = row[m]
            try:
                v_float = float(v)
            except Exception:
                v_float = np.nan
            vals.append(v_float / 100.0 if np.isfinite(v_float) else np.nan)
        values = vals + vals[:1]  # fecha o polígono

        ax = axs[i]

        # --- radar da instância ---
        ax.plot(angles, values, linewidth=1.25, linestyle='solid', color='navy', zorder=3)
        ax.fill(angles, values, color='cornflowerblue', alpha=0.32, zorder=2)

        # --- remover grades e valores radiais ---
        ax.grid(False)
        ax.xaxis.grid(False)
        ax.yaxis.grid(False)
        ax.set_yticks([])                       # sem círculos/valores radiais
        ax.spines['polar'].set_visible(False)
        ax.set_rlabel_position(0)

        # --- limites (um pouco acima de 1 para dar espaço aos rótulos) ---
        ax.set_ylim(0, 1.05)

        # --- rótulos das métricas ---
        ax.set_xticks([])
        label_r = 1.03
        for theta, label in zip(angles[:-1], metrics):
            ax.text(theta, label_r, label,
                    ha='center', va='center',
                    fontsize=6, fontweight='bold',
                    clip_on=False)

        # --- polígono externo (mesmo nº de lados que as métricas) ---
        outer_r = [1.0] * len(angles)           # angles já está fechado
        ax.fill(angles, outer_r, color='#f1f1f1', zorder=0)
        ax.plot(angles, outer_r, linewidth=1.0, color='#444', zorder=1)

        # --- título ---
        ax.set_title(f"Instância {int(row['instancia'])}", size=9, pad=8)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
    pdf.savefig(fig)
    plt.close(fig)

pdf.close()
print(f"✅ PDF salvo com sucesso em: {pdf_path}")


In [ ]:
# === Surrogate Decision Tree — Gráfico de Radar por Instância ===
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil
from pathlib import Path

# ---------------- Helpers ----------------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
def _first_float(x):
    """Extrai o primeiro número de uma string (ex.: '100.0% [95, 100]' -> 100.0)."""
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.replace(",", ".")
        m = _num_re.search(s)
        return float(m.group(0)) if m else np.nan
    return np.nan

# ---------------- Fonte de dados ----------------
df_src = df_surrogate_tree_metrics.copy()

# Remove linhas agregadas
df_f = df_src[~df_src["instancia"].isin(["MÉDIA", "MEDIANA"])].copy()

# (opcional) remova colunas que não quer no radar
drop_optional = []  # ex.: ["completeness(%)"]
df_f.drop(columns=[c for c in drop_optional if c in df_f.columns],
          inplace=True, errors="ignore")

# Converte tudo que for textual/percentual p/ número (pega o 1º número da célula)
df_num = df_f.applymap(_first_float)

# Reintroduz a coluna 'instancia' (numérica/índice)
df_num["instancia"] = df_f["instancia"]

# Remove colunas constantes (0%/100%/NaN)
const_cols = []
for c in df_num.columns:
    if c == "instancia":
        continue
    col = df_num[c].dropna()
    if col.empty or col.nunique(dropna=True) <= 1:
        const_cols.append(c)

df_num = df_num.drop(columns=const_cols, errors="ignore")

# Se sobrar só 'instancia', aborta
metric_cols = [c for c in df_num.columns if c != "instancia"]
if len(metric_cols) == 0:
    raise ValueError("Não sobraram métricas variáveis para o radar (todas constantes/NaN).")

# Normaliza para [0,1] assumindo que estão em 0–100
df_plot = df_num.copy()
df_plot[metric_cols] = df_plot[metric_cols].astype(float).clip(0, 100) / 100.0

# ---------------- Parâmetros do layout ----------------
metrics = metric_cols
num_metrics = len(metrics)
angles = np.linspace(0, 2*np.pi, num_metrics, endpoint=False).tolist()
angles += angles[:1]  # fecha o polígono

plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total_instances = len(df_plot)
pages = ceil(total_instances / plots_per_page)

# Saída: prioriza BASE_PATH, depois PATH_FILE_OUT, depois cwd
if 'BASE_PATH' in globals():
    SAFE_BASE = Path(BASE_PATH)
elif 'PATH_FILE_OUT' in globals():
    SAFE_BASE = Path(PATH_FILE_OUT)
else:
    SAFE_BASE = Path.cwd()

SAFE_BASE.mkdir(parents=True, exist_ok=True)
pdf_path = SAFE_BASE / "surrogate_radar_instancias.pdf"

with PdfPages(str(pdf_path)) as pdf:
    for page in range(pages):
        fig, axs = plt.subplots(
            rows_per_page,
            plots_per_row,
            figsize=(8.27, 11.69),  # A4 retrato
            subplot_kw=dict(polar=True)
        )
        fig.suptitle("Surrogate — Métricas por Instância",
                     fontsize=14, y=0.96, color='navy')
        axs = axs.flatten()

        for i in range(plots_per_page):
            idx = page * plots_per_page + i
            if idx >= total_instances:
                axs[i].axis('off')
                continue

            row = df_plot.iloc[idx]
            vals = [float(row[m]) for m in metrics]
            vals += vals[:1]

            ax = axs[i]
            # radar
            ax.plot(angles, vals, linewidth=1.25,
                    linestyle='solid', color='navy', zorder=3)
            ax.fill(angles, vals, color='cornflowerblue',
                    alpha=0.32, zorder=2)

            # limpar grades/valores
            ax.grid(False)
            ax.xaxis.grid(False)
            ax.yaxis.grid(False)
            ax.set_yticks([])
            ax.spines['polar'].set_visible(False)
            ax.set_rlabel_position(0)
            ax.set_ylim(0, 1.05)

            # rótulos das métricas ao redor
            ax.set_xticks([])
            label_r = 1.03
            for theta, label in zip(angles[:-1], metrics):
                ax.text(theta, label_r, label,
                        ha='center', va='center',
                        fontsize=6, fontweight='bold',
                        clip_on=False)

            # anel externo
            outer_r = [1.0] * len(angles)
            ax.fill(angles, outer_r, color='#f1f1f1', zorder=0)
            ax.plot(angles, outer_r, linewidth=1.0,
                    color='#444', zorder=1)

            # título
            try:
                inst_label = int(row['instancia'])
            except Exception:
                inst_label = row['instancia']
            ax.set_title(f"Instância {inst_label}", size=9, pad=8)

        plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
        pdf.savefig(fig, dpi=180)
        plt.close(fig)

print(f"✅ PDF salvo com sucesso em: {pdf_path}")
print("Colunas removidas por serem constantes/NaN:", const_cols)
print("Métricas plotadas:", metrics)


In [ ]:
# === Surrogate Decision Tree [Gráfico de Radar Global — todas as instâncias sobrepostas] ===
import re
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------- helper: extrai o 1º número de uma célula (ex.: "97.5% [93.1, 99.8]" -> 97.5) ----------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
def _first_float(x):
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.replace(",", ".")
        m = _num_re.search(s)
        return float(m.group(0)) if m else np.nan
    return np.nan

# ---------- 1) Fonte e limpeza ----------
df_src = df_surrogate_tree_metrics.copy()

# remove linhas agregadas
df_plot = df_src[~df_src["instancia"].isin(["MÉDIA", "MEDIANA"])].copy()

# (opcional) remover colunas não desejadas se existirem
drop_optional = ["class_consistency()"]
df_plot.drop(columns=[c for c in drop_optional if c in df_plot.columns],
             inplace=True, errors="ignore")

# remover colunas constantes 0% ou 100% (opcional)
cols_to_remove = []
for col in df_plot.columns:
    if col == "instancia":
        continue
    vals = df_plot[col].astype(str).tolist()
    uniq = list(set(vals))
    if len(uniq) == 1 and (uniq[0].strip().startswith("0") or uniq[0].strip().startswith("100")):
        cols_to_remove.append(col)

# 'completeness(%)' também pode ser removida se você não quiser no radar
df_plot = df_plot.drop(columns=cols_to_remove + ["completeness(%)"], errors="ignore")

# converte cada célula textual/percentual em número (pega o primeiro número)
df_num = df_plot.applymap(_first_float)
df_num["instancia"] = df_plot["instancia"]

# ---------- 2) Métricas e ângulos ----------
metrics = [c for c in df_num.columns if c != "instancia"]
if not metrics:
    raise ValueError("Não há métricas numéricas para plotar (verifique colunas do df_surrogate_tree_metrics).")

angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

# ---------- 3) Gráfico único com todas as instâncias ----------
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("Surrogate — Todas as instâncias sobrepostas",
             fontsize=16, color='darkorange')

# eixos/grades
rgrid_values = [10, 30, 50, 70, 90]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_yticks(rgrid_values)
ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=7)
ax.set_rlabel_position(0)
ax.set_ylim(0, 100)
ax.spines['polar'].set_visible(True)
ax.spines['polar'].set_color('darkorange')
ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

# plot das instâncias
for _, row in df_num.iterrows():
    values = [row[m] for m in metrics]
    values += values[:1]
    try:
        inst_label = int(row["instancia"])
    except Exception:
        inst_label = row["instancia"]

    ax.plot(
        angles,
        values,
        linewidth=1.2,
        linestyle='solid',
        alpha=0.55,
        label=f"Instância {inst_label}"
    )
    ax.fill(angles, values, alpha=0.05, color='orange')

# legenda (cuidado: pode ficar grande se houver muitas instâncias)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.05), fontsize=7)

plt.tight_layout()

# ---------- 4) Salvar usando BASE_PATH (ou PATH_FILE_OUT / cwd) ----------
if 'BASE_PATH' in globals():
    SAFE_BASE = Path(BASE_PATH)
elif 'PATH_FILE_OUT' in globals():
    SAFE_BASE = Path(PATH_FILE_OUT)
else:
    SAFE_BASE = Path.cwd()

SAFE_BASE.mkdir(parents=True, exist_ok=True)
out_path = SAFE_BASE / "surrogate_radar_global_overlay.png"
plt.savefig(out_path, dpi=180)
plt.show()

print(f"✅ Gráfico de radar global salvo em: {out_path}")
print("Colunas removidas (constantes ou filtradas):", cols_to_remove)
print("Métricas usadas no radar global:", metrics)


In [ ]:
# === Surrogate Decision Tree — Gráfico de Radar por Instâncias (Grupos de Métricas) ===
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from math import ceil
from pathlib import Path

# -------- helper p/ converter "100.0% [95, 100]" -> 100.0 --------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
def _first_float(x):
    if isinstance(x, (int, float, np.number)):
        return float(x)
    if isinstance(x, str):
        s = x.replace(",", ".")
        m = _num_re.search(s)
        return float(m.group(0)) if m else np.nan
    return np.nan

# -------- 1) Filtrar/normalizar fonte --------
df_src = df_surrogate_tree_metrics.copy()
df_f = df_src[~df_src["instancia"].isin(["MÉDIA", "MEDIANA"])].copy()

# Converte tudo que for textual/percentual p/ número (pega 1º número da célula)
df_num = df_f.applymap(_first_float)
df_num["instancia"] = df_f["instancia"]

# -------- 2) Definir grupos (mantém só o que existe no DF) --------
grupos = {
    "Fidelidade": [
        "fidelity(%)", "infidelity(%)", "faithfulness(%)",
        "soundness(%)", "directional soundness(%)"
    ],
    "Relevância": [
        "selectivity(%)"
    ],
    "Simplicidade": [
        "simplicity(%)"
    ],
    "Robustez": [
        "robustez(%)", "stability(%)"
    ],
    "Generalização": [
        "consistency(%)",
        "sufficiency-1(%)", "sufficiency-5(%)",
        "sufficiency-10(%)", "sufficiency-20(%)",
        "completeness(%)"   # se não existir, é ignorado
    ],
}

# Interseção com as colunas realmente presentes
for k, cols in list(grupos.items()):
    grupos[k] = [c for c in cols if c in df_num.columns]
# Remove grupos vazios
grupos = {k: v for k, v in grupos.items() if len(v) > 0}

if not grupos:
    raise ValueError("Nenhuma métrica de grupo disponível em df_surrogate_tree_metrics.")

# -------- 3) Médias por grupo (por instância) --------
df_grupos = pd.DataFrame({"instancia": df_num["instancia"]})
for nome_grupo, cols in grupos.items():
    df_grupos[nome_grupo] = df_num[cols].astype(float).mean(axis=1, skipna=True)

# Escala 0–100 (já está em %, mas garantimos limites)
for c in grupos.keys():
    df_grupos[c] = df_grupos[c].clip(0, 100)

# -------- 4) Radar por instância (grupos) --------
metrics = [c for c in df_grupos.columns if c != "instancia"]
n = len(metrics)
angles = np.linspace(0, 2*np.pi, n, endpoint=False).tolist()
angles += angles[:1]

plots_per_row = 2
rows_per_page = 4
plots_per_page = plots_per_row * rows_per_page
total = len(df_grupos)
pages = ceil(total / plots_per_page)
rgrid_values = [10, 30, 50, 70, 90]

# Diretório de saída: prioriza BASE_PATH, depois PATH_FILE_OUT, depois cwd
if "BASE_PATH" in globals():
    SAFE_BASE = Path(BASE_PATH)
elif "PATH_FILE_OUT" in globals():
    SAFE_BASE = Path(PATH_FILE_OUT)
else:
    SAFE_BASE = Path.cwd()

SAFE_BASE.mkdir(parents=True, exist_ok=True)
pdf_path = SAFE_BASE / "surrogate_radar_grupos_por_instancia.pdf"

with PdfPages(pdf_path) as pdf:
    for page in range(pages):
        fig, axs = plt.subplots(
            rows_per_page,
            plots_per_row,
            figsize=(8.27, 11.69),
            subplot_kw=dict(polar=True)
        )
        fig.suptitle(
            "Surrogate — Radar por Instância (Grupos de Métricas)",
            fontsize=14, y=0.96, color='darkorange'
        )
        axs = axs.flatten()

        for i in range(plots_per_page):
            idx = page*plots_per_page + i
            if idx >= total:
                axs[i].axis('off')
                continue

            row = df_grupos.iloc[idx]
            raw_vals = [row[m] for m in metrics]
            vals = raw_vals + raw_vals[:1]  # fecha polígono

            ax = axs[i]
            ax.plot(angles, vals, linewidth=2, linestyle='solid', color='darkorange')
            ax.fill(angles, vals, color='orange', alpha=0.30)

            ax.set_xticks(angles[:-1])
            ax.set_xticklabels(metrics, fontsize=7)
            for lab in ax.get_xticklabels():
                lab.set_horizontalalignment('center')
                lab.set_fontsize(6)
                lab.set_rotation(15)

            ax.set_yticks(rgrid_values)
            ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=5)
            ax.set_rlabel_position(0)
            ax.set_ylim(0, 100)  # percentuais
            ax.set_title(
                f"Instância {int(row['instancia'])}",
                size=9, pad=10, color='darkorange'
            )
            ax.spines['polar'].set_visible(True)
            ax.spines['polar'].set_color('darkorange')
            ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

        plt.tight_layout(rect=[0, 0.03, 1, 0.95], h_pad=3.0)
        pdf.savefig(fig, dpi=180)
        plt.show()
        plt.close(fig)

print(f"✅ PDF salvo em: {pdf_path}")
print("Grupos usados e suas colunas:")
for g, cols in grupos.items():
    print(f" - {g}: {cols}")


In [ ]:
# === Surrogate Decision Tree [Gráfico de Radar por Instâncias e Global por Grupos de Métricas] ===
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---------------- Helper: extrai o 1º número da célula ----------------
_num_re = re.compile(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?")
def _first_float(x):
    if isinstance(x, (int, float, np.number)): 
        return float(x)
    if isinstance(x, str):
        s = x.replace(",", ".")
        m = _num_re.search(s)
        return float(m.group(0)) if m else np.nan
    return np.nan

# ===== 1) Filtrar e preparar (usa df_surrogate_tree_metrics) =====
df_src = df_surrogate_tree_metrics.copy()

# Remove linhas agregadas
df_sur_metrics = df_src[
    ~df_src["instancia"].isin(["MÉDIA", "MEDIANA"])
].copy()

# Remover colunas que não devem entrar (se existirem)
drop_optional = ["class_consistency()"]   # ajuste se necessário
df_sur_metrics.drop(columns=[c for c in drop_optional if c in df_sur_metrics.columns],
                    inplace=True, errors="ignore")

# Remover colunas constantes 0%/100% (opcional)
cols_to_remove = []
for col in df_sur_metrics.columns:
    if col == "instancia":
        continue
    uniques = df_sur_metrics[col].astype(str).unique()
    if len(uniques) == 1 and (uniques[0].startswith("100") or uniques[0].startswith("0")):
        cols_to_remove.append(col)

df_sur_metrics = df_sur_metrics.drop(
    columns=cols_to_remove + ["completeness(%)"], errors="ignore"
)

# Converte células tipo "100.0% [95, 100]" para número (pega o 1º número)
df_sur_metrics_num = df_sur_metrics.applymap(_first_float)
df_sur_metrics_num["instancia"] = df_sur_metrics["instancia"]

# ===== 2) Definir grupos =====
grupos = {
    "Fidelidade": [
        "fidelity(%)", "infidelity(%)", "faithfulness(%)",
        "soundness(%)", "directional soundness(%)",
    ],
    "Relevância": [
        "selectivity(%)",
    ],
    "Simplicidade": [
        "simplicity(%)",
    ],
    "Robustez": [
        "robustez(%)",   # nome usado na tabela do surrogate
        "stability(%)",
    ],
    "Generalização": [
        "consistency(%)",
        "sufficiency-1(%)", "sufficiency-5(%)",
        "sufficiency-10(%)", "sufficiency-20(%)",
        "completeness(%)",
    ],
}

# Mantém apenas colunas existentes
for k, v in list(grupos.items()):
    grupos[k] = [c for c in v if c in df_sur_metrics_num.columns]
# Remove grupos vazios
grupos = {k: v for k, v in grupos.items() if len(v) > 0}
if not grupos:
    raise ValueError("Nenhuma coluna válida encontrada para os grupos nas métricas do surrogate.")

# ===== 3) Calcular médias por grupo (por instância) =====
df_grupos = pd.DataFrame({"instancia": df_sur_metrics_num["instancia"]})
for nome_grupo, cols in grupos.items():
    df_grupos[nome_grupo] = df_sur_metrics_num[cols].astype(float).mean(axis=1, skipna=True)

# Garante faixa 0–100
for c in df_grupos.columns:
    if c != "instancia":
        df_grupos[c] = df_grupos[c].clip(0, 100)

# ===== 4) Gráfico único com todas as instâncias (grupos) =====
metrics = [c for c in df_grupos.columns if c != "instancia"]
angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
fig.suptitle("Surrogate — Todas as Instâncias sobrepostas (Grupos de Métricas)", 
             fontsize=16, color='darkorange')

# Eixos e grade
rgrid_values = [10, 30, 50, 70, 90]
ax.set_xticks(angles[:-1])
ax.set_xticklabels(metrics, fontsize=8)
ax.set_yticks(rgrid_values)
ax.set_yticklabels([str(v) for v in rgrid_values], fontsize=7)
ax.set_rlabel_position(0)
ax.set_ylim(0, 100)
ax.spines['polar'].set_visible(True)
ax.spines['polar'].set_color('darkorange')
ax.grid(True, linestyle='dotted', linewidth=0.8, alpha=0.6, color='orange')

# Plot de todas as instâncias
for _, row in df_grupos.iterrows():
    values = [row[m] for m in metrics]
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, linestyle='solid', alpha=0.5,
            label=f"Instância {int(row['instancia'])}")
    ax.fill(angles, values, alpha=0.05, color='orange')

# Legenda (pode ficar grande; ajuste se necessário)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.05), fontsize=7)
plt.tight_layout()

# ===== 5) Salvando com BASE_PATH / PATH_FILE_OUT =====
if "BASE_PATH" in globals():
    SAFE_BASE = Path(BASE_PATH)
elif "PATH_FILE_OUT" in globals():
    SAFE_BASE = Path(PATH_FILE_OUT)
else:
    SAFE_BASE = Path.cwd()

SAFE_BASE.mkdir(parents=True, exist_ok=True)
out_file = SAFE_BASE / "surrogate_radar_grupos_global_instancias.png"
plt.savefig(out_file, dpi=180)
plt.show()

print(f"✅ Figura salva em: {out_file}")


In [ ]:
# === UpSet plot das métricas SURROGATE (sobreposição por instância) ===
# Usa df_surrogate_tree_metrics já existente. Salva em BASE_PATH/PATH_FILE_OUT / "upset_surrogate_metricas.png"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# upsetplot
try:
    from upsetplot import UpSet, from_indicators
except Exception:
    raise ImportError("Instale o pacote 'upsetplot': pip install upsetplot")

# ----------------- CONFIG -----------------
THRESHOLD_ABS = 60.0       # limiar absoluto em %
USE_PERCENTILE = False     # se True, usa percentil por métrica
PERCENTILE_Q = 75

# BASE_PATH seguro (aceita str ou Path; se não existir, usa cwd)
if 'PATH_FILE_OUT' in globals():
    SAFE_BASE = Path(PATH_FILE_OUT)
elif 'BASE_PATH' in globals():
    SAFE_BASE = Path(BASE_PATH)
else:
    SAFE_BASE = Path.cwd()

OUTFILE = SAFE_BASE / "upset_surrogate_metricas.png"

# ----------------- PREPARO -----------------
# preserva média/mediana numa cópia separada (para referência)
agg_rows = df_surrogate_tree_metrics[
    df_surrogate_tree_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA", "MEDIANA"])
].copy()

# trabalha numa cópia sem média/mediana para o UpSet
dfu = df_surrogate_tree_metrics[
    ~df_surrogate_tree_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA", "MEDIANA"])
].copy()

# pega colunas de métricas "(%)"
metric_cols = [c for c in dfu.columns if c.endswith("(%)")]
if not metric_cols:
    raise ValueError("Não encontrei colunas de métricas no formato 'nome(%)' em df_surrogate_tree_metrics.")

# “xx.x%” -> float
M = (
    dfu[metric_cols]
    .replace('%', '', regex=True)
    .replace('N/A', np.nan)
    .apply(pd.to_numeric, errors='coerce')
)

# limiares por métrica
if USE_PERCENTILE:
    thresholds = M.quantile(PERCENTILE_Q / 100.0, numeric_only=True)
else:
    thresholds = pd.Series(THRESHOLD_ABS, index=M.columns, dtype=float)

# indicadores booleanos: métrica >= limiar
indicators = M.ge(thresholds, axis=1)

# constrói dados para o UpSet
data_upset = from_indicators(indicators=indicators.columns.tolist(), data=indicators)

# ----------------- PLOT -----------------
plt.figure(figsize=(12, 7), dpi=150)
up = UpSet(
    data_upset,
    show_counts=True,
    sort_by="cardinality",
    sort_categories_by="cardinality",
    element_size=28
)
up.plot()

title = (f"UpSet Surrogate — interseções de métricas ≥ P{PERCENTILE_Q}"
         if USE_PERCENTILE else
         f"UpSet Surrogate — interseções de métricas ≥ {THRESHOLD_ABS:.0f}%")
plt.suptitle(title, y=1.02)

SAFE_BASE.mkdir(parents=True, exist_ok=True)
plt.savefig(OUTFILE, dpi=600, bbox_inches="tight")
plt.show()
print(f"✅ UpSet salvo em: {OUTFILE}")
# opcional: acessar agg_rows se quiser consultar média/mediana preservadas


In [ ]:
# === UpSet de Correlação entre Métricas SURROGATE ===
# Nomes das métricas à esquerda (padrão) + replicados na base (vertical)
# Requer: df_surrogate_tree_metrics e (opcional) BASE_PATH / PATH_FILE_OUT

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# upsetplot
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    raise ImportError("Instale: pip install upsetplot")

# ----------------- CONFIG -----------------
TAU = 0.30        # limiar de |correlação| p/ considerar “alta correlação”
MIN_GROUP = 2     # grupos com pelo menos 2 métricas

if "PATH_FILE_OUT" in globals():
    SAFE_BASE = Path(PATH_FILE_OUT)
elif "BASE_PATH" in globals():
    SAFE_BASE = Path(BASE_PATH)
else:
    SAFE_BASE = Path.cwd()

OUTFILE = SAFE_BASE / "upset_surrogate_corr.png"

# ----------------- PREPARO -----------------
# remove média/mediana
dfu = df_surrogate_tree_metrics[
    ~df_surrogate_tree_metrics["instancia"].astype(str).str.upper().isin(["MÉDIA","MEDIANA"])
].copy()

# seleciona métricas "(%)" e converte p/ float
metric_cols = [c for c in dfu.columns if c.endswith("(%)")]
if not metric_cols:
    raise ValueError("Não encontrei colunas de métricas no formato 'nome(%)' em df_surrogate_tree_metrics.")

M = (
    dfu[metric_cols]
      .replace('%', '', regex=True)
      .replace('N/A', np.nan)
      .apply(pd.to_numeric, errors='coerce')
)

# correlação coluna-coluna
corr = M.corr(method="pearson")

# memberships: para cada métrica i, conjunto das que têm |r| ≥ TAU com i
metrics = corr.columns.tolist()
memberships, labels_surrogate = [], []
for i in metrics:
    group = [
        j for j in metrics
        if j != i and np.isfinite(corr.loc[i, j]) and abs(corr.loc[i, j]) >= TAU
    ]
    if len(group) >= MIN_GROUP:
        memberships.append(group)
        labels_surrogate.append(i)

if not memberships:
    print(f"⚠️ Nenhum grupo de métricas com |r| ≥ {TAU:.2f}. Tente reduzir TAU.")
else:
    data = from_memberships(memberships)

    # ----------------- PLOT (horizontal: nomes à esquerda) -----------------
    fig = plt.figure(figsize=(22, 10), dpi=300)
    up = UpSet(
        data,
        show_counts=True,
        sort_by="cardinality",
        sort_categories_by="cardinality",
        element_size=30,
        orientation="horizontal",  # mantém nomes à esquerda
        subset_size="count"        # lida com grupos não únicos
    )
    up.plot()

    # --------- Faixa auxiliar com nomes das métricas na base ----------
    fig = plt.gcf()
    metric_set = set(metrics)
    left_axis = None
    for ax in fig.axes:
        ytxt = [t.get_text() for t in ax.get_yticklabels()]
        match_count = sum(1 for t in ytxt if t in metric_set)
        if match_count >= max(3, len(metric_set) // 2):
            left_axis = ax
            break

    if left_axis is None:
        best_ax, best_score = None, -1
        for ax in fig.axes:
            score = sum(1 for t in ax.get_yticklabels() if t.get_text() in metric_set)
            if score > best_score:
                best_ax, best_score = ax, score
        left_axis = best_ax

    if left_axis is not None:
        pos = left_axis.get_position()  # bbox do eixo da matriz

        gutter_h = 0.10  # altura da faixa inferior
        bottom = max(0.02, pos.y0 - gutter_h - 0.02)
        ax_bottom = fig.add_axes([pos.x0, bottom, pos.width, gutter_h])
        ax_bottom.set_xlim(0, 1)
        ax_bottom.set_ylim(-0.2, 1.0)
        ax_bottom.axis("off")

        ylabels = [lab.get_text() for lab in left_axis.get_yticklabels() if lab.get_text()]
        n = len(ylabels)
        if n > 0:
            xs = np.linspace(0.0, 1.0, n, endpoint=True)
            for x, name in zip(xs, ylabels):
                ax_bottom.text(
                    x, -0.12, name,
                    rotation=90,
                    ha="center",
                    va="top",
                    fontsize=10
                )

    plt.suptitle(f"UpSet Surrogate — grupos de métricas com |corr| ≥ {TAU:.2f}", y=0.98)
    SAFE_BASE.mkdir(parents=True, exist_ok=True)
    plt.savefig(OUTFILE, dpi=600, bbox_inches="tight")
    plt.show()
    print(f"✅ UpSet (correlação) Surrogate salvo em: {OUTFILE}")
